In [ ]:
# ======================================================================================
# DEFINITIVE CLEAN GAZE EXPERIMENT
# STEP 2 — 5-FOLD HELD-OUT-ECG GAZE ALIGNMENT
#
# FROZEN:
#   ECG input: final_digitisation_v9/X_gaze_final.npy
#   Targets: definitive_clean_gaze/clean_gaze_targets.npz
#
# Attention:
#   add_3 -> RMS over (time, channels) -> normalise across 12 leads
#
# Training:
#   5 folds, 2 held-out ECGs/fold
#   only conv2d_10 trainable
#   BatchNorm frozen
#   training=False throughout forward passes
#   gaze-only MSE
#   Adam 1e-4
#   25 epochs
#   40 noisy copies/training ECG
#   Gaussian noise SD = 0.03
#   batch size = 16
#
# Consultant and all-clinician arms use identical:
#   folds
#   starting weights
#   augmented X
#   batch order
#
# Only the gaze target differs.
# ======================================================================================

import gc
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import KFold
from scipy.stats import wilcoxon


# ======================================================================================
# 1. PARAMETERS
# ======================================================================================

SEED = 2026
N_SPLITS = 5

AUGMENTATIONS_PER_ECG = 40
NOISE_SD = 0.03

BATCH_SIZE = 16
EPOCHS = 25

LEARNING_RATE = 1e-4
RMS_EPSILON = 1e-8


ECG_ORDER = [
    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]


LEAD_ORDER = [
    "I",
    "II",
    "III",
    "aVR",
    "aVL",
    "aVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]


POPULATIONS = [
    "consultant",
    "all_clinicians",
]


# ======================================================================================
# 2. REPRODUCIBILITY
# ======================================================================================

np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as e:
    print("Could not explicitly enable deterministic operations:", repr(e))


# ======================================================================================
# 3. PATHS
# ======================================================================================

ROOT = Path("definitive_clean_gaze")

STEP1_MANIFEST = ROOT / "step1_manifest.json"
TARGET_PATH = ROOT / "clean_gaze_targets.npz"

X_PATH = Path(
    "final_digitisation_v9/X_gaze_final.npy"
)

STEP2_DIR = ROOT / "step2_cv"
STEP2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR = STEP2_DIR / "fold_weights"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for p in [
    STEP1_MANIFEST,
    TARGET_PATH,
    X_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(p)


# ======================================================================================
# 4. HASH HELPER
# ======================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 5. VERIFY FROZEN FILES
# ======================================================================================

with open(
    STEP1_MANIFEST,
    "r"
) as f:

    step1_manifest = json.load(f)


BASE_MODEL_PATH = Path(
    step1_manifest["base_model"]
)


if not BASE_MODEL_PATH.exists():
    raise FileNotFoundError(
        BASE_MODEL_PATH
    )


current_x_hash = sha256_file(
    X_PATH
)

current_target_hash = sha256_file(
    TARGET_PATH
)

current_base_hash = sha256_file(
    BASE_MODEL_PATH
)


print("=" * 100)
print("FROZEN INPUT VERIFICATION")
print("=" * 100)

print("V9 ECG hash:")
print(current_x_hash)

print("\nTarget hash:")
print(current_target_hash)

print("\nBase-model hash:")
print(current_base_hash)


if (
    current_x_hash
    !=
    step1_manifest["frozen_ecg_sha256"]
):
    raise RuntimeError(
        "Frozen V9 ECG input hash no longer matches Step 1."
    )


if (
    current_target_hash
    !=
    step1_manifest["clean_gaze_targets_sha256"]
):
    raise RuntimeError(
        "Gaze-target hash no longer matches Step 1."
    )


if (
    current_base_hash
    !=
    step1_manifest["base_model_sha256"]
):
    raise RuntimeError(
        "Base-model hash no longer matches Step 1."
    )


print(
    "\n✓ All frozen hashes match Step 1."
)


# ======================================================================================
# 6. LOAD DATA
# ======================================================================================

X = np.load(
    X_PATH
).astype(
    np.float32
)


target_data = np.load(
    TARGET_PATH
)


Y = {

    "consultant":
        target_data[
            "consultant"
        ].astype(
            np.float32
        ),

    "all_clinicians":
        target_data[
            "all_clinicians"
        ].astype(
            np.float32
        ),
}


saved_ecg_order = [
    str(x)
    for x in target_data[
        "ecg_order"
    ]
]


saved_lead_order = [
    str(x)
    for x in target_data[
        "lead_order"
    ]
]


if saved_ecg_order != ECG_ORDER:
    raise RuntimeError(
        "Saved ECG order mismatch."
    )


if saved_lead_order != LEAD_ORDER:
    raise RuntimeError(
        "Saved lead order mismatch."
    )


if X.shape != (
    10,
    12,
    1000,
    1
):
    raise RuntimeError(
        f"Unexpected X shape: {X.shape}"
    )


for population in POPULATIONS:

    if Y[population].shape != (
        10,
        12
    ):
        raise RuntimeError(
            f"Unexpected target shape for {population}: "
            f"{Y[population].shape}"
        )


print(
    "\n✓ Frozen arrays loaded."
)


# ======================================================================================
# 7. FROZEN ATTENTION READOUT
# ======================================================================================

def attention_distribution_from_add3(
    add3_tensor
):

    # add_3:
    # (batch, 12, 55, 128)

    rms = tf.sqrt(
        tf.reduce_mean(
            tf.square(
                add3_tensor
            ),
            axis=[
                2,
                3
            ]
        )
        +
        RMS_EPSILON
    )


    denominator = tf.reduce_sum(
        rms,
        axis=1,
        keepdims=True
    )


    return (
        rms
        /
        (
            denominator
            +
            RMS_EPSILON
        )
    )


# ======================================================================================
# 8. METRICS
# ======================================================================================

def cosine_metric(p, q):

    p = np.asarray(
        p,
        dtype=np.float64
    )

    q = np.asarray(
        q,
        dtype=np.float64
    )


    denominator = (
        np.linalg.norm(p)
        *
        np.linalg.norm(q)
    )


    if denominator <= 0:
        return np.nan


    return float(
        np.dot(p, q)
        /
        denominator
    )


def mse_metric(p, q):

    p = np.asarray(
        p,
        dtype=np.float64
    )

    q = np.asarray(
        q,
        dtype=np.float64
    )


    return float(
        np.mean(
            np.square(
                p - q
            )
        )
    )


def jsd_metric(
    p,
    q,
    eps=1e-12
):

    p = np.asarray(
        p,
        dtype=np.float64
    )

    q = np.asarray(
        q,
        dtype=np.float64
    )


    p = np.clip(
        p,
        eps,
        None
    )

    q = np.clip(
        q,
        eps,
        None
    )


    p /= p.sum()
    q /= q.sum()


    m = 0.5 * (
        p + q
    )


    kl_pm = np.sum(
        p
        *
        np.log(
            p / m
        )
    )


    kl_qm = np.sum(
        q
        *
        np.log(
            q / m
        )
    )


    return float(
        0.5
        *
        (
            kl_pm
            +
            kl_qm
        )
    )


# ======================================================================================
# 9. LOAD A FRESH MODEL FOR EACH ARM/FOLD
# ======================================================================================

def load_fresh_representation_model():

    tf.keras.backend.clear_session()

    tf.random.set_seed(
        SEED
    )


    model = tf.keras.models.load_model(
        BASE_MODEL_PATH,
        compile=False
    )


    # Freeze entire network.
    for layer in model.layers:
        layer.trainable = False


    # Only this layer may change.
    conv10 = model.get_layer(
        "conv2d_10"
    )

    conv10.trainable = True


    add3 = model.get_layer(
        "add_3"
    )


    rep_model = tf.keras.Model(
        inputs=model.input,
        outputs=add3.output
    )


    train_vars = list(
        conv10.trainable_variables
    )


    if len(train_vars) == 0:
        raise RuntimeError(
            "conv2d_10 has no trainable variables."
        )


    return (
        model,
        rep_model,
        conv10,
        train_vars
    )


# ======================================================================================
# 10. PREDICT LEAD ATTENTION
# ======================================================================================

def predict_attention(
    rep_model,
    x
):

    activations = rep_model(
        x,
        training=False
    )


    attention = (
        attention_distribution_from_add3(
            activations
        )
        .numpy()
    )


    if not np.all(
        np.isfinite(
            attention
        )
    ):
        raise RuntimeError(
            "Non-finite attention values."
        )


    if not np.allclose(
        attention.sum(
            axis=1
        ),
        1.0,
        atol=1e-5
    ):
        raise RuntimeError(
            "Attention distributions do not sum to 1."
        )


    if np.any(
        attention < 0
    ):
        raise RuntimeError(
            "Negative attention detected."
        )


    return attention


# ======================================================================================
# 11. SNAPSHOT / COMPARE WEIGHTS
# ======================================================================================

def snapshot_layer_weights(
    model
):

    snapshot = {}


    for layer in model.layers:

        snapshot[
            layer.name
        ] = [
            np.array(
                weight,
                copy=True
            )
            for weight in layer.get_weights()
        ]


    return snapshot


def compare_layer_weights(
    before,
    model
):

    rows = []


    for layer in model.layers:

        old_weights = before[
            layer.name
        ]

        new_weights = layer.get_weights()


        if len(old_weights) != len(new_weights):
            raise RuntimeError(
                f"Weight-count mismatch: {layer.name}"
            )


        max_difference = 0.0


        for old, new in zip(
            old_weights,
            new_weights
        ):

            if old.shape != new.shape:
                raise RuntimeError(
                    f"Weight-shape mismatch: {layer.name}"
                )


            if old.size > 0:

                difference = float(
                    np.max(
                        np.abs(
                            old - new
                        )
                    )
                )


                max_difference = max(
                    max_difference,
                    difference
                )


        rows.append(
            {
                "layer":
                    layer.name,

                "n_weight_tensors":
                    len(
                        new_weights
                    ),

                "max_abs_change":
                    max_difference,
            }
        )


    return pd.DataFrame(
        rows
    )


# ======================================================================================
# 12. FREEZE CROSS-VALIDATION SPLITS
# ======================================================================================

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)


folds = []


for fold_number, split in enumerate(
    kf.split(
        np.arange(
            len(
                ECG_ORDER
            )
        )
    ),
    start=1
):

    train_indices, test_indices = split


    folds.append(
        {
            "fold":
                fold_number,

            "train_indices":
                np.asarray(
                    train_indices,
                    dtype=int
                ),

            "test_indices":
                np.asarray(
                    test_indices,
                    dtype=int
                ),
        }
    )


print(
    "\n" +
    "=" * 100
)

print(
    "FROZEN 5-FOLD SPLIT"
)

print(
    "=" * 100
)


fold_manifest_rows = []


for fold in folds:

    train_names = [
        ECG_ORDER[i]
        for i in fold[
            "train_indices"
        ]
    ]


    test_names = [
        ECG_ORDER[i]
        for i in fold[
            "test_indices"
        ]
    ]


    print(
        f"\nFold {fold['fold']}"
    )

    print(
        "  held out:",
        test_names
    )

    print(
        "  training:",
        train_names
    )


    fold_manifest_rows.append(
        {
            "fold":
                fold[
                    "fold"
                ],

            "held_out_ECGs":
                " | ".join(
                    test_names
                ),

            "training_ECGs":
                " | ".join(
                    train_names
                ),
        }
    )


pd.DataFrame(
    fold_manifest_rows
).to_csv(
    STEP2_DIR
    /
    "fold_manifest.csv",
    index=False
)


# ======================================================================================
# 13. TRAIN ONE POPULATION × FOLD
# ======================================================================================

def train_one_arm(
    population,
    fold_number,
    train_indices,
    test_indices,
    X_train_augmented,
    epoch_permutations
):

    (
        model,
        rep_model,
        conv10,
        train_vars
    ) = load_fresh_representation_model()


    before_weights = snapshot_layer_weights(
        model
    )


    print(
        "\nTrainable conv2d_10 variables:"
    )

    for variable in train_vars:

        print(
            "   ",
            variable.name,
            tuple(
                variable.shape
            )
        )


    # ------------------------------------------------------------------
    # Baseline held-out attention
    # ------------------------------------------------------------------

    X_test = X[
        test_indices
    ]


    Y_test = Y[
        population
    ][
        test_indices
    ]


    baseline_attention = predict_attention(
        rep_model,
        X_test
    )


    # ------------------------------------------------------------------
    # Training targets
    # ------------------------------------------------------------------

    Y_train_augmented = np.repeat(
        Y[
            population
        ][
            train_indices
        ],
        AUGMENTATIONS_PER_ECG,
        axis=0
    ).astype(
        np.float32
    )


    if (
        len(
            X_train_augmented
        )
        !=
        len(
            Y_train_augmented
        )
    ):
        raise RuntimeError(
            "Augmented X/Y size mismatch."
        )


    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    )


    epoch_losses = []


    # ------------------------------------------------------------------
    # Deterministic custom loop
    # ------------------------------------------------------------------

    for epoch in range(
        EPOCHS
    ):

        permutation = epoch_permutations[
            epoch
        ]


        batch_losses = []


        for start in range(
            0,
            len(
                permutation
            ),
            BATCH_SIZE
        ):

            batch_idx = permutation[
                start:
                start + BATCH_SIZE
            ]


            xb = tf.convert_to_tensor(
                X_train_augmented[
                    batch_idx
                ],
                dtype=tf.float32
            )


            yb = tf.convert_to_tensor(
                Y_train_augmented[
                    batch_idx
                ],
                dtype=tf.float32
            )


            with tf.GradientTape() as tape:

                add3 = rep_model(
                    xb,
                    training=False
                )


                predicted_attention = (
                    attention_distribution_from_add3(
                        add3
                    )
                )


                loss = tf.reduce_mean(
                    tf.square(
                        predicted_attention
                        -
                        yb
                    )
                )


            gradients = tape.gradient(
                loss,
                train_vars
            )


            if any(
                gradient is None
                for gradient in gradients
            ):
                raise RuntimeError(
                    "At least one conv2d_10 gradient is None."
                )


            for gradient in gradients:

                if not np.all(
                    np.isfinite(
                        gradient.numpy()
                    )
                ):
                    raise RuntimeError(
                        "Non-finite gradient detected."
                    )


            optimizer.apply_gradients(
                zip(
                    gradients,
                    train_vars
                )
            )


            batch_losses.append(
                float(
                    loss.numpy()
                )
            )


        mean_epoch_loss = float(
            np.mean(
                batch_losses
            )
        )


        epoch_losses.append(
            mean_epoch_loss
        )


        print(
            f"    epoch {epoch + 1:02d}/{EPOCHS}: "
            f"loss={mean_epoch_loss:.8f}"
        )


    # ------------------------------------------------------------------
    # Held-out attention after training
    # ------------------------------------------------------------------

    trained_attention = predict_attention(
        rep_model,
        X_test
    )


    # ------------------------------------------------------------------
    # Confirm only conv2d_10 changed
    # ------------------------------------------------------------------

    weight_df = compare_layer_weights(
        before_weights,
        model
    )


    changed_layers = set(
        weight_df.loc[
            weight_df[
                "max_abs_change"
            ]
            >
            1e-12,
            "layer"
        ]
    )


    if changed_layers != {
        "conv2d_10"
    }:
        raise RuntimeError(
            "\nUnexpected changed layers.\n"
            "Expected: {'conv2d_10'}\n"
            f"Actual:   {changed_layers}"
        )


    print(
        "\n    ✓ Weight audit: ONLY conv2d_10 changed."
    )


    # ------------------------------------------------------------------
    # Save weights
    # ------------------------------------------------------------------

    weight_path = (
        MODEL_DIR
        /
        f"{population}_fold{fold_number}.weights.h5"
    )


    model.save_weights(
        weight_path
    )


    return {
        "baseline_attention":
            baseline_attention,

        "trained_attention":
            trained_attention,

        "Y_test":
            Y_test,

        "epoch_losses":
            epoch_losses,

        "weight_df":
            weight_df,

        "weight_path":
            weight_path,
    }


# ======================================================================================
# 14. RUN FIVE-FOLD EXPERIMENT
# ======================================================================================

metric_rows = []
prediction_rows = []
history_rows = []
weight_audit_rows = []


for fold in folds:

    fold_number = fold[
        "fold"
    ]


    train_indices = fold[
        "train_indices"
    ]


    test_indices = fold[
        "test_indices"
    ]


    print(
        "\n\n" +
        "#" * 100
    )

    print(
        f"FOLD {fold_number}"
    )

    print(
        "#" * 100
    )


    print(
        "Held-out ECGs:",
        [
            ECG_ORDER[i]
            for i in test_indices
        ]
    )


    # ------------------------------------------------------------------
    # Generate augmented X exactly once per fold
    # ------------------------------------------------------------------

    augmentation_rng = np.random.default_rng(
        SEED
        +
        10000
        *
        fold_number
    )


    X_train_base = np.repeat(
        X[
            train_indices
        ],
        AUGMENTATIONS_PER_ECG,
        axis=0
    ).astype(
        np.float32
    )


    noise = augmentation_rng.normal(
        loc=0.0,
        scale=NOISE_SD,
        size=X_train_base.shape
    ).astype(
        np.float32
    )


    X_train_augmented = (
        X_train_base
        +
        noise
    ).astype(
        np.float32
    )


    expected_training_samples = (
        len(
            train_indices
        )
        *
        AUGMENTATIONS_PER_ECG
    )


    if (
        X_train_augmented.shape[0]
        !=
        expected_training_samples
    ):
        raise RuntimeError(
            "Unexpected augmented training-set size."
        )


    # ------------------------------------------------------------------
    # Same epoch permutations for both populations
    # ------------------------------------------------------------------

    permutation_rng = np.random.default_rng(
        SEED
        +
        20000
        *
        fold_number
    )


    epoch_permutations = [
        permutation_rng.permutation(
            expected_training_samples
        )
        for _ in range(
            EPOCHS
        )
    ]


    # ------------------------------------------------------------------
    # Both populations
    # ------------------------------------------------------------------

    for population in POPULATIONS:

        print(
            "\n" +
            "=" * 100
        )

        print(
            f"FOLD {fold_number} — {population.upper()}"
        )

        print(
            "=" * 100
        )


        result = train_one_arm(
            population=population,
            fold_number=fold_number,
            train_indices=train_indices,
            test_indices=test_indices,
            X_train_augmented=X_train_augmented,
            epoch_permutations=epoch_permutations,
        )


        baseline_attention = result[
            "baseline_attention"
        ]


        trained_attention = result[
            "trained_attention"
        ]


        Y_test = result[
            "Y_test"
        ]


        # --------------------------------------------------------------
        # Training history
        # --------------------------------------------------------------

        for epoch_idx, loss_value in enumerate(
            result[
                "epoch_losses"
            ],
            start=1
        ):

            history_rows.append(
                {
                    "population":
                        population,

                    "fold":
                        fold_number,

                    "epoch":
                        epoch_idx,

                    "loss":
                        loss_value,
                }
            )


        # --------------------------------------------------------------
        # Weight audit
        # --------------------------------------------------------------

        temporary_weight_df = result[
            "weight_df"
        ].copy()


        temporary_weight_df[
            "population"
        ] = population


        temporary_weight_df[
            "fold"
        ] = fold_number


        weight_audit_rows.extend(
            temporary_weight_df.to_dict(
                "records"
            )
        )


        # --------------------------------------------------------------
        # Held-out ECGs
        # --------------------------------------------------------------

        for local_idx, ecg_index in enumerate(
            test_indices
        ):

            ecg = ECG_ORDER[
                ecg_index
            ]


            target = Y_test[
                local_idx
            ]


            base_pred = baseline_attention[
                local_idx
            ]


            trained_pred = trained_attention[
                local_idx
            ]


            base_cos = cosine_metric(
                base_pred,
                target
            )


            trained_cos = cosine_metric(
                trained_pred,
                target
            )


            base_mse = mse_metric(
                base_pred,
                target
            )


            trained_mse = mse_metric(
                trained_pred,
                target
            )


            base_jsd = jsd_metric(
                base_pred,
                target
            )


            trained_jsd = jsd_metric(
                trained_pred,
                target
            )


            metric_rows.append(
                {
                    "population":
                        population,

                    "fold":
                        fold_number,

                    "ECG":
                        ecg,

                    "baseline_cosine":
                        base_cos,

                    "trained_cosine":
                        trained_cos,

                    "delta_cosine":
                        trained_cos
                        -
                        base_cos,

                    "baseline_MSE":
                        base_mse,

                    "trained_MSE":
                        trained_mse,

                    "delta_MSE":
                        trained_mse
                        -
                        base_mse,

                    "baseline_JSD":
                        base_jsd,

                    "trained_JSD":
                        trained_jsd,

                    "delta_JSD":
                        trained_jsd
                        -
                        base_jsd,
                }
            )


            for lead_idx, lead in enumerate(
                LEAD_ORDER
            ):

                prediction_rows.append(
                    {
                        "population":
                            population,

                        "fold":
                            fold_number,

                        "ECG":
                            ecg,

                        "lead":
                            lead,

                        "target":
                            float(
                                target[
                                    lead_idx
                                ]
                            ),

                        "baseline":
                            float(
                                base_pred[
                                    lead_idx
                                ]
                            ),

                        "trained":
                            float(
                                trained_pred[
                                    lead_idx
                                ]
                            ),
                    }
                )


        gc.collect()


# ======================================================================================
# 15. CREATE / SAVE RAW OUTPUT TABLES
# ======================================================================================

metrics_df = pd.DataFrame(
    metric_rows
)


predictions_df = pd.DataFrame(
    prediction_rows
)


history_df = pd.DataFrame(
    history_rows
)


weight_audit_df = pd.DataFrame(
    weight_audit_rows
)


metrics_df.to_csv(
    STEP2_DIR
    /
    "heldout_metrics.csv",
    index=False
)


predictions_df.to_csv(
    STEP2_DIR
    /
    "heldout_attention_distributions.csv",
    index=False
)


history_df.to_csv(
    STEP2_DIR
    /
    "training_history.csv",
    index=False
)


weight_audit_df.to_csv(
    STEP2_DIR
    /
    "weight_change_audit.csv",
    index=False
)


# ======================================================================================
# 16. VERIFY HELD-OUT COVERAGE
# ======================================================================================

for population in POPULATIONS:

    temp = metrics_df[
        metrics_df[
            "population"
        ]
        ==
        population
    ]


    counts = temp[
        "ECG"
    ].value_counts()


    if len(counts) != 10:
        raise RuntimeError(
            f"{population}: not all 10 ECGs were evaluated."
        )


    if not (
        counts == 1
    ).all():
        raise RuntimeError(
            f"{population}: ECG holdout count error."
        )


print(
    "\n✓ Every ECG was held out exactly once for each population."
)


# ======================================================================================
# 17. WILCOXON
# ======================================================================================

def safe_wilcoxon(
    baseline,
    trained
):

    baseline = np.asarray(
        baseline,
        dtype=np.float64
    )


    trained = np.asarray(
        trained,
        dtype=np.float64
    )


    differences = (
        trained
        -
        baseline
    )


    if np.allclose(
        differences,
        0
    ):
        return (
            0.0,
            1.0
        )


    result = wilcoxon(
        trained,
        baseline,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto"
    )


    return (
        float(
            result.statistic
        ),
        float(
            result.pvalue
        )
    )


# ======================================================================================
# 18. SUMMARY STATISTICS
# ======================================================================================

summary_rows = []


for population in POPULATIONS:

    temp = metrics_df[
        metrics_df[
            "population"
        ]
        ==
        population
    ].copy()


    # ------------------------------------------------------------------
    # COSINE — higher is better
    # ------------------------------------------------------------------

    statistic, p_value = safe_wilcoxon(
        temp[
            "baseline_cosine"
        ],
        temp[
            "trained_cosine"
        ]
    )


    summary_rows.append(
        {
            "population":
                population,

            "metric":
                "cosine",

            "direction_better":
                "higher",

            "baseline_mean":
                temp[
                    "baseline_cosine"
                ].mean(),

            "trained_mean":
                temp[
                    "trained_cosine"
                ].mean(),

            "mean_delta":
                temp[
                    "delta_cosine"
                ].mean(),

            "median_delta":
                temp[
                    "delta_cosine"
                ].median(),

            "n_improved":
                int(
                    (
                        temp[
                            "delta_cosine"
                        ]
                        >
                        0
                    ).sum()
                ),

            "wilcoxon_statistic":
                statistic,

            "wilcoxon_p":
                p_value,
        }
    )


    # ------------------------------------------------------------------
    # MSE — lower is better
    # ------------------------------------------------------------------

    statistic, p_value = safe_wilcoxon(
        temp[
            "baseline_MSE"
        ],
        temp[
            "trained_MSE"
        ]
    )


    summary_rows.append(
        {
            "population":
                population,

            "metric":
                "MSE",

            "direction_better":
                "lower",

            "baseline_mean":
                temp[
                    "baseline_MSE"
                ].mean(),

            "trained_mean":
                temp[
                    "trained_MSE"
                ].mean(),

            "mean_delta":
                temp[
                    "delta_MSE"
                ].mean(),

            "median_delta":
                temp[
                    "delta_MSE"
                ].median(),

            "n_improved":
                int(
                    (
                        temp[
                            "delta_MSE"
                        ]
                        <
                        0
                    ).sum()
                ),

            "wilcoxon_statistic":
                statistic,

            "wilcoxon_p":
                p_value,
        }
    )


    # ------------------------------------------------------------------
    # JSD — lower is better
    # ------------------------------------------------------------------

    statistic, p_value = safe_wilcoxon(
        temp[
            "baseline_JSD"
        ],
        temp[
            "trained_JSD"
        ]
    )


    summary_rows.append(
        {
            "population":
                population,

            "metric":
                "JSD",

            "direction_better":
                "lower",

            "baseline_mean":
                temp[
                    "baseline_JSD"
                ].mean(),

            "trained_mean":
                temp[
                    "trained_JSD"
                ].mean(),

            "mean_delta":
                temp[
                    "delta_JSD"
                ].mean(),

            "median_delta":
                temp[
                    "delta_JSD"
                ].median(),

            "n_improved":
                int(
                    (
                        temp[
                            "delta_JSD"
                        ]
                        <
                        0
                    ).sum()
                ),

            "wilcoxon_statistic":
                statistic,

            "wilcoxon_p":
                p_value,
        }
    )


summary_df = pd.DataFrame(
    summary_rows
)


summary_df.to_csv(
    STEP2_DIR
    /
    "alignment_summary.csv",
    index=False
)


# ======================================================================================
# 19. PRINT HELD-OUT RESULTS
# ======================================================================================

print(
    "\n\n" +
    "=" * 100
)

print(
    "HELD-OUT ECG RESULTS"
)

print(
    "=" * 100
)


print(
    metrics_df[
        [
            "population",
            "fold",
            "ECG",
            "baseline_cosine",
            "trained_cosine",
            "delta_cosine",
            "baseline_MSE",
            "trained_MSE",
            "delta_MSE",
            "baseline_JSD",
            "trained_JSD",
            "delta_JSD",
        ]
    ]
    .round(
        7
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 20. PRINT PRIMARY SUMMARY
# ======================================================================================

print(
    "\n\n" +
    "=" * 100
)

print(
    "DEFINITIVE HELD-OUT GAZE-ALIGNMENT SUMMARY"
)

print(
    "=" * 100
)


print(
    summary_df
    .round(
        7
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 21. FINAL WEIGHT AUDIT
# ======================================================================================

changed_summary = (
    weight_audit_df[
        weight_audit_df[
            "max_abs_change"
        ]
        >
        1e-12
    ][
        [
            "population",
            "fold",
            "layer",
            "max_abs_change",
        ]
    ]
)


print(
    "\n\n" +
    "=" * 100
)

print(
    "LAYERS THAT CHANGED"
)

print(
    "=" * 100
)


print(
    changed_summary.to_string(
        index=False
    )
)


if set(
    changed_summary[
        "layer"
    ].unique()
) != {
    "conv2d_10"
}:
    raise RuntimeError(
        "Final weight audit failed."
    )


if len(
    changed_summary
) != (
    N_SPLITS
    *
    len(
        POPULATIONS
    )
):
    raise RuntimeError(
        "Expected exactly one changed layer per population × fold."
    )


print(
    "\n✓ Every fold/arm changed ONLY conv2d_10."
)


# ======================================================================================
# 22. SAVE STEP-2 MANIFEST
# ======================================================================================

step2_manifest = {

    "experiment":
        "Definitive clean gaze-aligned ECG experiment",

    "stage":
        "Step 2: five-fold held-out ECG gaze alignment",

    "seed":
        SEED,

    "n_folds":
        N_SPLITS,

    "held_out_ECGs_per_fold":
        2,

    "training_ECGs_per_fold":
        8,

    "attention_readout":
        (
            "RMS activation energy from add_3: "
            "sqrt(mean(add_3^2)) across temporal and channel axes, "
            "then normalised across 12 leads."
        ),

    "attention_layer":
        "add_3",

    "trainable_layer":
        "conv2d_10",

    "batchnorm_mode":
        "frozen",

    "forward_training_mode":
        False,

    "loss":
        "MSE between 12-lead RMS attention distribution and gaze target",

    "optimizer":
        "Adam",

    "learning_rate":
        LEARNING_RATE,

    "epochs":
        EPOCHS,

    "augmentations_per_training_ECG":
        AUGMENTATIONS_PER_ECG,

    "gaussian_noise_sigma":
        NOISE_SD,

    "batch_size":
        BATCH_SIZE,

    "same_folds_between_populations":
        True,

    "same_noisy_training_examples_between_populations":
        True,

    "same_batch_order_between_populations":
        True,

    "populations": {
        "consultant":
            9,

        "all_clinicians":
            20,
    },

    "frozen_V9_sha256":
        current_x_hash,

    "frozen_gaze_target_sha256":
        current_target_hash,

    "base_model_sha256":
        current_base_hash,

    "statistical_unit":
        "held-out ECG",

    "n_statistical_units_per_population":
        10,

    "statistical_test":
        "two-sided paired Wilcoxon signed-rank test",

    "metrics": [
        "cosine similarity",
        "mean squared error",
        "Jensen-Shannon divergence",
    ],
}


with open(
    STEP2_DIR
    /
    "step2_manifest.json",
    "w"
) as f:

    json.dump(
        step2_manifest,
        f,
        indent=2
    )


print(
    "\n\n" +
    "=" * 100
)

print(
    "STEP 2 COMPLETE"
)

print(
    "=" * 100
)


print(
    """
Please paste back:

1. FROZEN 5-FOLD SPLIT

2. DEFINITIVE HELD-OUT GAZE-ALIGNMENT SUMMARY

3. LAYERS THAT CHANGED

You do not need to paste all epoch losses unless there is an error.
"""
)

In [ ]:
# ======================================================================================
# STEP 2B — CONSULTANT vs ALL-CLINICIAN GAZE SUPERVISION
#
# NO TRAINING.
#
# Question:
# Does the CHANGE in held-out gaze alignment differ between the
# consultant-only and all-clinician targets?
#
# Statistical unit = ECG, n = 10 paired observations.
# ======================================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon


RESULT_PATH = Path(
    "definitive_clean_gaze/step2_cv/heldout_metrics.csv"
)

if not RESULT_PATH.exists():
    raise FileNotFoundError(RESULT_PATH)


df = pd.read_csv(
    RESULT_PATH
)


# ======================================================================================
# 1. PIVOT SO EACH ECG HAS BOTH POPULATIONS
# ======================================================================================

consultant = (
    df[
        df["population"]
        ==
        "consultant"
    ]
    .set_index("ECG")
    .sort_index()
)


all_clinicians = (
    df[
        df["population"]
        ==
        "all_clinicians"
    ]
    .set_index("ECG")
    .sort_index()
)


if list(consultant.index) != list(all_clinicians.index):
    raise RuntimeError(
        "ECGs do not align between populations."
    )


if len(consultant) != 10:
    raise RuntimeError(
        f"Expected 10 paired ECGs, found {len(consultant)}."
    )


# ======================================================================================
# 2. EXPRESS CHANGE SO POSITIVE ALWAYS MEANS IMPROVEMENT
#
# cosine:
#     trained - baseline
#
# MSE/JSD:
#     baseline - trained
#     because LOWER is better
# ======================================================================================

comparison = pd.DataFrame(
    index=consultant.index
)


comparison["consultant_cosine_improvement"] = (
    consultant["trained_cosine"]
    -
    consultant["baseline_cosine"]
)


comparison["all_clinicians_cosine_improvement"] = (
    all_clinicians["trained_cosine"]
    -
    all_clinicians["baseline_cosine"]
)


comparison["consultant_MSE_improvement"] = (
    consultant["baseline_MSE"]
    -
    consultant["trained_MSE"]
)


comparison["all_clinicians_MSE_improvement"] = (
    all_clinicians["baseline_MSE"]
    -
    all_clinicians["trained_MSE"]
)


comparison["consultant_JSD_improvement"] = (
    consultant["baseline_JSD"]
    -
    consultant["trained_JSD"]
)


comparison["all_clinicians_JSD_improvement"] = (
    all_clinicians["baseline_JSD"]
    -
    all_clinicians["trained_JSD"]
)


# Positive advantage means all-clinician supervision improved more.
comparison["all_minus_consultant_cosine"] = (
    comparison["all_clinicians_cosine_improvement"]
    -
    comparison["consultant_cosine_improvement"]
)


comparison["all_minus_consultant_MSE"] = (
    comparison["all_clinicians_MSE_improvement"]
    -
    comparison["consultant_MSE_improvement"]
)


comparison["all_minus_consultant_JSD"] = (
    comparison["all_clinicians_JSD_improvement"]
    -
    comparison["consultant_JSD_improvement"]
)


comparison = comparison.reset_index()


print("=" * 100)
print("PER-ECG POPULATION COMPARISON")
print("=" * 100)

print(
    comparison[
        [
            "ECG",
            "all_minus_consultant_cosine",
            "all_minus_consultant_MSE",
            "all_minus_consultant_JSD",
        ]
    ]
    .round(7)
    .to_string(index=False)
)


# ======================================================================================
# 3. PAIRED WILCOXON TESTS
# ======================================================================================

summary_rows = []


metric_definitions = [

    (
        "cosine",
        "consultant_cosine_improvement",
        "all_clinicians_cosine_improvement",
    ),

    (
        "MSE",
        "consultant_MSE_improvement",
        "all_clinicians_MSE_improvement",
    ),

    (
        "JSD",
        "consultant_JSD_improvement",
        "all_clinicians_JSD_improvement",
    ),
]


for metric, consultant_col, all_col in metric_definitions:

    c = comparison[
        consultant_col
    ].to_numpy(
        dtype=float
    )


    a = comparison[
        all_col
    ].to_numpy(
        dtype=float
    )


    result = wilcoxon(
        a,
        c,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto"
    )


    difference = (
        a - c
    )


    summary_rows.append(
        {
            "metric":
                metric,

            "consultant_mean_improvement":
                float(
                    np.mean(c)
                ),

            "all_clinician_mean_improvement":
                float(
                    np.mean(a)
                ),

            "all_minus_consultant_mean":
                float(
                    np.mean(
                        difference
                    )
                ),

            "all_minus_consultant_median":
                float(
                    np.median(
                        difference
                    )
                ),

            "ECGs_favouring_all_clinicians":
                int(
                    np.sum(
                        difference > 0
                    )
                ),

            "ECGs_favouring_consultants":
                int(
                    np.sum(
                        difference < 0
                    )
                ),

            "wilcoxon_statistic":
                float(
                    result.statistic
                ),

            "wilcoxon_p":
                float(
                    result.pvalue
                ),
        }
    )


summary = pd.DataFrame(
    summary_rows
)


print(
    "\n" +
    "=" * 100
)

print(
    "CONSULTANT vs ALL-CLINICIAN SUPERVISION SUMMARY"
)

print(
    "=" * 100
)


print(
    summary
    .round(7)
    .to_string(index=False)
)


# ======================================================================================
# 4. SAVE
# ======================================================================================

comparison.to_csv(
    "definitive_clean_gaze/step2_cv/"
    "population_improvement_by_ECG.csv",
    index=False
)


summary.to_csv(
    "definitive_clean_gaze/step2_cv/"
    "population_improvement_comparison.csv",
    index=False
)


print(
    "\n✓ Population comparison saved."
)

print(
    "\nNo model weights were changed."
)

In [ ]:
# ======================================================================================
# STEP 3 — TRAIN FINAL FULL-DATA GAZE MODELS
#
# Uses ALL 10 gaze ECGs.
#
# Produces:
#
#   definitive_clean_gaze/final_models/
#       clean_consultant.weights.h5
#       clean_all_clinicians.weights.h5
#
# IMPORTANT:
#   - same frozen V9 input
#   - same frozen gaze targets
#   - same RMS add_3 attention readout
#   - only conv2d_10 trainable
#   - BatchNorm frozen
#   - training=False for every forward pass
#   - gaze-only MSE
#   - Adam 1e-4
#   - 25 epochs
#   - 40 noisy copies per ECG
#   - sigma 0.03
#   - batch size 16
#
# Both populations receive IDENTICAL:
#   - starting weights
#   - augmented ECGs
#   - epoch/batch ordering
#
# Only the gaze target differs.
#
# These models are for subsequent downstream diagnostic evaluation.
# ======================================================================================

import gc
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# ======================================================================================
# 1. PARAMETERS — MUST MATCH STEP 2
# ======================================================================================

SEED = 2026

AUGMENTATIONS_PER_ECG = 40
NOISE_SD = 0.03

BATCH_SIZE = 16
EPOCHS = 25

LEARNING_RATE = 1e-4
RMS_EPSILON = 1e-8


ECG_ORDER = [
    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]


LEAD_ORDER = [
    "I",
    "II",
    "III",
    "aVR",
    "aVL",
    "aVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]


POPULATIONS = [
    "consultant",
    "all_clinicians",
]


# ======================================================================================
# 2. PATHS
# ======================================================================================

ROOT = Path(
    "definitive_clean_gaze"
)


X_PATH = Path(
    "final_digitisation_v9/X_gaze_final.npy"
)


TARGET_PATH = (
    ROOT
    /
    "clean_gaze_targets.npz"
)


STEP1_MANIFEST_PATH = (
    ROOT
    /
    "step1_manifest.json"
)


STEP2_MANIFEST_PATH = (
    ROOT
    /
    "step2_cv"
    /
    "step2_manifest.json"
)


FINAL_DIR = (
    ROOT
    /
    "final_models"
)


FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for path in [
    X_PATH,
    TARGET_PATH,
    STEP1_MANIFEST_PATH,
    STEP2_MANIFEST_PATH,
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


# ======================================================================================
# 3. HASH HELPER
# ======================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# ======================================================================================
# 4. LOAD MANIFESTS + VERIFY FROZEN INPUTS
# ======================================================================================

with open(
    STEP1_MANIFEST_PATH,
    "r"
) as f:

    step1_manifest = json.load(
        f
    )


with open(
    STEP2_MANIFEST_PATH,
    "r"
) as f:

    step2_manifest = json.load(
        f
    )


BASE_MODEL_PATH = Path(
    step1_manifest[
        "base_model"
    ]
)


if not BASE_MODEL_PATH.exists():

    raise FileNotFoundError(
        BASE_MODEL_PATH
    )


x_hash = sha256_file(
    X_PATH
)


target_hash = sha256_file(
    TARGET_PATH
)


base_hash = sha256_file(
    BASE_MODEL_PATH
)


print("=" * 100)
print("FROZEN INPUT VERIFICATION")
print("=" * 100)


print(
    "V9 ECG:",
    x_hash
)


print(
    "Targets:",
    target_hash
)


print(
    "Base model:",
    base_hash
)


if (
    x_hash
    !=
    step1_manifest[
        "frozen_ecg_sha256"
    ]
):

    raise RuntimeError(
        "V9 ECG hash mismatch."
    )


if (
    target_hash
    !=
    step1_manifest[
        "clean_gaze_targets_sha256"
    ]
):

    raise RuntimeError(
        "Gaze target hash mismatch."
    )


if (
    base_hash
    !=
    step1_manifest[
        "base_model_sha256"
    ]
):

    raise RuntimeError(
        "Base model hash mismatch."
    )


print(
    "\n✓ Frozen Step-1 inputs verified."
)


# ======================================================================================
# 5. VERIFY STEP-2 TRAINING DEFINITION
# ======================================================================================

required_step2 = {

    "attention_layer":
        "add_3",

    "trainable_layer":
        "conv2d_10",

    "epochs":
        25,

    "augmentations_per_training_ECG":
        40,

    "gaussian_noise_sigma":
        0.03,

    "batch_size":
        16,
}


for key, expected in required_step2.items():

    actual = step2_manifest[
        key
    ]

    if actual != expected:

        raise RuntimeError(
            f"Step-2 definition mismatch for {key}: "
            f"{actual} != {expected}"
        )


print(
    "✓ Step-3 configuration matches frozen Step-2 definition."
)


# ======================================================================================
# 6. LOAD ARRAYS
# ======================================================================================

X = np.load(
    X_PATH
).astype(
    np.float32
)


target_data = np.load(
    TARGET_PATH
)


Y = {

    "consultant":
        target_data[
            "consultant"
        ].astype(
            np.float32
        ),

    "all_clinicians":
        target_data[
            "all_clinicians"
        ].astype(
            np.float32
        ),
}


if X.shape != (
    10,
    12,
    1000,
    1
):

    raise RuntimeError(
        f"Bad X shape: {X.shape}"
    )


for population in POPULATIONS:

    if Y[
        population
    ].shape != (
        10,
        12
    ):

        raise RuntimeError(
            f"Bad target shape: {population}"
        )


# ======================================================================================
# 7. FROZEN RMS ATTENTION READOUT
# ======================================================================================

def attention_distribution_from_add3(
    add3_tensor
):

    rms = tf.sqrt(

        tf.reduce_mean(
            tf.square(
                add3_tensor
            ),
            axis=[
                2,
                3
            ]
        )

        +

        RMS_EPSILON
    )


    return (
        rms
        /
        (
            tf.reduce_sum(
                rms,
                axis=1,
                keepdims=True
            )
            +
            RMS_EPSILON
        )
    )


# ======================================================================================
# 8. LOAD FRESH BASE MODEL
# ======================================================================================

def load_fresh_model():

    tf.keras.backend.clear_session()

    tf.random.set_seed(
        SEED
    )


    model = tf.keras.models.load_model(
        BASE_MODEL_PATH,
        compile=False
    )


    # Freeze everything.
    for layer in model.layers:

        layer.trainable = False


    conv10 = model.get_layer(
        "conv2d_10"
    )


    conv10.trainable = True


    add3 = model.get_layer(
        "add_3"
    )


    rep_model = tf.keras.Model(
        inputs=model.input,
        outputs=add3.output
    )


    train_vars = list(
        conv10.trainable_variables
    )


    if len(
        train_vars
    ) == 0:

        raise RuntimeError(
            "conv2d_10 has no trainable variables."
        )


    return (
        model,
        rep_model,
        train_vars
    )


# ======================================================================================
# 9. WEIGHT AUDIT HELPERS
# ======================================================================================

def snapshot_weights(
    model
):

    return {

        layer.name: [
            np.array(
                w,
                copy=True
            )
            for w
            in layer.get_weights()
        ]

        for layer
        in model.layers
    }


def changed_layers(
    before,
    model
):

    rows = []


    for layer in model.layers:

        old = before[
            layer.name
        ]


        new = layer.get_weights()


        max_change = 0.0


        for a, b in zip(
            old,
            new
        ):

            if a.shape != b.shape:

                raise RuntimeError(
                    f"Shape mismatch: {layer.name}"
                )


            if a.size:

                max_change = max(

                    max_change,

                    float(
                        np.max(
                            np.abs(
                                a - b
                            )
                        )
                    )
                )


        rows.append(
            {
                "layer":
                    layer.name,

                "max_abs_change":
                    max_change,
            }
        )


    return pd.DataFrame(
        rows
    )


# ======================================================================================
# 10. CREATE IDENTICAL FULL-DATA AUGMENTATION ONCE
# ======================================================================================

augmentation_rng = np.random.default_rng(
    SEED + 500000
)


X_train_base = np.repeat(

    X,

    AUGMENTATIONS_PER_ECG,

    axis=0

).astype(
    np.float32
)


noise = augmentation_rng.normal(

    loc=0.0,

    scale=NOISE_SD,

    size=X_train_base.shape

).astype(
    np.float32
)


X_train_augmented = (
    X_train_base
    +
    noise
).astype(
    np.float32
)


if X_train_augmented.shape != (
    400,
    12,
    1000,
    1
):

    raise RuntimeError(
        f"Unexpected augmented X shape: {X_train_augmented.shape}"
    )


print("\n" + "=" * 100)
print("FULL-DATA TRAINING SET")
print("=" * 100)

print(
    "Training examples:",
    X_train_augmented.shape[0]
)

print(
    "40 augmentations × 10 ECGs = 400 confirmed."
)


# ======================================================================================
# 11. IDENTICAL EPOCH/BATCH ORDER FOR BOTH POPULATIONS
# ======================================================================================

permutation_rng = np.random.default_rng(
    SEED + 600000
)


epoch_permutations = [

    permutation_rng.permutation(
        400
    )

    for _
    in range(
        EPOCHS
    )
]


# ======================================================================================
# 12. TRAIN FINAL MODELS
# ======================================================================================

history_rows = []

weight_audit_rows = []

final_alignment_rows = []

final_weight_hashes = {}


for population in POPULATIONS:

    print(
        "\n\n" +
        "#" * 100
    )

    print(
        f"FINAL MODEL — {population.upper()}"
    )

    print(
        "#" * 100
    )


    (
        model,
        rep_model,
        train_vars
    ) = load_fresh_model()


    before = snapshot_weights(
        model
    )


    # ------------------------------------------------------------------
    # Target augmentation
    # ------------------------------------------------------------------

    Y_augmented = np.repeat(

        Y[
            population
        ],

        AUGMENTATIONS_PER_ECG,

        axis=0

    ).astype(
        np.float32
    )


    if Y_augmented.shape != (
        400,
        12
    ):

        raise RuntimeError(
            f"Unexpected Y shape for {population}: "
            f"{Y_augmented.shape}"
        )


    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    )


    # ------------------------------------------------------------------
    # Training loop
    # ------------------------------------------------------------------

    for epoch in range(
        EPOCHS
    ):

        permutation = epoch_permutations[
            epoch
        ]


        losses = []


        for start in range(
            0,
            400,
            BATCH_SIZE
        ):

            idx = permutation[
                start:
                start + BATCH_SIZE
            ]


            xb = tf.convert_to_tensor(
                X_train_augmented[
                    idx
                ],
                dtype=tf.float32
            )


            yb = tf.convert_to_tensor(
                Y_augmented[
                    idx
                ],
                dtype=tf.float32
            )


            with tf.GradientTape() as tape:

                add3 = rep_model(
                    xb,
                    training=False
                )


                pred = (
                    attention_distribution_from_add3(
                        add3
                    )
                )


                loss = tf.reduce_mean(
                    tf.square(
                        pred - yb
                    )
                )


            gradients = tape.gradient(
                loss,
                train_vars
            )


            if any(
                g is None
                for g
                in gradients
            ):

                raise RuntimeError(
                    "None gradient detected."
                )


            for gradient in gradients:

                if not np.all(
                    np.isfinite(
                        gradient.numpy()
                    )
                ):

                    raise RuntimeError(
                        "Non-finite gradient detected."
                    )


            optimizer.apply_gradients(
                zip(
                    gradients,
                    train_vars
                )
            )


            losses.append(
                float(
                    loss.numpy()
                )
            )


        mean_loss = float(
            np.mean(
                losses
            )
        )


        history_rows.append(
            {
                "population":
                    population,

                "epoch":
                    epoch + 1,

                "loss":
                    mean_loss,
            }
        )


        print(
            f"epoch {epoch + 1:02d}/{EPOCHS}: "
            f"loss={mean_loss:.8f}"
        )


    # ------------------------------------------------------------------
    # Weight-change audit
    # ------------------------------------------------------------------

    audit = changed_layers(
        before,
        model
    )


    changed = set(
        audit.loc[
            audit[
                "max_abs_change"
            ]
            >
            1e-12,
            "layer"
        ]
    )


    if changed != {
        "conv2d_10"
    }:

        raise RuntimeError(
            f"{population}: unexpected changed layers: {changed}"
        )


    print(
        "\n✓ ONLY conv2d_10 changed."
    )


    audit[
        "population"
    ] = population


    weight_audit_rows.extend(
        audit.to_dict(
            "records"
        )
    )


    # ------------------------------------------------------------------
    # Training-set alignment QC
    #
    # This is IN-SAMPLE and must not be interpreted as validation.
    # ------------------------------------------------------------------

    final_attention = (
        attention_distribution_from_add3(
            rep_model(
                X,
                training=False
            )
        )
        .numpy()
    )


    for i, ecg in enumerate(
        ECG_ORDER
    ):

        target = Y[
            population
        ][i]


        prediction = final_attention[
            i
        ]


        cosine = float(
            np.dot(
                target,
                prediction
            )
            /
            (
                np.linalg.norm(
                    target
                )
                *
                np.linalg.norm(
                    prediction
                )
            )
        )


        mse = float(
            np.mean(
                np.square(
                    target
                    -
                    prediction
                )
            )
        )


        final_alignment_rows.append(
            {
                "population":
                    population,

                "ECG":
                    ecg,

                "cosine":
                    cosine,

                "MSE":
                    mse,

                "interpretation":
                    "in-sample training QC only",
            }
        )


    # ------------------------------------------------------------------
    # Save final weights
    # ------------------------------------------------------------------

    if population == "consultant":

        weight_path = (
            FINAL_DIR
            /
            "clean_consultant.weights.h5"
        )

    else:

        weight_path = (
            FINAL_DIR
            /
            "clean_all_clinicians.weights.h5"
        )


    model.save_weights(
        weight_path
    )


    final_weight_hashes[
        population
    ] = sha256_file(
        weight_path
    )


    print(
        "\nSaved:"
    )

    print(
        weight_path
    )


    print(
        "SHA256:"
    )

    print(
        final_weight_hashes[
            population
        ]
    )


    gc.collect()


# ======================================================================================
# 13. SAVE AUDITS
# ======================================================================================

history_df = pd.DataFrame(
    history_rows
)


weight_audit_df = pd.DataFrame(
    weight_audit_rows
)


alignment_df = pd.DataFrame(
    final_alignment_rows
)


history_df.to_csv(

    FINAL_DIR
    /
    "full_data_training_history.csv",

    index=False
)


weight_audit_df.to_csv(

    FINAL_DIR
    /
    "full_data_weight_audit.csv",

    index=False
)


alignment_df.to_csv(

    FINAL_DIR
    /
    "full_data_alignment_QC.csv",

    index=False
)


# ======================================================================================
# 14. PRINT TRAINING-QC SUMMARY
# ======================================================================================

print(
    "\n\n" +
    "=" * 100
)

print(
    "FINAL FULL-DATA MODEL QC"
)

print(
    "=" * 100
)


qc_summary = (

    alignment_df

    .groupby(
        "population"
    )

    .agg(
        mean_cosine=(
            "cosine",
            "mean"
        ),

        min_cosine=(
            "cosine",
            "min"
        ),

        mean_MSE=(
            "MSE",
            "mean"
        ),
    )

    .reset_index()
)


print(
    qc_summary
    .round(
        7
    )
    .to_string(
        index=False
    )
)


print(
    "\nNOTE: These are IN-SAMPLE full-data training QC values."
)

print(
    "The valid generalisation estimate remains the 5-fold held-out results from Step 2."
)


# ======================================================================================
# 15. CHANGED-LAYER SUMMARY
# ======================================================================================

changed_summary = (

    weight_audit_df[
        weight_audit_df[
            "max_abs_change"
        ]
        >
        1e-12
    ]

    [
        [
            "population",
            "layer",
            "max_abs_change"
        ]
    ]
)


print(
    "\n" +
    "=" * 100
)

print(
    "FINAL MODEL LAYERS THAT CHANGED"
)

print(
    "=" * 100
)


print(
    changed_summary.to_string(
        index=False
    )
)


# ======================================================================================
# 16. FINAL-MODEL MANIFEST
# ======================================================================================

manifest = {

    "experiment":
        "Definitive clean gaze-aligned ECG experiment",

    "stage":
        "Step 3: final full-data gaze models",

    "training_ECG_count":
        10,

    "attention_readout":
        (
            "RMS activation energy from add_3 across "
            "temporal and channel axes, normalised across 12 leads."
        ),

    "trainable_layer":
        "conv2d_10",

    "loss":
        "gaze-only mean squared error",

    "epochs":
        EPOCHS,

    "augmentations_per_ECG":
        AUGMENTATIONS_PER_ECG,

    "gaussian_noise_sigma":
        NOISE_SD,

    "batch_size":
        BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "same_augmented_inputs_between_populations":
        True,

    "same_batch_order_between_populations":
        True,

    "consultant_weights":
        str(
            FINAL_DIR
            /
            "clean_consultant.weights.h5"
        ),

    "consultant_weights_sha256":
        final_weight_hashes[
            "consultant"
        ],

    "all_clinician_weights":
        str(
            FINAL_DIR
            /
            "clean_all_clinicians.weights.h5"
        ),

    "all_clinician_weights_sha256":
        final_weight_hashes[
            "all_clinicians"
        ],

    "frozen_V9_sha256":
        x_hash,

    "frozen_target_sha256":
        target_hash,

    "base_model_sha256":
        base_hash,

    "generalisation_results_source":
        (
            "Step 2 five-fold held-out-ECG evaluation; "
            "full-data alignment values are training QC only."
        ),
}


with open(

    FINAL_DIR
    /
    "final_model_manifest.json",

    "w"

) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ======================================================================================
# 17. FINISH
# ======================================================================================

print(
    "\n\n" +
    "=" * 100
)

print(
    "STEP 3 COMPLETE"
)

print(
    "=" * 100
)


print(
    """
Please paste back only:

1. FINAL FULL-DATA MODEL QC

2. FINAL MODEL LAYERS THAT CHANGED

3. The SHA256 values printed for:
       clean_consultant.weights.h5
       clean_all_clinicians.weights.h5

After that we freeze the two final gaze models and move to
downstream diagnostic evaluation.
"""
)

In [ ]:
# ======================================================================================
# FREEZE FINAL MODEL HASHES
# No training.
# ======================================================================================

import hashlib
import json
from pathlib import Path


FINAL_DIR = Path(
    "definitive_clean_gaze/final_models"
)

CONSULTANT_PATH = (
    FINAL_DIR
    /
    "clean_consultant.weights.h5"
)

ALL_PATH = (
    FINAL_DIR
    /
    "clean_all_clinicians.weights.h5"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


consultant_hash = sha256_file(
    CONSULTANT_PATH
)

all_hash = sha256_file(
    ALL_PATH
)


print("=" * 100)
print("FROZEN FINAL GAZE MODELS")
print("=" * 100)

print("\nConsultant:")
print(CONSULTANT_PATH)
print(consultant_hash)

print("\nAll clinicians:")
print(ALL_PATH)
print(all_hash)


# Compare with the manifest generated during Step 3.
manifest_path = (
    FINAL_DIR
    /
    "final_model_manifest.json"
)

with open(
    manifest_path,
    "r"
) as f:

    manifest = json.load(f)


assert (
    consultant_hash
    ==
    manifest["consultant_weights_sha256"]
)

assert (
    all_hash
    ==
    manifest["all_clinician_weights_sha256"]
)


print(
    "\n✓ Both hashes match the Step-3 manifest."
)

print(
    "✓ Final gaze models are now frozen."
)

In [ ]:
# ======================================================================================
# STEP 4A — PTB-XL PROVENANCE + LABEL + SPLIT AUDIT
#
# READ ONLY.
# NO MODEL INFERENCE.
# NO TRAINING.
#
# Goals:
#   1. Locate the local PTB-XL dataset deterministically.
#   2. Audit record/fold counts.
#   3. Reconstruct the source repository's 23 diagnostic-subclass labels.
#   4. Reconstruct the MultiLabelBinarizer class order.
#   5. Confirm fold 10 is suitable as the source-model held-out test fold.
#   6. Confirm patient separation between fold 10 and folds 1–9.
#   7. Verify raw 100-Hz waveform shape and lead order.
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer


# ======================================================================================
# 1. EXACT EXPECTED MODEL OUTPUT ORDER
#
# This should be reproduced by MultiLabelBinarizer fitted on folds != 10.
# ======================================================================================

EXPECTED_MODEL_CLASSES = [
    "AMI",
    "CLBBB",
    "CRBBB",
    "ILBBB",
    "IMI",
    "IRBBB",
    "ISCA",
    "ISCI",
    "ISC_",
    "IVCD",
    "LAFB/LPFB",
    "LAO/LAE",
    "LMI",
    "LVH",
    "NORM",
    "NST_",
    "PMI",
    "RAO/RAE",
    "RVH",
    "SEHYP",
    "STTC",
    "WPW",
    "_AVB",
]


EXPECTED_LEADS = [
    "I",
    "II",
    "III",
    "AVR",
    "AVL",
    "AVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]


# ======================================================================================
# 2. LOCATE PTB-XL ROOT
#
# We require ptbxl_database.csv and scp_statements.csv in the same directory.
# ======================================================================================

database_candidates = list(
    Path(".").glob(
        "**/ptbxl_database.csv"
    )
)


roots = []


for database_path in database_candidates:

    root = database_path.parent

    if (
        root
        /
        "scp_statements.csv"
    ).exists():

        roots.append(
            root
        )


roots = sorted(
    set(
        roots
    ),
    key=lambda p: str(p)
)


print("=" * 100)
print("PTB-XL DATASET SEARCH")
print("=" * 100)


if len(roots) == 0:

    raise FileNotFoundError(
        "\nCould not find a directory containing BOTH:\n"
        "  ptbxl_database.csv\n"
        "  scp_statements.csv\n\n"
        "Place/unzip PTB-XL somewhere below the current working directory."
    )


print(
    "Candidate PTB-XL roots:"
)


for i, root in enumerate(
    roots,
    start=1
):

    print(
        f"  {i}. {root}"
    )


if len(roots) != 1:

    raise RuntimeError(
        "\nMore than one PTB-XL dataset was found. "
        "Do not automatically choose between different releases/copies."
    )


PTBXL_ROOT = roots[0]


DATABASE_PATH = (
    PTBXL_ROOT
    /
    "ptbxl_database.csv"
)


SCP_PATH = (
    PTBXL_ROOT
    /
    "scp_statements.csv"
)


print(
    "\nSelected root:"
)

print(
    PTBXL_ROOT
)


# ======================================================================================
# 3. LOAD METADATA
# ======================================================================================

metadata = pd.read_csv(
    DATABASE_PATH,
    index_col="ecg_id"
)


scp = pd.read_csv(
    SCP_PATH,
    index_col=0
)


print("\n" + "=" * 100)
print("PTB-XL METADATA")
print("=" * 100)


print(
    "ptbxl_database shape:",
    metadata.shape
)


print(
    "scp_statements shape:",
    scp.shape
)


print(
    "ECG IDs unique:",
    metadata.index.is_unique
)


required_metadata_columns = [
    "patient_id",
    "scp_codes",
    "strat_fold",
    "filename_lr",
]


missing = [
    column
    for column in required_metadata_columns
    if column not in metadata.columns
]


if missing:

    raise RuntimeError(
        f"Missing PTB-XL metadata columns: {missing}"
    )


required_scp_columns = [
    "diagnostic",
    "diagnostic_subclass",
]


missing_scp = [
    column
    for column in required_scp_columns
    if column not in scp.columns
]


if missing_scp:

    raise RuntimeError(
        f"Missing SCP columns: {missing_scp}"
    )


# ======================================================================================
# 4. AUDIT STRATIFIED FOLDS
# ======================================================================================

fold_counts = (
    metadata[
        "strat_fold"
    ]
    .value_counts()
    .sort_index()
)


print("\n" + "=" * 100)
print("STRATIFIED FOLD COUNTS")
print("=" * 100)


print(
    fold_counts.to_string()
)


observed_folds = set(
    metadata[
        "strat_fold"
    ].dropna().astype(int)
)


if observed_folds != set(
    range(
        1,
        11
    )
):

    raise RuntimeError(
        f"Unexpected strat_fold values: {sorted(observed_folds)}"
    )


train_mask = (
    metadata[
        "strat_fold"
    ]
    !=
    10
)


test_mask = (
    metadata[
        "strat_fold"
    ]
    ==
    10
)


print(
    "\nFolds 1–9:",
    int(
        train_mask.sum()
    )
)


print(
    "Fold 10:",
    int(
        test_mask.sum()
    )
)


# ======================================================================================
# 5. PATIENT-LEVEL TRAIN/TEST SEPARATION
# ======================================================================================

train_patients = set(
    metadata.loc[
        train_mask,
        "patient_id"
    ].dropna()
)


test_patients = set(
    metadata.loc[
        test_mask,
        "patient_id"
    ].dropna()
)


patient_overlap = (
    train_patients
    &
    test_patients
)


print("\n" + "=" * 100)
print("PATIENT-LEVEL SPLIT AUDIT")
print("=" * 100)


print(
    "Unique patients folds 1–9:",
    len(
        train_patients
    )
)


print(
    "Unique patients fold 10:",
    len(
        test_patients
    )
)


print(
    "Patients occurring in BOTH:",
    len(
        patient_overlap
    )
)


if len(
    patient_overlap
) != 0:

    raise RuntimeError(
        "Patient overlap exists between fold 10 and folds 1–9."
    )


print(
    "\n✓ No patient overlap between source training and test partitions."
)


# ======================================================================================
# 6. PARSE SCP CODES
# ======================================================================================

def parse_scp_codes(value):

    if isinstance(
        value,
        dict
    ):

        return value


    if pd.isna(
        value
    ):

        return {}


    parsed = ast.literal_eval(
        str(
            value
        )
    )


    if not isinstance(
        parsed,
        dict
    ):

        raise RuntimeError(
            f"scp_codes did not parse as dict: {value}"
        )


    return parsed


metadata[
    "_scp_dict"
] = metadata[
    "scp_codes"
].apply(
    parse_scp_codes
)


# ======================================================================================
# 7. RECONSTRUCT SOURCE REPOSITORY SUBCLASS MAPPING
#
# Source logic:
#
#   agg_df = scp_statements[diagnostic == 1]
#
# For every SCP code present in each ECG:
#     diagnostic_subclass is appended.
#
# No confidence-value threshold is applied in the source notebook.
# ======================================================================================

diagnostic_scp = scp[
    scp[
        "diagnostic"
    ]
    ==
    1
].copy()


def aggregate_diagnostic_subclasses(
    scp_dict
):

    labels = []


    for code in scp_dict.keys():

        if code not in diagnostic_scp.index:
            continue


        subclass = diagnostic_scp.loc[
            code,
            "diagnostic_subclass"
        ]


        if pd.isna(
            subclass
        ):
            continue


        subclass = str(
            subclass
        )


        labels.append(
            subclass
        )


    return sorted(
        set(
            labels
        )
    )


metadata[
    "_diagnostic_subclasses"
] = metadata[
    "_scp_dict"
].apply(
    aggregate_diagnostic_subclasses
)


# ======================================================================================
# 8. RECONSTRUCT MULTILABELBINARIZER ORDER FROM SOURCE TRAINING PARTITION
# ======================================================================================

train_label_lists = metadata.loc[
    train_mask,
    "_diagnostic_subclasses"
].tolist()


test_label_lists = metadata.loc[
    test_mask,
    "_diagnostic_subclasses"
].tolist()


mlb = MultiLabelBinarizer()


Y_train = mlb.fit_transform(
    train_label_lists
)


Y_test = mlb.transform(
    test_label_lists
)


reconstructed_classes = list(
    mlb.classes_
)


print("\n" + "=" * 100)
print("RECONSTRUCTED 23-CLASS MODEL OUTPUT ORDER")
print("=" * 100)


for i, class_name in enumerate(
    reconstructed_classes
):

    print(
        f"{i:02d}  {class_name}"
    )


print(
    "\nNumber of classes:",
    len(
        reconstructed_classes
    )
)


if reconstructed_classes != EXPECTED_MODEL_CLASSES:

    print(
        "\nEXPECTED:"
    )

    print(
        EXPECTED_MODEL_CLASSES
    )


    print(
        "\nRECONSTRUCTED:"
    )

    print(
        reconstructed_classes
    )


    raise RuntimeError(
        "Reconstructed PTB-XL subclass order does not match "
        "the frozen 23-output model vocabulary."
    )


print(
    "\n✓ Exact 23-output class vocabulary/order reproduced."
)


# ======================================================================================
# 9. CHECK WHETHER TEST FOLD CONTAINS SAME CLASS SET
# ======================================================================================

test_class_set = sorted(
    {
        label
        for labels in test_label_lists
        for label in labels
    }
)


train_class_set = sorted(
    reconstructed_classes
)


print("\n" + "=" * 100)
print("TEST-FOLD CLASS COVERAGE")
print("=" * 100)


print(
    "Classes represented in folds 1–9:",
    len(
        train_class_set
    )
)


print(
    "Classes represented in fold 10:",
    len(
        test_class_set
    )
)


missing_from_test = sorted(
    set(
        train_class_set
    )
    -
    set(
        test_class_set
    )
)


extra_in_test = sorted(
    set(
        test_class_set
    )
    -
    set(
        train_class_set
    )
)


print(
    "Training classes absent from fold 10:",
    missing_from_test
)


print(
    "Unexpected fold-10 classes:",
    extra_in_test
)


# ======================================================================================
# 10. CLASS PREVALENCE
# ======================================================================================

train_positive_counts = Y_train.sum(
    axis=0
)


test_positive_counts = Y_test.sum(
    axis=0
)


prevalence_df = pd.DataFrame(
    {
        "class":
            reconstructed_classes,

        "train_positive":
            train_positive_counts.astype(
                int
            ),

        "test_positive":
            test_positive_counts.astype(
                int
            ),

        "train_prevalence":
            train_positive_counts
            /
            len(
                Y_train
            ),

        "test_prevalence":
            test_positive_counts
            /
            len(
                Y_test
            ),
    }
)


print("\n" + "=" * 100)
print("23-CLASS PREVALENCE")
print("=" * 100)


print(
    prevalence_df
    .round(
        5
    )
    .to_string(
        index=False
    )
)


classes_with_no_test_positives = prevalence_df.loc[
    prevalence_df[
        "test_positive"
    ]
    ==
    0,
    "class"
].tolist()


classes_with_all_test_positives = prevalence_df.loc[
    prevalence_df[
        "test_positive"
    ]
    ==
    len(
        Y_test
    ),
    "class"
].tolist()


print(
    "\nClasses with zero positives in test:",
    classes_with_no_test_positives
)


print(
    "Classes with no negatives in test:",
    classes_with_all_test_positives
)


# ======================================================================================
# 11. ALL-ZERO TARGET ROWS
#
# The source preprocessing does not explicitly drop ECGs that have no mapped
# diagnostic subclass, so quantify them rather than silently removing them.
# ======================================================================================

train_all_zero = (
    Y_train.sum(
        axis=1
    )
    ==
    0
)


test_all_zero = (
    Y_test.sum(
        axis=1
    )
    ==
    0
)


print("\n" + "=" * 100)
print("ALL-ZERO DIAGNOSTIC-SUBCLASS ROWS")
print("=" * 100)


print(
    "Folds 1–9:",
    int(
        train_all_zero.sum()
    )
)


print(
    "Fold 10:",
    int(
        test_all_zero.sum()
    )
)


# ======================================================================================
# 12. CHECK RAW LOW-RESOLUTION FILE PATHS
# ======================================================================================

filename_lr = metadata[
    "filename_lr"
].astype(
    str
)


raw_exists = filename_lr.apply(
    lambda relative_path: (
        PTBXL_ROOT
        /
        relative_path
    ).with_suffix(
        ".dat"
    ).exists()
)


print("\n" + "=" * 100)
print("100-HZ WAVEFORM FILE AUDIT")
print("=" * 100)


print(
    "Records with .dat file present:",
    int(
        raw_exists.sum()
    ),
    "/",
    len(
        metadata
    )
)


test_raw_exists = raw_exists.loc[
    test_mask
]


print(
    "Fold-10 .dat files present:",
    int(
        test_raw_exists.sum()
    ),
    "/",
    int(
        test_mask.sum()
    )
)


if not test_raw_exists.all():

    missing_paths = metadata.loc[
        test_mask
        &
        ~raw_exists,
        "filename_lr"
    ].head(
        20
    )


    raise RuntimeError(
        "\nSome fold-10 raw files are missing:\n"
        +
        missing_paths.to_string()
    )


# ======================================================================================
# 13. READ ONE RAW ECG AND VERIFY SIGNAL GEOMETRY
# ======================================================================================

try:

    import wfdb

except ImportError as exc:

    raise ImportError(
        "\nThe metadata/label audit passed so far, but wfdb is not installed.\n"
        "Install it with:\n\n"
        "    pip install wfdb\n\n"
        "Then rerun this Step-4A cell."
    ) from exc


first_test_row = metadata.loc[
    test_mask
].iloc[
    0
]


sample_path = (
    PTBXL_ROOT
    /
    str(
        first_test_row[
            "filename_lr"
        ]
    )
)


signal, signal_metadata = wfdb.rdsamp(
    str(
        sample_path
    )
)


print("\n" + "=" * 100)
print("RAW SIGNAL GEOMETRY")
print("=" * 100)


print(
    "Example record:",
    sample_path
)


print(
    "Raw shape:",
    signal.shape
)


print(
    "Sampling frequency:",
    signal_metadata.get(
        "fs"
    )
)


signal_leads = [
    str(
        lead
    ).upper()
    for lead
    in signal_metadata.get(
        "sig_name",
        []
    )
]


print(
    "Lead order:",
    signal_leads
)


print(
    "Finite:",
    np.all(
        np.isfinite(
            signal
        )
    )
)


if signal.shape != (
    1000,
    12
):

    raise RuntimeError(
        f"Expected raw 100-Hz shape (1000, 12), found {signal.shape}."
    )


if float(
    signal_metadata.get(
        "fs"
    )
) != 100.0:

    raise RuntimeError(
        f"Expected 100 Hz, found {signal_metadata.get('fs')}."
    )


if signal_leads != EXPECTED_LEADS:

    raise RuntimeError(
        "\nUnexpected PTB-XL lead order.\n"
        f"Expected: {EXPECTED_LEADS}\n"
        f"Actual:   {signal_leads}"
    )


if not np.all(
    np.isfinite(
        signal
    )
):

    raise RuntimeError(
        "Example signal contains non-finite values."
    )


print(
    "\n✓ Raw PTB-XL geometry matches model input convention."
)


# ======================================================================================
# 14. PREPARED MODEL INPUT SHAPE
#
# Source preprocessing loads:
#     raw = (N, 1000, 12)
#
# then transposes:
#     (N, 12, 1000)
#
# and adds a single channel:
#     (N, 12, 1000, 1)
#
# We demonstrate this on ONE ECG only.
# ======================================================================================

sample_model_input = (
    signal.T[
        None,
        :,
        :,
        None
    ]
).astype(
    np.float32
)


print("\n" + "=" * 100)
print("MODEL-INPUT GEOMETRY CHECK")
print("=" * 100)


print(
    "One prepared ECG:",
    sample_model_input.shape
)


if sample_model_input.shape != (
    1,
    12,
    1000,
    1
):

    raise RuntimeError(
        "Prepared model-input shape is incorrect."
    )


print(
    "\n✓ Prepared PTB-XL input has shape (1, 12, 1000, 1)."
)


# ======================================================================================
# 15. SAVE READ-ONLY AUDIT OUTPUTS
# ======================================================================================

AUDIT_DIR = Path(
    "definitive_clean_gaze/ptbxl_audit"
)


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


prevalence_df.to_csv(
    AUDIT_DIR
    /
    "ptbxl_23class_prevalence.csv",
    index=False
)


fold_counts.rename(
    "n_records"
).to_csv(
    AUDIT_DIR
    /
    "ptbxl_fold_counts.csv"
)


audit_manifest = {

    "stage":
        "Step 4A PTB-XL read-only provenance audit",

    "ptbxl_root":
        str(
            PTBXL_ROOT
        ),

    "metadata_records":
        int(
            len(
                metadata
            )
        ),

    "fold_10_records":
        int(
            test_mask.sum()
        ),

    "folds_1_to_9_records":
        int(
            train_mask.sum()
        ),

    "patient_overlap_between_folds_1_to_9_and_10":
        int(
            len(
                patient_overlap
            )
        ),

    "model_class_order":
        reconstructed_classes,

    "n_model_classes":
        int(
            len(
                reconstructed_classes
            )
        ),

    "fold10_missing_classes":
        missing_from_test,

    "fold10_all_zero_rows":
        int(
            test_all_zero.sum()
        ),

    "raw_signal_shape":
        list(
            signal.shape
        ),

    "sampling_frequency":
        float(
            signal_metadata.get(
                "fs"
            )
        ),

    "lead_order":
        signal_leads,

    "model_input_shape":
        list(
            sample_model_input.shape
        ),

    "model_inference_performed":
        False,

    "training_performed":
        False,
}


with open(
    AUDIT_DIR
    /
    "ptbxl_audit_manifest.json",
    "w"
) as f:

    json.dump(
        audit_manifest,
        f,
        indent=2
    )


# ======================================================================================
# 16. FINAL SUMMARY
# ======================================================================================

print("\n\n" + "=" * 100)
print("STEP 4A COMPLETE — READ-ONLY AUDIT")
print("=" * 100)


print(
    """
Please paste back these FIVE sections:

1. PTB-XL METADATA

2. STRATIFIED FOLD COUNTS

3. RECONSTRUCTED 23-CLASS MODEL OUTPUT ORDER

4. 23-CLASS PREVALENCE
   (including the zero-positive / no-negative lines)

5. RAW SIGNAL GEOMETRY

Do not run diagnostic model inference yet.
"""
)

In [ ]:
# ======================================================================================
# STEP 4B — PTB-XL FOLD-10 DIAGNOSTIC EVALUATION
#
# INFERENCE ONLY.
# NO TRAINING.
# NO THRESHOLD TUNING.
#
# Models:
#   1. Base pretrained model
#   2. Final consultant-gaze model
#   3. Final all-clinician-gaze model
#
# Test set:
#   PTB-XL strat_fold == 10
#   N = 2198
#
# Metrics:
#   - per-class AUROC
#   - per-class Average Precision (AP)
#   - macro AUROC
#   - macro AP
#
# All 2198 records are retained, including records with zero positive
# diagnostic-subclass labels.
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import ast
import gc
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import wfdb

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)


# ======================================================================================
# 1. FROZEN DEFINITIONS
# ======================================================================================

MODEL_CLASSES = [
    "AMI",
    "CLBBB",
    "CRBBB",
    "ILBBB",
    "IMI",
    "IRBBB",
    "ISCA",
    "ISCI",
    "ISC_",
    "IVCD",
    "LAFB/LPFB",
    "LAO/LAE",
    "LMI",
    "LVH",
    "NORM",
    "NST_",
    "PMI",
    "RAO/RAE",
    "RVH",
    "SEHYP",
    "STTC",
    "WPW",
    "_AVB",
]


EXPECTED_LEADS = [
    "I",
    "II",
    "III",
    "AVR",
    "AVL",
    "AVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]


EXPECTED_TEST_N = 2198

PRED_BATCH_SIZE = 64


# ======================================================================================
# 2. PATHS
# ======================================================================================

PTBXL_ROOT = Path(
    "ptb-xl"
)


DATABASE_PATH = (
    PTBXL_ROOT
    /
    "ptbxl_database.csv"
)


SCP_PATH = (
    PTBXL_ROOT
    /
    "scp_statements.csv"
)


ROOT = Path(
    "definitive_clean_gaze"
)


STEP1_MANIFEST_PATH = (
    ROOT
    /
    "step1_manifest.json"
)


FINAL_MANIFEST_PATH = (
    ROOT
    /
    "final_models"
    /
    "final_model_manifest.json"
)


CONSULTANT_WEIGHTS = (
    ROOT
    /
    "final_models"
    /
    "clean_consultant.weights.h5"
)


ALL_CLINICIAN_WEIGHTS = (
    ROOT
    /
    "final_models"
    /
    "clean_all_clinicians.weights.h5"
)


OUT_DIR = (
    ROOT
    /
    "ptbxl_evaluation"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for path in [
    DATABASE_PATH,
    SCP_PATH,
    STEP1_MANIFEST_PATH,
    FINAL_MANIFEST_PATH,
    CONSULTANT_WEIGHTS,
    ALL_CLINICIAN_WEIGHTS,
]:

    if not path.exists():
        raise FileNotFoundError(
            path
        )


# ======================================================================================
# 3. HASH HELPER
# ======================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# ======================================================================================
# 4. VERIFY FROZEN MODELS
# ======================================================================================

with open(
    STEP1_MANIFEST_PATH,
    "r"
) as f:

    step1_manifest = json.load(
        f
    )


with open(
    FINAL_MANIFEST_PATH,
    "r"
) as f:

    final_manifest = json.load(
        f
    )


BASE_MODEL_PATH = Path(
    step1_manifest[
        "base_model"
    ]
)


BASE_HASH = sha256_file(
    BASE_MODEL_PATH
)


CONSULTANT_HASH = sha256_file(
    CONSULTANT_WEIGHTS
)


ALL_HASH = sha256_file(
    ALL_CLINICIAN_WEIGHTS
)


print("=" * 100)
print("FROZEN MODEL VERIFICATION")
print("=" * 100)


print(
    "Base:"
)

print(
    BASE_HASH
)


print(
    "\nConsultant:"
)

print(
    CONSULTANT_HASH
)


print(
    "\nAll clinicians:"
)

print(
    ALL_HASH
)


if (
    BASE_HASH
    !=
    step1_manifest[
        "base_model_sha256"
    ]
):

    raise RuntimeError(
        "Base model hash mismatch."
    )


if (
    CONSULTANT_HASH
    !=
    final_manifest[
        "consultant_weights_sha256"
    ]
):

    raise RuntimeError(
        "Consultant model hash mismatch."
    )


if (
    ALL_HASH
    !=
    final_manifest[
        "all_clinician_weights_sha256"
    ]
):

    raise RuntimeError(
        "All-clinician model hash mismatch."
    )


print(
    "\n✓ All three diagnostic models are frozen and verified."
)


# ======================================================================================
# 5. LOAD PTB-XL METADATA
# ======================================================================================

metadata = pd.read_csv(
    DATABASE_PATH,
    index_col="ecg_id"
)


scp = pd.read_csv(
    SCP_PATH,
    index_col=0
)


metadata[
    "scp_codes"
] = metadata[
    "scp_codes"
].apply(
    ast.literal_eval
)


test_metadata = metadata[
    metadata[
        "strat_fold"
    ]
    ==
    10
].copy()


if len(
    test_metadata
) != EXPECTED_TEST_N:

    raise RuntimeError(
        f"Expected {EXPECTED_TEST_N} fold-10 ECGs, "
        f"found {len(test_metadata)}."
    )


print("\n" + "=" * 100)
print("PTB-XL TEST SET")
print("=" * 100)


print(
    "Fold-10 ECGs:",
    len(
        test_metadata
    )
)


print(
    "Unique patients:",
    test_metadata[
        "patient_id"
    ].nunique()
)


# ======================================================================================
# 6. RECONSTRUCT EXACT 23-CLASS TARGETS
# ======================================================================================

diagnostic_scp = scp[
    scp[
        "diagnostic"
    ]
    ==
    1
].copy()


def aggregate_subclasses(
    code_dictionary
):

    output = set()


    for code in code_dictionary.keys():

        if code not in diagnostic_scp.index:
            continue


        subclass = diagnostic_scp.loc[
            code,
            "diagnostic_subclass"
        ]


        if pd.isna(
            subclass
        ):
            continue


        output.add(
            str(
                subclass
            )
        )


    return output


test_labels = test_metadata[
    "scp_codes"
].apply(
    aggregate_subclasses
)


class_to_index = {
    class_name: i
    for i, class_name in enumerate(
        MODEL_CLASSES
    )
}


Y_test = np.zeros(
    (
        len(
            test_metadata
        ),
        len(
            MODEL_CLASSES
        )
    ),
    dtype=np.int8
)


for row_idx, labels in enumerate(
    test_labels
):

    for label in labels:

        if label not in class_to_index:

            raise RuntimeError(
                f"Unexpected diagnostic subclass: {label}"
            )


        Y_test[
            row_idx,
            class_to_index[
                label
            ]
        ] = 1


if Y_test.shape != (
    2198,
    23
):

    raise RuntimeError(
        f"Unexpected Y_test shape: {Y_test.shape}"
    )


class_positive_counts = Y_test.sum(
    axis=0
)


if np.any(
    class_positive_counts == 0
):

    raise RuntimeError(
        "At least one diagnostic class has zero positives."
    )


if np.any(
    class_positive_counts == len(
        Y_test
    )
):

    raise RuntimeError(
        "At least one diagnostic class has zero negatives."
    )


all_zero_rows = (
    Y_test.sum(
        axis=1
    )
    ==
    0
)


print(
    "Target shape:",
    Y_test.shape
)


print(
    "All-zero 23-class target rows:",
    int(
        all_zero_rows.sum()
    )
)


if int(
    all_zero_rows.sum()
) != 40:

    raise RuntimeError(
        "Expected exactly 40 all-zero fold-10 targets."
    )


print(
    "\n✓ Exact fold-10 targets reconstructed."
)


# ======================================================================================
# 7. LOAD ALL 2198 RAW ECGs
#
# Source waveform:
#     (1000, 12)
#
# Model input:
#     (12, 1000, 1)
# ======================================================================================

print("\n" + "=" * 100)
print("LOADING PTB-XL FOLD-10 WAVEFORMS")
print("=" * 100)


X_test = np.empty(
    (
        EXPECTED_TEST_N,
        12,
        1000,
        1
    ),
    dtype=np.float32
)


for i, (_, row) in enumerate(
    test_metadata.iterrows()
):

    record_path = (
        PTBXL_ROOT
        /
        str(
            row[
                "filename_lr"
            ]
        )
    )


    signal, signal_metadata = wfdb.rdsamp(
        str(
            record_path
        )
    )


    if signal.shape != (
        1000,
        12
    ):

        raise RuntimeError(
            f"Bad signal shape at row {i}: {signal.shape}"
        )


    leads = [
        str(x).upper()
        for x in signal_metadata[
            "sig_name"
        ]
    ]


    if leads != EXPECTED_LEADS:

        raise RuntimeError(
            f"Lead-order mismatch at row {i}: {leads}"
        )


    if float(
        signal_metadata[
            "fs"
        ]
    ) != 100.0:

        raise RuntimeError(
            f"Sampling-rate mismatch at row {i}."
        )


    if not np.all(
        np.isfinite(
            signal
        )
    ):

        raise RuntimeError(
            f"Non-finite ECG values at row {i}."
        )


    X_test[
        i,
        :,
        :,
        0
    ] = signal.T.astype(
        np.float32
    )


    if (
        (i + 1) % 250 == 0
        or
        (i + 1) == EXPECTED_TEST_N
    ):

        print(
            f"Loaded {i + 1}/{EXPECTED_TEST_N}"
        )


print(
    "\nFinal X_test shape:",
    X_test.shape
)


print(
    "Finite:",
    np.all(
        np.isfinite(
            X_test
        )
    )
)


print(
    "Global minimum:",
    float(
        X_test.min()
    )
)


print(
    "Global maximum:",
    float(
        X_test.max()
    )
)


print(
    "\n✓ All fold-10 waveforms loaded."
)


# ======================================================================================
# 8. LOAD MODEL VARIANTS
# ======================================================================================

def load_base_model():

    tf.keras.backend.clear_session()


    model = tf.keras.models.load_model(
        BASE_MODEL_PATH,
        compile=False
    )


    if model.output_shape != (
        None,
        23
    ):

        raise RuntimeError(
            f"Unexpected output shape: {model.output_shape}"
        )


    final_activation = (
        model.layers[
            -1
        ]
        .activation
        .__name__
    )


    if final_activation != "sigmoid":

        raise RuntimeError(
            f"Expected sigmoid output, found {final_activation}."
        )


    return model


# ======================================================================================
# 9. WEIGHT-DIFFERENCE AUDIT AFTER LOADING SAVED FINAL WEIGHTS
# ======================================================================================

def model_weight_snapshot(
    model
):

    return {
        layer.name: [
            np.array(
                weight,
                copy=True
            )
            for weight in layer.get_weights()
        ]
        for layer in model.layers
    }


def compare_to_snapshot(
    base_snapshot,
    model
):

    rows = []


    for layer in model.layers:

        old_weights = base_snapshot[
            layer.name
        ]


        new_weights = layer.get_weights()


        max_change = 0.0


        for old, new in zip(
            old_weights,
            new_weights
        ):

            if old.shape != new.shape:

                raise RuntimeError(
                    f"Weight shape mismatch: {layer.name}"
                )


            if old.size:

                max_change = max(
                    max_change,
                    float(
                        np.max(
                            np.abs(
                                old - new
                            )
                        )
                    )
                )


        rows.append(
            {
                "layer":
                    layer.name,

                "max_abs_change":
                    max_change,
            }
        )


    return pd.DataFrame(
        rows
    )


# Base reference
base_model = load_base_model()


base_snapshot = model_weight_snapshot(
    base_model
)


# Consultant
consultant_model = load_base_model()


consultant_model.load_weights(
    CONSULTANT_WEIGHTS
)


consultant_weight_audit = compare_to_snapshot(
    base_snapshot,
    consultant_model
)


# All clinicians
all_model = load_base_model()


all_model.load_weights(
    ALL_CLINICIAN_WEIGHTS
)


all_weight_audit = compare_to_snapshot(
    base_snapshot,
    all_model
)


for name, audit in [
    (
        "consultant",
        consultant_weight_audit
    ),
    (
        "all_clinicians",
        all_weight_audit
    ),
]:

    changed = set(
        audit.loc[
            audit[
                "max_abs_change"
            ]
            >
            1e-12,
            "layer"
        ]
    )


    if changed != {
        "conv2d_10"
    }:

        raise RuntimeError(
            f"{name}: unexpected changed layers {changed}"
        )


print("\n" + "=" * 100)
print("DOWNSTREAM MODEL WEIGHT AUDIT")
print("=" * 100)


print(
    "Consultant vs base changed layers:"
)

print(
    consultant_weight_audit.loc[
        consultant_weight_audit[
            "max_abs_change"
        ]
        >
        1e-12
    ].to_string(
        index=False
    )
)


print(
    "\nAll-clinician vs base changed layers:"
)

print(
    all_weight_audit.loc[
        all_weight_audit[
            "max_abs_change"
        ]
        >
        1e-12
    ].to_string(
        index=False
    )
)


print(
    "\n✓ Saved final gaze models differ from base ONLY at conv2d_10."
)


# ======================================================================================
# 10. INFERENCE
# ======================================================================================

def run_prediction(
    model,
    X,
    name
):

    print(
        f"\nRunning inference: {name}"
    )


    prediction = model.predict(
        X,
        batch_size=PRED_BATCH_SIZE,
        verbose=1
    )


    prediction = np.asarray(
        prediction,
        dtype=np.float32
    )


    if prediction.shape != (
        EXPECTED_TEST_N,
        23
    ):

        raise RuntimeError(
            f"{name}: bad prediction shape {prediction.shape}"
        )


    if not np.all(
        np.isfinite(
            prediction
        )
    ):

        raise RuntimeError(
            f"{name}: non-finite predictions."
        )


    if np.any(
        prediction < 0
    ) or np.any(
        prediction > 1
    ):

        raise RuntimeError(
            f"{name}: sigmoid predictions outside [0,1]."
        )


    print(
        f"{name}: prediction range "
        f"{prediction.min():.6f} to {prediction.max():.6f}"
    )


    return prediction


print("\n" + "=" * 100)
print("PTB-XL FOLD-10 MODEL INFERENCE")
print("=" * 100)


P_base = run_prediction(
    base_model,
    X_test,
    "BASE"
)


P_consultant = run_prediction(
    consultant_model,
    X_test,
    "CONSULTANT-GAZE"
)


P_all = run_prediction(
    all_model,
    X_test,
    "ALL-CLINICIAN-GAZE"
)


# ======================================================================================
# 11. VERIFY THAT GAZE SUPERVISION ACTUALLY CHANGED DIAGNOSTIC OUTPUTS
# ======================================================================================

consultant_output_change = np.abs(
    P_consultant
    -
    P_base
)


all_output_change = np.abs(
    P_all
    -
    P_base
)


print("\n" + "=" * 100)
print("PREDICTION CHANGE AUDIT")
print("=" * 100)


print(
    "Consultant vs base:"
)

print(
    "  mean absolute probability change:",
    float(
        consultant_output_change.mean()
    )
)

print(
    "  max absolute probability change:",
    float(
        consultant_output_change.max()
    )
)


print(
    "\nAll clinicians vs base:"
)

print(
    "  mean absolute probability change:",
    float(
        all_output_change.mean()
    )
)

print(
    "  max absolute probability change:",
    float(
        all_output_change.max()
    )
)


# ======================================================================================
# 12. PER-CLASS METRICS
# ======================================================================================

def evaluate_predictions(
    model_name,
    predictions
):

    rows = []


    for class_idx, class_name in enumerate(
        MODEL_CLASSES
    ):

        y_true = Y_test[
            :,
            class_idx
        ]


        y_score = predictions[
            :,
            class_idx
        ]


        auroc = roc_auc_score(
            y_true,
            y_score
        )


        ap = average_precision_score(
            y_true,
            y_score
        )


        rows.append(
            {
                "model":
                    model_name,

                "class":
                    class_name,

                "positive_n":
                    int(
                        y_true.sum()
                    ),

                "negative_n":
                    int(
                        len(
                            y_true
                        )
                        -
                        y_true.sum()
                    ),

                "AUROC":
                    float(
                        auroc
                    ),

                "AP":
                    float(
                        ap
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


metrics_base = evaluate_predictions(
    "base",
    P_base
)


metrics_consultant = evaluate_predictions(
    "consultant",
    P_consultant
)


metrics_all = evaluate_predictions(
    "all_clinicians",
    P_all
)


all_metrics = pd.concat(
    [
        metrics_base,
        metrics_consultant,
        metrics_all
    ],
    ignore_index=True
)


# ======================================================================================
# 13. PAIRED PER-CLASS COMPARISON TABLE
# ======================================================================================

comparison = (
    metrics_base[
        [
            "class",
            "positive_n",
            "negative_n",
            "AUROC",
            "AP"
        ]
    ]
    .rename(
        columns={
            "AUROC":
                "base_AUROC",

            "AP":
                "base_AP",
        }
    )
)


comparison = comparison.merge(

    metrics_consultant[
        [
            "class",
            "AUROC",
            "AP"
        ]
    ].rename(
        columns={
            "AUROC":
                "consultant_AUROC",

            "AP":
                "consultant_AP",
        }
    ),

    on="class",
    how="left"
)


comparison = comparison.merge(

    metrics_all[
        [
            "class",
            "AUROC",
            "AP"
        ]
    ].rename(
        columns={
            "AUROC":
                "all_clinicians_AUROC",

            "AP":
                "all_clinicians_AP",
        }
    ),

    on="class",
    how="left"
)


comparison[
    "consultant_delta_AUROC"
] = (
    comparison[
        "consultant_AUROC"
    ]
    -
    comparison[
        "base_AUROC"
    ]
)


comparison[
    "all_clinicians_delta_AUROC"
] = (
    comparison[
        "all_clinicians_AUROC"
    ]
    -
    comparison[
        "base_AUROC"
    ]
)


comparison[
    "consultant_delta_AP"
] = (
    comparison[
        "consultant_AP"
    ]
    -
    comparison[
        "base_AP"
    ]
)


comparison[
    "all_clinicians_delta_AP"
] = (
    comparison[
        "all_clinicians_AP"
    ]
    -
    comparison[
        "base_AP"
    ]
)


# Preserve frozen class order.
comparison[
    "_order"
] = comparison[
    "class"
].map(
    {
        class_name: i
        for i, class_name in enumerate(
            MODEL_CLASSES
        )
    }
)


comparison = (
    comparison
    .sort_values(
        "_order"
    )
    .drop(
        columns="_order"
    )
    .reset_index(
        drop=True
    )
)


# ======================================================================================
# 14. MACRO SUMMARY
# ======================================================================================

macro_rows = []


for name, predictions, model_metrics in [

    (
        "base",
        P_base,
        metrics_base
    ),

    (
        "consultant",
        P_consultant,
        metrics_consultant
    ),

    (
        "all_clinicians",
        P_all,
        metrics_all
    ),
]:

    macro_auroc = float(
        model_metrics[
            "AUROC"
        ].mean()
    )


    macro_ap = float(
        model_metrics[
            "AP"
        ].mean()
    )


    micro_auroc = float(
        roc_auc_score(
            Y_test.ravel(),
            predictions.ravel()
        )
    )


    micro_ap = float(
        average_precision_score(
            Y_test.ravel(),
            predictions.ravel()
        )
    )


    macro_rows.append(
        {
            "model":
                name,

            "macro_AUROC":
                macro_auroc,

            "macro_AP":
                macro_ap,

            "micro_AUROC":
                micro_auroc,

            "micro_AP":
                micro_ap,
        }
    )


macro_summary = pd.DataFrame(
    macro_rows
)


base_macro_auroc = float(
    macro_summary.loc[
        macro_summary[
            "model"
        ]
        ==
        "base",
        "macro_AUROC"
    ].iloc[
        0
    ]
)


base_macro_ap = float(
    macro_summary.loc[
        macro_summary[
            "model"
        ]
        ==
        "base",
        "macro_AP"
    ].iloc[
        0
    ]
)


macro_summary[
    "delta_macro_AUROC_vs_base"
] = (
    macro_summary[
        "macro_AUROC"
    ]
    -
    base_macro_auroc
)


macro_summary[
    "delta_macro_AP_vs_base"
] = (
    macro_summary[
        "macro_AP"
    ]
    -
    base_macro_ap
)


# ======================================================================================
# 15. COUNT PER-CLASS DIRECTIONS
# ======================================================================================

direction_summary = pd.DataFrame(
    [
        {
            "model":
                "consultant",

            "classes_AUROC_improved":
                int(
                    (
                        comparison[
                            "consultant_delta_AUROC"
                        ]
                        >
                        0
                    ).sum()
                ),

            "classes_AUROC_worsened":
                int(
                    (
                        comparison[
                            "consultant_delta_AUROC"
                        ]
                        <
                        0
                    ).sum()
                ),

            "classes_AP_improved":
                int(
                    (
                        comparison[
                            "consultant_delta_AP"
                        ]
                        >
                        0
                    ).sum()
                ),

            "classes_AP_worsened":
                int(
                    (
                        comparison[
                            "consultant_delta_AP"
                        ]
                        <
                        0
                    ).sum()
                ),
        },

        {
            "model":
                "all_clinicians",

            "classes_AUROC_improved":
                int(
                    (
                        comparison[
                            "all_clinicians_delta_AUROC"
                        ]
                        >
                        0
                    ).sum()
                ),

            "classes_AUROC_worsened":
                int(
                    (
                        comparison[
                            "all_clinicians_delta_AUROC"
                        ]
                        <
                        0
                    ).sum()
                ),

            "classes_AP_improved":
                int(
                    (
                        comparison[
                            "all_clinicians_delta_AP"
                        ]
                        >
                        0
                    ).sum()
                ),

            "classes_AP_worsened":
                int(
                    (
                        comparison[
                            "all_clinicians_delta_AP"
                        ]
                        <
                        0
                    ).sum()
                ),
        },
    ]
)


# ======================================================================================
# 16. SAVE PREDICTIONS
# ======================================================================================

prediction_output = pd.DataFrame(
    {
        "ecg_id":
            test_metadata.index.to_numpy(),

        "patient_id":
            test_metadata[
                "patient_id"
            ].to_numpy(),

        "all_zero_target":
            all_zero_rows.astype(
                int
            ),
    }
)


# Save each true label and each model probability.
for class_idx, class_name in enumerate(
    MODEL_CLASSES
):

    prediction_output[
        f"true__{class_name}"
    ] = Y_test[
        :,
        class_idx
    ]


    prediction_output[
        f"base__{class_name}"
    ] = P_base[
        :,
        class_idx
    ]


    prediction_output[
        f"consultant__{class_name}"
    ] = P_consultant[
        :,
        class_idx
    ]


    prediction_output[
        f"all_clinicians__{class_name}"
    ] = P_all[
        :,
        class_idx
    ]


prediction_output.to_csv(
    OUT_DIR
    /
    "ptbxl_fold10_predictions.csv",
    index=False
)


all_metrics.to_csv(
    OUT_DIR
    /
    "ptbxl_per_class_metrics_long.csv",
    index=False
)


comparison.to_csv(
    OUT_DIR
    /
    "ptbxl_per_class_comparison.csv",
    index=False
)


macro_summary.to_csv(
    OUT_DIR
    /
    "ptbxl_macro_summary.csv",
    index=False
)


direction_summary.to_csv(
    OUT_DIR
    /
    "ptbxl_direction_summary.csv",
    index=False
)


consultant_weight_audit.to_csv(
    OUT_DIR
    /
    "consultant_downstream_weight_audit.csv",
    index=False
)


all_weight_audit.to_csv(
    OUT_DIR
    /
    "all_clinicians_downstream_weight_audit.csv",
    index=False
)


# ======================================================================================
# 17. PRINT MACRO RESULTS
# ======================================================================================

print("\n\n" + "=" * 100)
print("PTB-XL FOLD-10 MACRO DIAGNOSTIC PERFORMANCE")
print("=" * 100)


print(
    macro_summary
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 18. PRINT PER-CLASS RESULTS
# ======================================================================================

print("\n\n" + "=" * 100)
print("PTB-XL PER-CLASS PERFORMANCE")
print("=" * 100)


print(
    comparison[
        [
            "class",
            "positive_n",

            "base_AUROC",
            "consultant_AUROC",
            "consultant_delta_AUROC",
            "all_clinicians_AUROC",
            "all_clinicians_delta_AUROC",

            "base_AP",
            "consultant_AP",
            "consultant_delta_AP",
            "all_clinicians_AP",
            "all_clinicians_delta_AP",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 19. PRINT DIRECTION SUMMARY
# ======================================================================================

print("\n\n" + "=" * 100)
print("NUMBER OF CLASSES IMPROVED / WORSENED")
print("=" * 100)


print(
    direction_summary.to_string(
        index=False
    )
)


# ======================================================================================
# 20. MANIFEST
# ======================================================================================

evaluation_manifest = {

    "stage":
        "Step 4B PTB-XL fold-10 frozen diagnostic evaluation",

    "test_partition":
        "PTB-XL strat_fold 10",

    "test_records":
        int(
            len(
                Y_test
            )
        ),

    "all_zero_target_records_retained":
        int(
            all_zero_rows.sum()
        ),

    "classes":
        MODEL_CLASSES,

    "n_classes":
        23,

    "input_sampling_frequency_Hz":
        100,

    "input_shape":
        [
            12,
            1000,
            1
        ],

    "base_model_sha256":
        BASE_HASH,

    "consultant_weights_sha256":
        CONSULTANT_HASH,

    "all_clinician_weights_sha256":
        ALL_HASH,

    "metrics":
        [
            "per-class AUROC",
            "per-class Average Precision",
            "macro AUROC",
            "macro Average Precision",
            "micro AUROC",
            "micro Average Precision",
        ],

    "threshold_tuning_performed":
        False,

    "training_performed":
        False,

    "records_excluded":
        0,

    "primary_comparison":
        "frozen gaze models versus unchanged base model on identical fold-10 records",
}


with open(
    OUT_DIR
    /
    "ptbxl_evaluation_manifest.json",
    "w"
) as f:

    json.dump(
        evaluation_manifest,
        f,
        indent=2
    )


print("\n\n" + "=" * 100)
print("STEP 4B COMPLETE — INFERENCE ONLY")
print("=" * 100)


print(
    """
Please paste back:

1. DOWNSTREAM MODEL WEIGHT AUDIT

2. PREDICTION CHANGE AUDIT

3. PTB-XL FOLD-10 MACRO DIAGNOSTIC PERFORMANCE

4. NUMBER OF CLASSES IMPROVED / WORSENED

5. PTB-XL PER-CLASS PERFORMANCE

Do not retrain or adjust either gaze model based on these results.

After we see the result, we will run the paired bootstrap confidence-interval
analysis from the SAVED predictions rather than touching the models again.
"""
)

In [ ]:
# ======================================================================================
# STEP 4C — PATIENT-CLUSTER PAIRED BOOTSTRAP
#
# NO TRAINING.
# NO MODEL INFERENCE.
#
# Uses only:
#   definitive_clean_gaze/ptbxl_evaluation/ptbxl_fold10_predictions.csv
#
# Resampling unit:
#   PATIENT
#
# Comparisons:
#   Consultant - Base
#   All clinicians - Base
#   All clinicians - Consultant
#
# Metrics:
#   macro AUROC
#   macro AP
#   micro AUROC
#   micro AP
#
# Also produces paired per-class AUROC/AP confidence intervals.
# ======================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)


# ======================================================================================
# 1. SETTINGS
# ======================================================================================

SEED = 2026

N_BOOT = 3000

CI_LOW = 2.5
CI_HIGH = 97.5


MODEL_CLASSES = [
    "AMI",
    "CLBBB",
    "CRBBB",
    "ILBBB",
    "IMI",
    "IRBBB",
    "ISCA",
    "ISCI",
    "ISC_",
    "IVCD",
    "LAFB/LPFB",
    "LAO/LAE",
    "LMI",
    "LVH",
    "NORM",
    "NST_",
    "PMI",
    "RAO/RAE",
    "RVH",
    "SEHYP",
    "STTC",
    "WPW",
    "_AVB",
]


MODELS = [
    "base",
    "consultant",
    "all_clinicians",
]


COMPARISONS = [
    (
        "consultant_vs_base",
        "consultant",
        "base",
    ),
    (
        "all_clinicians_vs_base",
        "all_clinicians",
        "base",
    ),
    (
        "all_clinicians_vs_consultant",
        "all_clinicians",
        "consultant",
    ),
]


# ======================================================================================
# 2. PATHS
# ======================================================================================

OUT_DIR = Path(
    "definitive_clean_gaze/ptbxl_evaluation"
)


PRED_PATH = (
    OUT_DIR
    /
    "ptbxl_fold10_predictions.csv"
)


if not PRED_PATH.exists():
    raise FileNotFoundError(
        PRED_PATH
    )


# ======================================================================================
# 3. LOAD SAVED PREDICTIONS
# ======================================================================================

df = pd.read_csv(
    PRED_PATH
)


if len(df) != 2198:
    raise RuntimeError(
        f"Expected 2198 ECGs, found {len(df)}."
    )


if df["patient_id"].nunique() != 1904:
    raise RuntimeError(
        "Expected 1904 unique patients."
    )


# True-label matrix
Y = np.column_stack(
    [
        df[
            f"true__{class_name}"
        ].to_numpy(
            dtype=np.int8
        )
        for class_name in MODEL_CLASSES
    ]
)


# Prediction matrices
P = {}


for model in MODELS:

    P[model] = np.column_stack(
        [
            df[
                f"{model}__{class_name}"
            ].to_numpy(
                dtype=np.float64
            )
            for class_name in MODEL_CLASSES
        ]
    )


if Y.shape != (
    2198,
    23
):
    raise RuntimeError(
        f"Unexpected Y shape: {Y.shape}"
    )


for model in MODELS:

    if P[model].shape != (
        2198,
        23
    ):
        raise RuntimeError(
            f"Unexpected prediction shape for {model}."
        )


print("=" * 100)
print("SAVED PREDICTION AUDIT")
print("=" * 100)

print(
    "ECGs:",
    len(df)
)

print(
    "Patients:",
    df["patient_id"].nunique()
)

print(
    "Classes:",
    Y.shape[1]
)

print(
    "All-zero target ECGs:",
    int(
        (
            Y.sum(
                axis=1
            )
            ==
            0
        ).sum()
    )
)


# ======================================================================================
# 4. PATIENT CLUSTERS
# ======================================================================================

patient_groups = (
    df
    .groupby(
        "patient_id",
        sort=False
    )
    .indices
)


patient_ids = np.asarray(
    list(
        patient_groups.keys()
    ),
    dtype=object
)


N_PATIENTS = len(
    patient_ids
)


if N_PATIENTS != 1904:
    raise RuntimeError(
        f"Expected 1904 patient clusters, found {N_PATIENTS}."
    )


# ======================================================================================
# 5. METRIC FUNCTION
# ======================================================================================

def calculate_metrics(
    y_true,
    y_score
):

    n_classes = y_true.shape[1]


    per_auc = np.full(
        n_classes,
        np.nan,
        dtype=np.float64
    )


    per_ap = np.full(
        n_classes,
        np.nan,
        dtype=np.float64
    )


    for j in range(
        n_classes
    ):

        yj = y_true[
            :,
            j
        ]


        # AUROC/AP only considered valid for the paired
        # class analysis when both classes occur.
        if (
            np.sum(
                yj == 1
            )
            ==
            0
            or
            np.sum(
                yj == 0
            )
            ==
            0
        ):
            continue


        per_auc[j] = roc_auc_score(
            yj,
            y_score[
                :,
                j
            ]
        )


        per_ap[j] = average_precision_score(
            yj,
            y_score[
                :,
                j
            ]
        )


    # Macro result is only retained when every one of the
    # 23 classes is represented in that bootstrap sample.
    if np.all(
        np.isfinite(
            per_auc
        )
    ):

        macro_auc = float(
            np.mean(
                per_auc
            )
        )

        macro_ap = float(
            np.mean(
                per_ap
            )
        )

    else:

        macro_auc = np.nan
        macro_ap = np.nan


    micro_auc = roc_auc_score(
        y_true.ravel(),
        y_score.ravel()
    )


    micro_ap = average_precision_score(
        y_true.ravel(),
        y_score.ravel()
    )


    return {
        "macro_AUROC":
            macro_auc,

        "macro_AP":
            macro_ap,

        "micro_AUROC":
            float(
                micro_auc
            ),

        "micro_AP":
            float(
                micro_ap
            ),

        "per_class_AUROC":
            per_auc,

        "per_class_AP":
            per_ap,
    }


# ======================================================================================
# 6. POINT ESTIMATES
# ======================================================================================

point = {}


for model in MODELS:

    point[model] = calculate_metrics(
        Y,
        P[model]
    )


print("\n" + "=" * 100)
print("POINT ESTIMATES")
print("=" * 100)


point_rows = []


for model in MODELS:

    point_rows.append(
        {
            "model":
                model,

            "macro_AUROC":
                point[
                    model
                ][
                    "macro_AUROC"
                ],

            "macro_AP":
                point[
                    model
                ][
                    "macro_AP"
                ],

            "micro_AUROC":
                point[
                    model
                ][
                    "micro_AUROC"
                ],

            "micro_AP":
                point[
                    model
                ][
                    "micro_AP"
                ],
        }
    )


point_df = pd.DataFrame(
    point_rows
)


print(
    point_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 7. STORAGE
# ======================================================================================

OVERALL_METRICS = [
    "macro_AUROC",
    "macro_AP",
    "micro_AUROC",
    "micro_AP",
]


overall_delta = {}


for comparison_name, _, _ in COMPARISONS:

    overall_delta[
        comparison_name
    ] = {
        metric: np.full(
            N_BOOT,
            np.nan,
            dtype=np.float64
        )
        for metric in OVERALL_METRICS
    }


per_class_auc_delta = {
    comparison_name:
        np.full(
            (
                N_BOOT,
                len(
                    MODEL_CLASSES
                )
            ),
            np.nan,
            dtype=np.float64
        )
    for comparison_name, _, _ in COMPARISONS
}


per_class_ap_delta = {
    comparison_name:
        np.full(
            (
                N_BOOT,
                len(
                    MODEL_CLASSES
                )
            ),
            np.nan,
            dtype=np.float64
        )
    for comparison_name, _, _ in COMPARISONS
}


# ======================================================================================
# 8. PATIENT-LEVEL PAIRED BOOTSTRAP
# ======================================================================================

rng = np.random.default_rng(
    SEED
)


print("\n" + "=" * 100)
print("PATIENT-CLUSTER BOOTSTRAP")
print("=" * 100)


for b in range(
    N_BOOT
):

    sampled_positions = rng.integers(
        low=0,
        high=N_PATIENTS,
        size=N_PATIENTS
    )


    sampled_patients = patient_ids[
        sampled_positions
    ]


    # If a patient is sampled multiple times, all ECGs belonging
    # to that patient are replicated correspondingly.
    bootstrap_indices = np.concatenate(
        [
            np.asarray(
                patient_groups[
                    patient_id
                ],
                dtype=int
            )
            for patient_id in sampled_patients
        ]
    )


    Y_boot = Y[
        bootstrap_indices
    ]


    metrics_boot = {}


    for model in MODELS:

        metrics_boot[
            model
        ] = calculate_metrics(
            Y_boot,
            P[
                model
            ][
                bootstrap_indices
            ]
        )


    # Same bootstrap sample used for every model -> paired comparison.
    for comparison_name, model_a, model_b in COMPARISONS:

        for metric in OVERALL_METRICS:

            a = metrics_boot[
                model_a
            ][
                metric
            ]

            base = metrics_boot[
                model_b
            ][
                metric
            ]


            if (
                np.isfinite(
                    a
                )
                and
                np.isfinite(
                    base
                )
            ):

                overall_delta[
                    comparison_name
                ][
                    metric
                ][b] = (
                    a
                    -
                    base
                )


        per_class_auc_delta[
            comparison_name
        ][b, :] = (
            metrics_boot[
                model_a
            ][
                "per_class_AUROC"
            ]
            -
            metrics_boot[
                model_b
            ][
                "per_class_AUROC"
            ]
        )


        per_class_ap_delta[
            comparison_name
        ][b, :] = (
            metrics_boot[
                model_a
            ][
                "per_class_AP"
            ]
            -
            metrics_boot[
                model_b
            ][
                "per_class_AP"
            ]
        )


    if (
        (b + 1) % 250 == 0
        or
        (b + 1) == N_BOOT
    ):

        print(
            f"Completed {b + 1}/{N_BOOT}"
        )


# ======================================================================================
# 9. CI HELPER
# ======================================================================================

def summarize_bootstrap(
    values,
    point_difference
):

    values = np.asarray(
        values,
        dtype=np.float64
    )


    valid = values[
        np.isfinite(
            values
        )
    ]


    if len(valid) == 0:

        return {
            "point_delta":
                point_difference,

            "bootstrap_mean_delta":
                np.nan,

            "CI_low":
                np.nan,

            "CI_high":
                np.nan,

            "valid_bootstraps":
                0,

            "fraction_delta_below_zero":
                np.nan,

            "fraction_delta_above_zero":
                np.nan,
        }


    ci_low, ci_high = np.percentile(
        valid,
        [
            CI_LOW,
            CI_HIGH
        ]
    )


    return {
        "point_delta":
            float(
                point_difference
            ),

        "bootstrap_mean_delta":
            float(
                np.mean(
                    valid
                )
            ),

        "CI_low":
            float(
                ci_low
            ),

        "CI_high":
            float(
                ci_high
            ),

        "valid_bootstraps":
            int(
                len(
                    valid
                )
            ),

        "fraction_delta_below_zero":
            float(
                np.mean(
                    valid < 0
                )
            ),

        "fraction_delta_above_zero":
            float(
                np.mean(
                    valid > 0
                )
            ),
    }


# ======================================================================================
# 10. OVERALL BOOTSTRAP SUMMARY
# ======================================================================================

overall_rows = []


for comparison_name, model_a, model_b in COMPARISONS:

    for metric in OVERALL_METRICS:

        point_difference = (
            point[
                model_a
            ][
                metric
            ]
            -
            point[
                model_b
            ][
                metric
            ]
        )


        result = summarize_bootstrap(
            overall_delta[
                comparison_name
            ][
                metric
            ],
            point_difference
        )


        overall_rows.append(
            {
                "comparison":
                    comparison_name,

                "metric":
                    metric,

                **result,
            }
        )


overall_summary = pd.DataFrame(
    overall_rows
)


print("\n\n" + "=" * 100)
print("PATIENT-CLUSTER BOOTSTRAP — OVERALL PERFORMANCE DIFFERENCE")
print("=" * 100)


print(
    overall_summary
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 11. PER-CLASS BOOTSTRAP SUMMARY
# ======================================================================================

per_class_rows = []


for comparison_name, model_a, model_b in COMPARISONS:

    for class_idx, class_name in enumerate(
        MODEL_CLASSES
    ):

        # --------------------------------------------------------------
        # AUROC
        # --------------------------------------------------------------

        point_delta_auc = (
            point[
                model_a
            ][
                "per_class_AUROC"
            ][
                class_idx
            ]
            -
            point[
                model_b
            ][
                "per_class_AUROC"
            ][
                class_idx
            ]
        )


        auc_result = summarize_bootstrap(
            per_class_auc_delta[
                comparison_name
            ][
                :,
                class_idx
            ],
            point_delta_auc
        )


        per_class_rows.append(
            {
                "comparison":
                    comparison_name,

                "class":
                    class_name,

                "metric":
                    "AUROC",

                "positive_n":
                    int(
                        Y[
                            :,
                            class_idx
                        ].sum()
                    ),

                **auc_result,
            }
        )


        # --------------------------------------------------------------
        # AP
        # --------------------------------------------------------------

        point_delta_ap = (
            point[
                model_a
            ][
                "per_class_AP"
            ][
                class_idx
            ]
            -
            point[
                model_b
            ][
                "per_class_AP"
            ][
                class_idx
            ]
        )


        ap_result = summarize_bootstrap(
            per_class_ap_delta[
                comparison_name
            ][
                :,
                class_idx
            ],
            point_delta_ap
        )


        per_class_rows.append(
            {
                "comparison":
                    comparison_name,

                "class":
                    class_name,

                "metric":
                    "AP",

                "positive_n":
                    int(
                        Y[
                            :,
                            class_idx
                        ].sum()
                    ),

                **ap_result,
            }
        )


per_class_summary = pd.DataFrame(
    per_class_rows
)


# ======================================================================================
# 12. SAVE
# ======================================================================================

overall_summary.to_csv(
    OUT_DIR
    /
    "ptbxl_patient_cluster_bootstrap_overall.csv",
    index=False
)


per_class_summary.to_csv(
    OUT_DIR
    /
    "ptbxl_patient_cluster_bootstrap_per_class.csv",
    index=False
)


point_df.to_csv(
    OUT_DIR
    /
    "ptbxl_bootstrap_point_estimates.csv",
    index=False
)


np.savez_compressed(
    OUT_DIR
    /
    "ptbxl_patient_cluster_bootstrap_replicates.npz",

    consultant_vs_base_macro_AUROC=
        overall_delta[
            "consultant_vs_base"
        ][
            "macro_AUROC"
        ],

    consultant_vs_base_macro_AP=
        overall_delta[
            "consultant_vs_base"
        ][
            "macro_AP"
        ],

    all_vs_base_macro_AUROC=
        overall_delta[
            "all_clinicians_vs_base"
        ][
            "macro_AUROC"
        ],

    all_vs_base_macro_AP=
        overall_delta[
            "all_clinicians_vs_base"
        ][
            "macro_AP"
        ],

    all_vs_consultant_macro_AUROC=
        overall_delta[
            "all_clinicians_vs_consultant"
        ][
            "macro_AUROC"
        ],

    all_vs_consultant_macro_AP=
        overall_delta[
            "all_clinicians_vs_consultant"
        ][
            "macro_AP"
        ],
)


# ======================================================================================
# 13. MANIFEST
# ======================================================================================

manifest = {

    "stage":
        "Step 4C patient-cluster paired bootstrap",

    "source":
        str(
            PRED_PATH
        ),

    "bootstrap_seed":
        SEED,

    "bootstrap_replicates":
        N_BOOT,

    "resampling_unit":
        "patient",

    "unique_patients":
        N_PATIENTS,

    "ECG_records":
        len(
            df
        ),

    "confidence_interval":
        "2.5th to 97.5th percentile",

    "comparisons": [
        comparison_name
        for comparison_name, _, _ in COMPARISONS
    ],

    "model_inference_performed":
        False,

    "training_performed":
        False,
}


with open(
    OUT_DIR
    /
    "ptbxl_patient_cluster_bootstrap_manifest.json",
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


print("\n\n" + "=" * 100)
print("STEP 4C COMPLETE — NO MODEL INFERENCE OR TRAINING")
print("=" * 100)


print(
    """
Please paste back ONLY:

PATIENT-CLUSTER BOOTSTRAP — OVERALL PERFORMANCE DIFFERENCE

The per-class bootstrap file is already saved and we can inspect it
after establishing the overall result.
"""
)

In [ ]:
# ======================================================================================
# STEP 4D — SUMMARISE SAVED PER-CLASS PATIENT-BOOTSTRAP RESULTS
#
# NO TRAINING.
# NO INFERENCE.
# ======================================================================================

from pathlib import Path

import numpy as np
import pandas as pd


PATH = Path(
    "definitive_clean_gaze/ptbxl_evaluation/"
    "ptbxl_patient_cluster_bootstrap_per_class.csv"
)


df = pd.read_csv(PATH)


# Restrict primary interpretation to the two gaze-vs-base comparisons.
primary = df[
    df["comparison"].isin(
        [
            "consultant_vs_base",
            "all_clinicians_vs_base",
        ]
    )
].copy()


# CI classification:
#
# CI entirely > 0  -> bootstrap-supported improvement
# CI entirely < 0  -> bootstrap-supported deterioration
# otherwise        -> uncertain
def classify_effect(row):

    if row["CI_low"] > 0:
        return "improved"

    if row["CI_high"] < 0:
        return "worsened"

    return "uncertain"


primary["bootstrap_direction"] = primary.apply(
    classify_effect,
    axis=1
)


# ======================================================================================
# 1. COUNTS
# ======================================================================================

counts = (
    primary
    .groupby(
        [
            "comparison",
            "metric",
            "bootstrap_direction",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)


for column in [
    "improved",
    "worsened",
    "uncertain",
]:

    if column not in counts.columns:
        counts[column] = 0


print("=" * 100)
print("PER-CLASS BOOTSTRAP DIRECTION COUNTS")
print("=" * 100)

print(
    counts[
        [
            "comparison",
            "metric",
            "improved",
            "worsened",
            "uncertain",
        ]
    ].to_string(
        index=False
    )
)


# ======================================================================================
# 2. FULL CLASS TABLE
# ======================================================================================

display_cols = [
    "comparison",
    "class",
    "metric",
    "positive_n",
    "point_delta",
    "CI_low",
    "CI_high",
    "valid_bootstraps",
    "bootstrap_direction",
]


print("\n" + "=" * 100)
print("PER-CLASS BOOTSTRAP RESULTS")
print("=" * 100)

print(
    primary[
        display_cols
    ]
    .round(6)
    .to_string(
        index=False
    )
)


# ======================================================================================
# 3. ROBUST IMPROVEMENTS ONLY
# ======================================================================================

improved = primary[
    primary[
        "bootstrap_direction"
    ]
    ==
    "improved"
].copy()


print("\n" + "=" * 100)
print("CLASSES WITH CI ENTIRELY ABOVE ZERO")
print("=" * 100)

if len(improved) == 0:

    print("NONE")

else:

    print(
        improved[
            display_cols
        ]
        .sort_values(
            [
                "comparison",
                "metric",
                "point_delta"
            ],
            ascending=[
                True,
                True,
                False
            ]
        )
        .round(6)
        .to_string(
            index=False
        )
    )


# ======================================================================================
# 4. ROBUST DETERIORATIONS ONLY
# ======================================================================================

worsened = primary[
    primary[
        "bootstrap_direction"
    ]
    ==
    "worsened"
].copy()


print("\n" + "=" * 100)
print("CLASSES WITH CI ENTIRELY BELOW ZERO")
print("=" * 100)

print(
    worsened[
        display_cols
    ]
    .sort_values(
        [
            "comparison",
            "metric",
            "point_delta"
        ]
    )
    .round(6)
    .to_string(
        index=False
    )
)


# ======================================================================================
# 5. FLAG VERY SMALL POSITIVE COUNTS
#
# This does NOT exclude them from any primary analysis.
# It is only an interpretation flag.
# ======================================================================================

primary[
    "rare_class_flag"
] = np.where(
    primary[
        "positive_n"
    ]
    <
    20,
    "YES (<20 positives)",
    "NO"
)


primary.to_csv(
    "definitive_clean_gaze/ptbxl_evaluation/"
    "ptbxl_per_class_bootstrap_interpreted.csv",
    index=False
)


print(
    "\n✓ Interpreted per-class bootstrap table saved."
)

In [ ]:
# ======================================================================================
# STEP 5A — LUDB PROVENANCE / SIGNAL / DIAGNOSTIC-ENDPOINT AUDIT
#
# READ ONLY.
#
# NO MODEL INFERENCE.
# NO TRAINING.
# NO DIAGNOSTIC LABEL MAPPING.
# NO RESAMPLING FOR MODEL USE.
#
# Questions:
#   1. Which LUDB release/copy is present locally?
#   2. Are all 200 records present?
#   3. What are the native signal geometry, lead order and sampling frequency?
#   4. What diagnostic information is actually supplied?
#   5. Is there a direct, defensible correspondence with any of the frozen
#      PTB-XL model's 23 output classes?
#   6. What did our historical LUDB code actually do?
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import ast
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import wfdb
except ImportError as exc:
    raise ImportError(
        "wfdb is required. Install with:\n\n"
        "    pip install wfdb\n"
    ) from exc


# ======================================================================================
# 1. FROZEN MODEL VOCABULARY — FOR COMPARISON ONLY
#
# IMPORTANT:
# We do NOT map LUDB diagnoses to these classes in this audit.
# ======================================================================================

MODEL_CLASSES = [
    "AMI",
    "CLBBB",
    "CRBBB",
    "ILBBB",
    "IMI",
    "IRBBB",
    "ISCA",
    "ISCI",
    "ISC_",
    "IVCD",
    "LAFB/LPFB",
    "LAO/LAE",
    "LMI",
    "LVH",
    "NORM",
    "NST_",
    "PMI",
    "RAO/RAE",
    "RVH",
    "SEHYP",
    "STTC",
    "WPW",
    "_AVB",
]


EXPECTED_NATIVE_LEADS = [
    "I",
    "II",
    "III",
    "AVR",
    "AVL",
    "AVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]


# ======================================================================================
# 2. LOCATE LUDB
#
# Prefer a directory containing:
#     ludb.csv
#     data/
#
# Do not silently choose between multiple copies.
# ======================================================================================

csv_candidates = list(
    Path(".").glob(
        "**/ludb.csv"
    )
)


candidate_roots = []


for csv_path in csv_candidates:

    root = csv_path.parent

    data_dir = (
        root
        /
        "data"
    )

    if data_dir.exists():

        candidate_roots.append(
            root
        )


candidate_roots = sorted(
    set(
        candidate_roots
    ),
    key=lambda p: str(p)
)


print("=" * 100)
print("LUDB DATASET SEARCH")
print("=" * 100)


if len(candidate_roots) == 0:

    raise FileNotFoundError(
        "\nCould not find LUDB locally.\n\n"
        "Expected something like:\n"
        "    ludb/\n"
        "        ludb.csv\n"
        "        data/\n"
    )


print(
    "Candidate LUDB roots:"
)


for i, root in enumerate(
    candidate_roots,
    start=1
):

    print(
        f"  {i}. {root}"
    )


if len(candidate_roots) != 1:

    raise RuntimeError(
        "\nMore than one LUDB copy was found.\n"
        "Do not automatically choose between them."
    )


LUDB_ROOT = candidate_roots[0]

LUDB_CSV = (
    LUDB_ROOT
    /
    "ludb.csv"
)

DATA_DIR = (
    LUDB_ROOT
    /
    "data"
)


print(
    "\nSelected root:"
)

print(
    LUDB_ROOT
)


# ======================================================================================
# 3. TOP-LEVEL RELEASE FILES
# ======================================================================================

print("\n" + "=" * 100)
print("LUDB TOP-LEVEL FILE AUDIT")
print("=" * 100)


for filename in [
    "README",
    "RECORDS",
    "ANNOTATORS",
    "SHA256SUMS.txt",
    "LICENSE.txt",
    "ludb.csv",
]:

    path = (
        LUDB_ROOT
        /
        filename
    )

    print(
        f"{filename:20s}",
        "FOUND"
        if path.exists()
        else
        "MISSING"
    )


# ======================================================================================
# 4. LOAD ludb.csv WITHOUT ASSUMING ITS SCHEMA
# ======================================================================================

ludb = pd.read_csv(
    LUDB_CSV
)


print("\n" + "=" * 100)
print("LUDB.CSV STRUCTURE")
print("=" * 100)


print(
    "Shape:",
    ludb.shape
)


print(
    "\nColumns:"
)


for i, column in enumerate(
    ludb.columns
):

    print(
        f"{i:02d}  {column}"
    )


print(
    "\nFirst 5 rows:"
)


print(
    ludb.head(
        5
    ).to_string(
        index=False
    )
)


# ======================================================================================
# 5. IDENTIFY RECORD FILES
#
# WFDB headers are the safest source of record identity.
# ======================================================================================

header_paths = sorted(
    DATA_DIR.glob(
        "*.hea"
    )
)


record_names = [
    path.stem
    for path in header_paths
]


print("\n" + "=" * 100)
print("WFDB RECORD COUNT")
print("=" * 100)


print(
    "Header files:",
    len(
        header_paths
    )
)


print(
    "Unique record names:",
    len(
        set(
            record_names
        )
    )
)


if len(
    header_paths
) != 200:

    print(
        "\nWARNING:"
        f" official LUDB contains 200 records, but this copy has {len(header_paths)} headers."
    )


if len(
    set(
        record_names
    )
) != len(
    record_names
):

    raise RuntimeError(
        "Duplicate WFDB record names detected."
    )


# ======================================================================================
# 6. READ ALL RECORD HEADERS / SIGNAL GEOMETRY
#
# We load each record to verify:
#     n samples
#     n leads
#     fs
#     lead order
#     finiteness
#
# This is READ-ONLY.
# ======================================================================================

geometry_rows = []

header_comment_rows = []


for i, record_name in enumerate(
    record_names
):

    record_path = (
        DATA_DIR
        /
        record_name
    )


    signal, fields = wfdb.rdsamp(
        str(
            record_path
        )
    )


    leads = [
        str(x).upper()
        for x in fields.get(
            "sig_name",
            []
        )
    ]


    comments = fields.get(
        "comments",
        []
    )


    geometry_rows.append(
        {
            "record":
                record_name,

            "n_samples":
                int(
                    signal.shape[0]
                ),

            "n_leads":
                int(
                    signal.shape[1]
                ),

            "fs":
                float(
                    fields.get(
                        "fs",
                        np.nan
                    )
                ),

            "lead_order":
                " | ".join(
                    leads
                ),

            "finite":
                bool(
                    np.all(
                        np.isfinite(
                            signal
                        )
                    )
                ),

            "min_value":
                float(
                    np.min(
                        signal
                    )
                ),

            "max_value":
                float(
                    np.max(
                        signal
                    )
                ),
        }
    )


    for comment in comments:

        header_comment_rows.append(
            {
                "record":
                    record_name,

                "comment":
                    str(
                        comment
                    ),
            }
        )


    if (
        (i + 1) % 50 == 0
        or
        (i + 1) == len(
            record_names
        )
    ):

        print(
            f"Audited {i + 1}/{len(record_names)} records"
        )


geometry_df = pd.DataFrame(
    geometry_rows
)


comments_df = pd.DataFrame(
    header_comment_rows
)


print("\n" + "=" * 100)
print("NATIVE LUDB SIGNAL GEOMETRY")
print("=" * 100)


print(
    "Unique sampling frequencies:"
)

print(
    geometry_df[
        "fs"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\nUnique sample lengths:"
)

print(
    geometry_df[
        "n_samples"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\nUnique lead counts:"
)

print(
    geometry_df[
        "n_leads"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\nUnique lead orders:"
)

print(
    geometry_df[
        "lead_order"
    ]
    .value_counts()
    .to_string()
)


print(
    "\nAll finite:",
    bool(
        geometry_df[
            "finite"
        ].all()
    )
)


# Strong geometry checks
if not (
    geometry_df[
        "n_leads"
    ]
    ==
    12
).all():

    raise RuntimeError(
        "At least one LUDB record does not have 12 leads."
    )


if not (
    geometry_df[
        "fs"
    ]
    ==
    500.0
).all():

    raise RuntimeError(
        "At least one LUDB record is not sampled at 500 Hz."
    )


if not (
    geometry_df[
        "n_samples"
    ]
    ==
    5000
).all():

    raise RuntimeError(
        "At least one LUDB record is not exactly 10 seconds / 5000 samples."
    )


expected_lead_string = " | ".join(
    EXPECTED_NATIVE_LEADS
)


if not (
    geometry_df[
        "lead_order"
    ]
    ==
    expected_lead_string
).all():

    raise RuntimeError(
        "\nAt least one LUDB record has unexpected lead order."
    )


if not geometry_df[
    "finite"
].all():

    raise RuntimeError(
        "Non-finite waveform values detected."
    )


print(
    "\n✓ All records are native 500-Hz, 5000-sample, 12-lead ECGs "
    "in the expected lead order."
)


# ======================================================================================
# 7. MODEL-GEOMETRY IMPLICATION
#
# NO RESAMPLING IS ACTUALLY PERFORMED HERE.
# ======================================================================================

print("\n" + "=" * 100)
print("MODEL-GEOMETRY COMPATIBILITY")
print("=" * 100)


print(
    "Native LUDB record:"
)

print(
    "    (12 leads, 5000 samples @ 500 Hz)"
)


print(
    "\nFrozen PTB-XL model expects:"
)

print(
    "    (12 leads, 1000 samples @ 100 Hz, 1 channel)"
)


print(
    "\nTherefore a predefined 500 -> 100 Hz resampling operation "
    "would be required before any model inference."
)


print(
    "\nNO RESAMPLING WAS PERFORMED IN THIS AUDIT."
)


# ======================================================================================
# 8. SUMMARISE HEADER COMMENTS
#
# This lets us see exactly how diagnosis/rhythm metadata are written.
# ======================================================================================

print("\n" + "=" * 100)
print("WFDB HEADER COMMENT EXAMPLES")
print("=" * 100)


if len(
    comments_df
) == 0:

    print(
        "No WFDB comments found."
    )

else:

    print(
        comments_df.head(
            80
        ).to_string(
            index=False
        )
    )


# Frequency of comment prefixes
if len(
    comments_df
) > 0:

    comments_df[
        "prefix"
    ] = comments_df[
        "comment"
    ].astype(
        str
    ).str.split(
        ":",
        n=1
    ).str[
        0
    ].str.strip()


    print(
        "\nComment-prefix counts:"
    )


    print(
        comments_df[
            "prefix"
        ]
        .value_counts()
        .head(
            30
        )
        .to_string()
    )


# ======================================================================================
# 9. AUTOMATIC ludb.csv COLUMN AUDIT
#
# For categorical/string-like columns, print unique values and frequencies.
#
# We deliberately DO NOT interpret these yet.
# ======================================================================================

print("\n" + "=" * 100)
print("LUDB.CSV CATEGORICAL / DIAGNOSTIC CONTENT")
print("=" * 100)


categorical_columns = []


for column in ludb.columns:

    series = ludb[
        column
    ]


    # Candidate categorical/diagnosis-like columns:
    # object/string OR modest number of unique values.
    nunique = series.nunique(
        dropna=True
    )


    if (
        pd.api.types.is_object_dtype(
            series
        )
        or
        pd.api.types.is_string_dtype(
            series
        )
        or
        nunique <= 50
    ):

        categorical_columns.append(
            column
        )


for column in categorical_columns:

    series = ludb[
        column
    ]


    print(
        "\n" +
        "-" * 100
    )

    print(
        f"COLUMN: {column}"
    )


    print(
        "Missing:",
        int(
            series.isna().sum()
        )
    )


    print(
        "Unique non-missing:",
        int(
            series.nunique(
                dropna=True
            )
        )
    )


    value_counts = (
        series.astype(
            "string"
        )
        .fillna(
            "<NA>"
        )
        .value_counts(
            dropna=False
        )
    )


    # Prevent pathological output if an unexpected free-text column exists.
    print(
        value_counts.head(
            100
        ).to_string()
    )


# ======================================================================================
# 10. SEARCH COLUMN NAMES FOR LIKELY CLINICAL ENDPOINTS
# ======================================================================================

clinical_keywords = [
    "diagn",
    "rhythm",
    "block",
    "conduction",
    "ische",
    "infar",
    "scar",
    "repolar",
    "axis",
    "hyper",
    "pace",
    "extra",
    "abnormal",
]


likely_clinical_columns = []


for column in ludb.columns:

    normalized = str(
        column
    ).lower()


    if any(
        keyword in normalized
        for keyword in clinical_keywords
    ):

        likely_clinical_columns.append(
            column
        )


print("\n" + "=" * 100)
print("LIKELY CLINICAL-ENDPOINT COLUMNS")
print("=" * 100)


if likely_clinical_columns:

    for column in likely_clinical_columns:

        print(
            column
        )

else:

    print(
        "NONE IDENTIFIED FROM COLUMN NAMES — inspect the categorical output above."
    )


# ======================================================================================
# 11. SEARCH EXACT MODEL-CLASS TOKENS IN LUDB METADATA
#
# IMPORTANT:
# Exact lexical overlap is NOT proof of label equivalence.
# This is merely an audit of whether the PTB-XL abbreviations themselves
# occur anywhere in LUDB metadata.
# ======================================================================================

ludb_text = "\n".join(
    ludb.astype(
        str
    ).fillna(
        ""
    ).agg(
        " | ".join,
        axis=1
    ).tolist()
)


comment_text = "\n".join(
    comments_df[
        "comment"
    ].astype(
        str
    ).tolist()
) if len(
    comments_df
) else ""


combined_metadata_text = (
    ludb_text
    +
    "\n"
    +
    comment_text
).upper()


lexical_rows = []


for class_name in MODEL_CLASSES:

    literal_count = combined_metadata_text.count(
        class_name.upper()
    )


    lexical_rows.append(
        {
            "PTBXL_model_class":
                class_name,

            "literal_occurrences_in_LUDB_metadata":
                literal_count,
        }
    )


lexical_df = pd.DataFrame(
    lexical_rows
)


print("\n" + "=" * 100)
print("EXACT PTB-XL CLASS-TOKEN OVERLAP")
print("=" * 100)


print(
    lexical_df.to_string(
        index=False
    )
)


print(
    "\nIMPORTANT: lexical overlap does NOT establish diagnostic equivalence."
)


# ======================================================================================
# 12. IDENTIFY POSSIBLE RECORD-ID COLUMN IN ludb.csv
# ======================================================================================

record_id_candidates = []


for column in ludb.columns:

    values = set(
        ludb[
            column
        ]
        .dropna()
        .astype(
            str
        )
    )


    overlap = len(
        values
        &
        set(
            record_names
        )
    )


    if overlap > 0:

        record_id_candidates.append(
            {
                "column":
                    column,

                "matching_record_names":
                    overlap,

                "column_unique_values":
                    len(
                        values
                    ),
            }
        )


record_id_df = pd.DataFrame(
    record_id_candidates
)


print("\n" + "=" * 100)
print("LUDB.CSV ↔ WFDB RECORD-ID MATCH")
print("=" * 100)


if len(
    record_id_df
) == 0:

    print(
        "No direct string-identical record-ID column found."
    )

else:

    print(
        record_id_df
        .sort_values(
            "matching_record_names",
            ascending=False
        )
        .to_string(
            index=False
        )
    )


# ======================================================================================
# 13. PATIENT / RECORD UNIQUENESS AUDIT
#
# We do not assume whether each record = one patient.
# ======================================================================================

patient_candidates = [
    column
    for column in ludb.columns
    if any(
        token in str(
            column
        ).lower()
        for token in [
            "patient",
            "subject",
            "id",
        ]
    )
]


print("\n" + "=" * 100)
print("POSSIBLE PATIENT / SUBJECT IDENTIFIERS")
print("=" * 100)


if len(
    patient_candidates
) == 0:

    print(
        "No obvious patient/subject ID column by name."
    )

else:

    for column in patient_candidates:

        print(
            f"{column}: "
            f"{ludb[column].nunique(dropna=True)} unique / "
            f"{len(ludb)} rows"
        )


# ======================================================================================
# 14. HISTORICAL LOCAL-CODE SEARCH
#
# We want to know what the OLD LUDB evaluation actually did.
#
# Search only text-like coding files beneath the current project.
# We do NOT execute them.
# ======================================================================================

SEARCH_EXTENSIONS = {
    ".py",
    ".ipynb",
    ".txt",
    ".md",
}


SEARCH_TERMS = [
    "ludb",
    "by_name",
    "skip_mismatch",
    "gaze_model.h5",
    "gaze_finetuned.h5",
    "gaze_model_3.h5",
    "resample",
    "downsample",
]


historical_hits = []


for path in Path(".").rglob("*"):

    if not path.is_file():
        continue


    if path.suffix.lower() not in SEARCH_EXTENSIONS:
        continue


    # Avoid generated outputs / environments.
    path_text = str(
        path
    ).lower()


    if any(
        excluded in path_text
        for excluded in [
            ".venv",
            "site-packages",
            "__pycache__",
            "node_modules",
            "definitive_clean_gaze",
        ]
    ):
        continue


    try:

        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception:

        continue


    text_lower = text.lower()


    matched_terms = [
        term
        for term in SEARCH_TERMS
        if term.lower() in text_lower
    ]


    if matched_terms:

        historical_hits.append(
            {
                "file":
                    str(
                        path
                    ),

                "matched_terms":
                    ", ".join(
                        matched_terms
                    ),
            }
        )


historical_hits_df = pd.DataFrame(
    historical_hits
)


print("\n" + "=" * 100)
print("HISTORICAL LUDB-RELATED CODE FILES")
print("=" * 100)


if len(
    historical_hits_df
) == 0:

    print(
        "No historical LUDB-related code/notebook files found."
    )

else:

    print(
        historical_hits_df.to_string(
            index=False
        )
    )


# ======================================================================================
# 15. PRINT TARGETED CODE SNIPPETS
#
# Only snippets around important LUDB/preprocessing/loading terms.
# Maximum output is deliberately capped.
# ======================================================================================

print("\n" + "=" * 100)
print("HISTORICAL LUDB CODE SNIPPETS")
print("=" * 100)


MAX_FILES = 10
MAX_MATCHES_PER_FILE = 12


if len(
    historical_hits_df
) == 0:

    print(
        "NONE"
    )

else:

    for file_number, file_path in enumerate(
        historical_hits_df[
            "file"
        ].tolist()[
            :MAX_FILES
        ],
        start=1
    ):

        path = Path(
            file_path
        )


        try:

            text = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

        except Exception:

            continue


        # For .ipynb the raw JSON is still searchable.
        lines = text.splitlines()


        print(
            "\n" +
            "#" * 100
        )

        print(
            f"FILE {file_number}: {path}"
        )

        print(
            "#" * 100
        )


        match_count = 0


        for line_number, line in enumerate(
            lines
        ):

            line_lower = line.lower()


            if not any(
                term.lower() in line_lower
                for term in SEARCH_TERMS
            ):

                continue


            start = max(
                0,
                line_number - 2
            )

            end = min(
                len(
                    lines
                ),
                line_number + 3
            )


            print(
                f"\n--- around line {line_number + 1} ---"
            )


            for j in range(
                start,
                end
            ):

                # Cap individual very-long notebook JSON lines.
                display_line = lines[
                    j
                ]


                if len(
                    display_line
                ) > 500:

                    display_line = (
                        display_line[
                            :500
                        ]
                        +
                        " ... [TRUNCATED]"
                    )


                print(
                    f"{j + 1:05d}: {display_line}"
                )


            match_count += 1


            if match_count >= MAX_MATCHES_PER_FILE:

                print(
                    "\n[Further matches in this file omitted]"
                )

                break


# ======================================================================================
# 16. SAVE AUDIT TABLES
# ======================================================================================

AUDIT_DIR = Path(
    "definitive_clean_gaze/ludb_audit"
)


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


geometry_df.to_csv(
    AUDIT_DIR
    /
    "ludb_signal_geometry.csv",
    index=False
)


comments_df.to_csv(
    AUDIT_DIR
    /
    "ludb_header_comments.csv",
    index=False
)


lexical_df.to_csv(
    AUDIT_DIR
    /
    "ludb_ptbxl_literal_class_overlap.csv",
    index=False
)


if len(
    historical_hits_df
) > 0:

    historical_hits_df.to_csv(
        AUDIT_DIR
        /
        "historical_ludb_code_files.csv",
        index=False
    )


if len(
    record_id_df
) > 0:

    record_id_df.to_csv(
        AUDIT_DIR
        /
        "ludb_record_id_candidates.csv",
        index=False
    )


# Save raw ludb.csv column inventory.
column_inventory = []


for column in ludb.columns:

    column_inventory.append(
        {
            "column":
                column,

            "dtype":
                str(
                    ludb[
                        column
                    ].dtype
                ),

            "missing_n":
                int(
                    ludb[
                        column
                    ].isna().sum()
                ),

            "unique_nonmissing_n":
                int(
                    ludb[
                        column
                    ].nunique(
                        dropna=True
                    )
                ),
        }
    )


pd.DataFrame(
    column_inventory
).to_csv(
    AUDIT_DIR
    /
    "ludb_column_inventory.csv",
    index=False
)


# ======================================================================================
# 17. AUDIT MANIFEST
# ======================================================================================

manifest = {

    "stage":
        "Step 5A LUDB read-only provenance and endpoint audit",

    "LUDB_root":
        str(
            LUDB_ROOT
        ),

    "ludb_csv_shape":
        [
            int(
                ludb.shape[0]
            ),
            int(
                ludb.shape[1]
            ),
        ],

    "WFDB_record_count":
        int(
            len(
                record_names
            )
        ),

    "sampling_frequencies":
        sorted(
            geometry_df[
                "fs"
            ].unique().tolist()
        ),

    "sample_lengths":
        sorted(
            geometry_df[
                "n_samples"
            ].unique().tolist()
        ),

    "lead_counts":
        sorted(
            geometry_df[
                "n_leads"
            ].unique().tolist()
        ),

    "lead_orders":
        sorted(
            geometry_df[
                "lead_order"
            ].unique().tolist()
        ),

    "frozen_classifier_expected_sampling_frequency_Hz":
        100,

    "frozen_classifier_expected_samples":
        1000,

    "LUDB_to_classifier_resampling_required":
        True,

    "resampling_performed_in_this_step":
        False,

    "diagnostic_mapping_performed":
        False,

    "model_inference_performed":
        False,

    "training_performed":
        False,

    "historical_code_files_found":
        (
            historical_hits_df[
                "file"
            ].tolist()
            if len(
                historical_hits_df
            ) > 0
            else []
        ),
}


with open(
    AUDIT_DIR
    /
    "ludb_audit_manifest.json",
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ======================================================================================
# 18. FINAL REPORT
# ======================================================================================

print("\n\n" + "=" * 100)
print("STEP 5A COMPLETE — READ ONLY")
print("=" * 100)


print(
    """
Please paste back these FOUR sections:

1. LUDB.CSV STRUCTURE
   - shape
   - column names
   - first five rows

2. NATIVE LUDB SIGNAL GEOMETRY

3. LUDB.CSV CATEGORICAL / DIAGNOSTIC CONTENT
   This may be long — most importantly include any columns referring to:
       rhythm
       conduction abnormalities
       myocardial infarction / ischemia / scar
       pacemaker
       hypertrophy
       other diagnoses

4. HISTORICAL LUDB CODE FILES + relevant HISTORICAL LUDB CODE SNIPPETS

DO NOT run LUDB through the three models yet.

Once we see those outputs, we will decide whether LUDB supports:
    A. a legitimate mapped diagnostic evaluation,
    B. a narrower endpoint such as normal/abnormal,
or
    C. no defensible classifier endpoint at all.
"""
)

In [ ]:
# ======================================================================================
# STEP 5B — FREEZE LUDB TASKS, FEATURE LAYER AND PREPROCESSING
#
# NO LUDB MODEL INFERENCE
# NO CLASSIFIER TRAINING
#
# Purpose:
#   1. Define LUDB downstream tasks BEFORE seeing model performance.
#   2. Verify their exact counts.
#   3. Freeze the representation layer.
#   4. Freeze 500 Hz -> 100 Hz preprocessing.
#
# LUDB will be treated as:
#   EXTERNAL REPRESENTATION-TRANSFER / LINEAR-PROBE EVALUATION
#
# It is NOT direct validation of the frozen 23-class classifier.
# ======================================================================================

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.signal import resample_poly
import wfdb


# ======================================================================================
# 1. PATHS
# ======================================================================================

LUDB_ROOT = Path("Test ECG")
LUDB_CSV = LUDB_ROOT / "ludb.csv"
DATA_DIR = LUDB_ROOT / "data"

ROOT = Path("definitive_clean_gaze")

STEP1_MANIFEST = (
    ROOT
    /
    "step1_manifest.json"
)

OUT_DIR = (
    ROOT
    /
    "ludb_evaluation"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================================
# 2. LOAD METADATA
# ======================================================================================

ludb = pd.read_csv(
    LUDB_CSV
)


if len(ludb) != 200:
    raise RuntimeError(
        f"Expected 200 LUDB records, found {len(ludb)}."
    )


if ludb["ID"].nunique() != 200:
    raise RuntimeError(
        "LUDB ID is not unique."
    )


print("=" * 100)
print("LUDB RECORD AUDIT")
print("=" * 100)

print(
    "Records:",
    len(ludb)
)

print(
    "Unique IDs:",
    ludb["ID"].nunique()
)


# ======================================================================================
# 3. STRING HELPERS
# ======================================================================================

def clean_text(value):

    if pd.isna(value):
        return ""

    return (
        str(value)
        .replace("\\n", "\n")
        .strip()
        .lower()
    )


def split_lines(value):

    text = clean_text(value)

    if not text:
        return []

    return [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]


# ======================================================================================
# 4. TASK 1 — ATRIAL FIBRILLATION
#
# Exploratory because prevalence is low.
# Includes:
#     Atrial fibrillation
#     Atrial fibrillation + aberrant conduction
# ======================================================================================

rhythm_text = ludb[
    "Rhythms"
].apply(
    clean_text
)


ludb[
    "target_AFib"
] = rhythm_text.str.contains(
    "atrial fibrillation",
    regex=False
).astype(
    int
)


# ======================================================================================
# 5. TASK 2 — ANY CONDUCTION ABNORMALITY
#
# Positive:
#     any non-missing entry in "Conduction abnormalities"
#
# Negative:
#     no documented conduction abnormality
#
# This is a LUDB-specific downstream task, NOT a direct PTB-XL output mapping.
# ======================================================================================

conduction_text = ludb[
    "Conduction abnormalities"
].apply(
    clean_text
)


ludb[
    "target_Conduction"
] = (
    conduction_text
    !=
    ""
).astype(
    int
)


# ======================================================================================
# 6. TASK 3 — LEFT VENTRICULAR HYPERTROPHY
#
# Exact LUDB diagnosis string.
# ======================================================================================

hypertrophy_text = ludb[
    "Hypertrophies"
].apply(
    clean_text
)


ludb[
    "target_LVH"
] = hypertrophy_text.str.contains(
    "left ventricular hypertrophy",
    regex=False
).astype(
    int
)


# ======================================================================================
# 7. TASK 4 — CARDIAC PACING
#
# Exploratory because only ~10 positives expected.
#
# Positive:
#     any non-empty Cardiac pacing field.
#
# IMPORTANT:
#     This is NOT mapped to PTB-XL "PMI".
# ======================================================================================

pacing_text = ludb[
    "Cardiac pacing"
].apply(
    clean_text
)


ludb[
    "target_Paced"
] = (
    pacing_text
    !=
    ""
).astype(
    int
)


# ======================================================================================
# 8. TASK 5 — DEFINITE ISCHAEMIA / SCAR / STEMI
#
# LUDB contains:
#
#   definite:
#       Ischemia: ...
#       Scar formation: ...
#       STEMI: ...
#
#   ambiguous:
#       Undefined ischemia/scar/supp.nstemi: ...
#
# Definition:
#
#   POSITIVE
#       >=1 definite line
#
#   NEGATIVE
#       completely empty Ischemia field
#
#   EXCLUDED / INDETERMINATE
#       field contains only "Undefined ischemia/scar/supp.nstemi"
#
# If definite and undefined findings coexist:
#       POSITIVE
#
# This definition is frozen before model evaluation.
# ======================================================================================

def classify_ischaemia(value):

    lines = split_lines(
        value
    )


    if len(lines) == 0:

        return 0


    definite = False
    ambiguous = False


    for line in lines:

        if (
            line.startswith(
                "ischemia:"
            )
            or
            line.startswith(
                "scar formation:"
            )
            or
            line.startswith(
                "stemi:"
            )
        ):

            definite = True


        if line.startswith(
            "undefined ischemia/scar/supp.nstemi:"
        ):

            ambiguous = True


    if definite:
        return 1


    if ambiguous:
        return np.nan


    # Defensive:
    # unexpected non-empty Ischemia content should not silently
    # become negative.
    raise RuntimeError(
        f"Unexpected Ischemia field content: {value}"
    )


ludb[
    "target_Ischaemia_related"
] = ludb[
    "Ischemia"
].apply(
    classify_ischaemia
)


# ======================================================================================
# 9. TASK COUNTS
# ======================================================================================

TASKS = {

    "Conduction":
        "target_Conduction",

    "LVH":
        "target_LVH",

    "Ischaemia_related":
        "target_Ischaemia_related",

    "AFib":
        "target_AFib",

    "Paced":
        "target_Paced",
}


PRIMARY_TASKS = [
    "Conduction",
    "LVH",
    "Ischaemia_related",
]


EXPLORATORY_TASKS = [
    "AFib",
    "Paced",
]


task_rows = []


for task_name, column in TASKS.items():

    valid = ludb[
        column
    ].notna()


    y = ludb.loc[
        valid,
        column
    ].astype(
        int
    )


    task_rows.append(
        {
            "task":
                task_name,

            "role":
                (
                    "primary"
                    if task_name in PRIMARY_TASKS
                    else
                    "exploratory"
                ),

            "N":
                int(
                    valid.sum()
                ),

            "positive":
                int(
                    y.sum()
                ),

            "negative":
                int(
                    len(y)
                    -
                    y.sum()
                ),

            "excluded_indeterminate":
                int(
                    (
                        ~valid
                    ).sum()
                ),

            "prevalence":
                float(
                    y.mean()
                ),
        }
    )


task_df = pd.DataFrame(
    task_rows
)


print("\n" + "=" * 100)
print("FROZEN LUDB TASK DEFINITIONS")
print("=" * 100)

print(
    task_df
    .round(4)
    .to_string(
        index=False
    )
)


# ======================================================================================
# 10. CHECK AGAINST HISTORICAL COUNTS
#
# We are NOT using the old results.
#
# These checks merely tell us whether our clean definitions reproduce
# the previously observed dataset counts.
# ======================================================================================

EXPECTED_COUNTS = {

    "AFib":
        (
            200,
            15
        ),

    "Ischaemia_related":
        (
            192,
            43
        ),

    "Conduction":
        (
            200,
            66
        ),

    "LVH":
        (
            200,
            108
        ),

    "Paced":
        (
            200,
            10
        ),
}


print("\n" + "=" * 100)
print("HISTORICAL COUNT REPRODUCTION")
print("=" * 100)


all_counts_match = True


for task_name, (
    expected_n,
    expected_positive
) in EXPECTED_COUNTS.items():

    row = task_df[
        task_df[
            "task"
        ]
        ==
        task_name
    ].iloc[
        0
    ]


    actual_n = int(
        row[
            "N"
        ]
    )


    actual_positive = int(
        row[
            "positive"
        ]
    )


    match = (
        actual_n
        ==
        expected_n
        and
        actual_positive
        ==
        expected_positive
    )


    print(
        f"{task_name:20s} "
        f"N={actual_n:3d} "
        f"positive={actual_positive:3d} "
        f"| expected N={expected_n:3d}, positive={expected_positive:3d} "
        f"| {'MATCH' if match else 'DIFFERENT'}"
    )


    if not match:

        all_counts_match = False


if all_counts_match:

    print(
        "\n✓ All five clean task definitions reproduce the historical dataset counts."
    )

else:

    print(
        "\n⚠ At least one clean task definition differs from the historical count."
    )


# ======================================================================================
# 11. FREEZE FEATURE LAYER
#
# Historical LUDB work used dense_1, a 64-dimensional frozen representation.
#
# We inspect it here before using LUDB signals.
# ======================================================================================

with open(
    STEP1_MANIFEST,
    "r"
) as f:

    step1_manifest = json.load(
        f
    )


BASE_MODEL_PATH = Path(
    step1_manifest[
        "base_model"
    ]
)


base_model = tf.keras.models.load_model(
    BASE_MODEL_PATH,
    compile=False
)


layer_names = [
    layer.name
    for layer in base_model.layers
]


if "dense_1" not in layer_names:

    raise RuntimeError(
        "dense_1 not found in base model."
    )


dense1_index = layer_names.index(
    "dense_1"
)


print("\n" + "=" * 100)
print("FROZEN REPRESENTATION LAYER AUDIT")
print("=" * 100)


start = max(
    0,
    dense1_index - 4
)

end = min(
    len(
        base_model.layers
    ),
    dense1_index + 4
)


for i in range(
    start,
    end
):

    layer = base_model.layers[
        i
    ]


    activation = ""


    if hasattr(
        layer,
        "activation"
    ):

        try:

            activation = (
                " | activation="
                +
                layer.activation.__name__
            )

        except Exception:
            pass


    print(
        f"{i:02d} | "
        f"{layer.name:28s} | "
        f"{layer.__class__.__name__:24s} | "
        f"output={layer.output.shape}"
        f"{activation}"
    )


dense1 = base_model.get_layer(
    "dense_1"
)


print(
    "\nSelected frozen feature layer:"
)

print(
    "    dense_1"
)

print(
    "Feature dimension:",
    dense1.output.shape
)


if dense1.output.shape[-1] != 64:

    raise RuntimeError(
        f"Expected dense_1 dimension 64; found {dense1.output.shape}."
    )


print(
    "\n✓ dense_1 frozen as the LUDB representation-transfer feature layer."
)


# ======================================================================================
# 12. FREEZE LUDB SIGNAL PREPROCESSING
#
# Native:
#     500 Hz × 5000 samples
#
# External model input:
#     100 Hz × 1000 samples
#
# Method:
#     scipy.signal.resample_poly
#     up=1, down=5
#
# This provides anti-aliasing before downsampling.
#
# No z-score or amplitude normalization is added.
# ======================================================================================

def preprocess_ludb_signal(
    signal,
    fs
):

    signal = np.asarray(
        signal,
        dtype=np.float64
    )


    if signal.shape != (
        5000,
        12
    ):

        raise RuntimeError(
            f"Unexpected LUDB native shape: {signal.shape}"
        )


    if float(
        fs
    ) != 500.0:

        raise RuntimeError(
            f"Unexpected LUDB sampling frequency: {fs}"
        )


    downsampled = resample_poly(
        signal,
        up=1,
        down=5,
        axis=0
    )


    if downsampled.shape != (
        1000,
        12
    ):

        raise RuntimeError(
            f"Unexpected resampled shape: {downsampled.shape}"
        )


    if not np.all(
        np.isfinite(
            downsampled
        )
    ):

        raise RuntimeError(
            "Non-finite values after LUDB resampling."
        )


    # Model convention:
    # (lead, time, channel)

    model_input = (
        downsampled.T[
            :,
            :,
            None
        ]
    ).astype(
        np.float32
    )


    if model_input.shape != (
        12,
        1000,
        1
    ):

        raise RuntimeError(
            f"Unexpected model-input shape: {model_input.shape}"
        )


    return model_input


# Test preprocessing on ONE record only.
first_id = int(
    ludb.iloc[
        0
    ][
        "ID"
    ]
)


sample_signal, sample_fields = wfdb.rdsamp(
    str(
        DATA_DIR
        /
        str(
            first_id
        )
    )
)


sample_preprocessed = preprocess_ludb_signal(
    sample_signal,
    sample_fields[
        "fs"
    ]
)


print("\n" + "=" * 100)
print("FROZEN LUDB PREPROCESSING")
print("=" * 100)


print(
    "Native shape:",
    sample_signal.shape
)

print(
    "Native fs:",
    sample_fields[
        "fs"
    ]
)

print(
    "Prepared model shape:",
    sample_preprocessed.shape
)

print(
    "Prepared minimum:",
    float(
        sample_preprocessed.min()
    )
)

print(
    "Prepared maximum:",
    float(
        sample_preprocessed.max()
    )
)


print(
    "\nFrozen preprocessing:"
)

print(
    "    500 Hz -> 100 Hz using scipy.signal.resample_poly(up=1, down=5)"
)

print(
    "    no additional amplitude normalisation"
)

print(
    "    lead order unchanged"
)


print(
    "\nNO MODEL INFERENCE WAS PERFORMED."
)


# ======================================================================================
# 13. SAVE CLEAN LUDB TASK TABLE
# ======================================================================================

save_columns = [
    "ID",

    "target_Conduction",
    "target_LVH",
    "target_Ischaemia_related",
    "target_AFib",
    "target_Paced",
]


ludb[
    save_columns
].to_csv(
    OUT_DIR
    /
    "ludb_frozen_targets.csv",
    index=False
)


task_df.to_csv(
    OUT_DIR
    /
    "ludb_task_counts.csv",
    index=False
)


# ======================================================================================
# 14. SAVE MANIFEST
# ======================================================================================

manifest = {

    "stage":
        "Step 5B frozen LUDB downstream-task definition",

    "analysis_type":
        "external frozen-representation linear-probe evaluation",

    "not_direct_23class_validation":
        True,

    "records":
        200,

    "primary_tasks":
        PRIMARY_TASKS,

    "exploratory_tasks":
        EXPLORATORY_TASKS,

    "task_counts":
        task_df.to_dict(
            orient="records"
        ),

    "feature_layer":
        "dense_1",

    "feature_dimension":
        64,

    "native_sampling_frequency_Hz":
        500,

    "native_samples":
        5000,

    "model_sampling_frequency_Hz":
        100,

    "model_samples":
        1000,

    "resampling":
        "scipy.signal.resample_poly(up=1, down=5, axis=time)",

    "amplitude_normalisation":
        "none",

    "model_inference_performed":
        False,

    "probe_training_performed":
        False,

    "gaze_model_training_performed":
        False,
}


with open(
    OUT_DIR
    /
    "ludb_step5b_manifest.json",
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ======================================================================================
# 15. FINISH
# ======================================================================================

print("\n\n" + "=" * 100)
print("STEP 5B COMPLETE — TASKS/PREPROCESSING FROZEN")
print("=" * 100)


print(
    """
Please paste back only:

1. FROZEN LUDB TASK DEFINITIONS

2. HISTORICAL COUNT REPRODUCTION

3. FROZEN REPRESENTATION LAYER AUDIT

4. FROZEN LUDB PREPROCESSING

If those pass, Step 5C will be the actual external representation-transfer
experiment:

    Base vs Consultant vs All-clinician

using identical 5-fold out-of-fold linear probes on these frozen LUDB tasks.
"""
)

In [ ]:
# ======================================================================================
# STEP 5B.1 — RESOLVE LUDB ISCHAEMIA ENDPOINT
#
# READ ONLY.
# NO MODEL INFERENCE.
# NO TRAINING.
#
# Goal:
#   Identify the 12 records currently classified as indeterminate and
#   inspect the exact historical code used to produce N=192, positive=43.
# ======================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ======================================================================================
# 1. SHOW ALL CURRENTLY INDETERMINATE RECORDS
# ======================================================================================

indeterminate = ludb[
    ludb["target_Ischaemia_related"].isna()
][
    [
        "ID",
        "Ischemia",
    ]
].copy()


print("=" * 100)
print("CURRENTLY INDETERMINATE ISCHAEMIA RECORDS")
print("=" * 100)

print(
    "N =",
    len(indeterminate)
)

print(
    indeterminate.to_string(
        index=False
    )
)


# ======================================================================================
# 2. BREAK EACH ISCHAEMIA FIELD INTO LINES
# ======================================================================================

print("\n" + "=" * 100)
print("INDETERMINATE RECORDS — LINE-BY-LINE")
print("=" * 100)


for _, row in indeterminate.iterrows():

    print(
        f"\nID {row['ID']}"
    )

    print(
        "-" * 60
    )

    lines = split_lines(
        row["Ischemia"]
    )

    for line in lines:

        print(
            repr(
                line
            )
        )


# ======================================================================================
# 3. READ HISTORICAL NOTEBOOK AND SEARCH FOR ISCHAEMIA TASK DEFINITION
# ======================================================================================

HISTORICAL_NOTEBOOK = Path(
    "Re-analysis.ipynb"
)


if not HISTORICAL_NOTEBOOK.exists():

    raise FileNotFoundError(
        HISTORICAL_NOTEBOOK
    )


nb = json.loads(
    HISTORICAL_NOTEBOOK.read_text(
        encoding="utf-8"
    )
)


search_terms = [
    "Ischaemia_related",
    "ischemia_related",
    "ischaemia",
    "ischemia",
    "scar",
    "stemi",
    "supp.nstemi",
]


print("\n" + "=" * 100)
print("HISTORICAL ISCHAEMIA-RELATED CODE CELLS")
print("=" * 100)


matched_cells = 0


for cell_index, cell in enumerate(
    nb.get(
        "cells",
        []
    )
):

    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    source_lower = source.lower()


    if any(
        term.lower() in source_lower
        for term in search_terms
    ):

        matched_cells += 1

        print(
            "\n" +
            "#" * 100
        )

        print(
            f"CODE CELL {cell_index}"
        )

        print(
            "#" * 100
        )

        print(
            source
        )


print(
    "\nMatched code cells:",
    matched_cells
)


# ======================================================================================
# 4. ALSO SEARCH "Test on LUDB all classes.ipynb"
# ======================================================================================

SECOND_NOTEBOOK = Path(
    "Test on LUDB all classes.ipynb"
)


if SECOND_NOTEBOOK.exists():

    nb2 = json.loads(
        SECOND_NOTEBOOK.read_text(
            encoding="utf-8"
        )
    )


    print("\n" + "=" * 100)
    print("SECOND HISTORICAL NOTEBOOK — ISCHAEMIA-RELATED CODE")
    print("=" * 100)


    matched_cells_2 = 0


    for cell_index, cell in enumerate(
        nb2.get(
            "cells",
            []
        )
    ):

        if cell.get(
            "cell_type"
        ) != "code":

            continue


        source = "".join(
            cell.get(
                "source",
                []
            )
        )


        source_lower = source.lower()


        if any(
            term.lower() in source_lower
            for term in search_terms
        ):

            matched_cells_2 += 1

            print(
                "\n" +
                "#" * 100
            )

            print(
                f"CODE CELL {cell_index}"
            )

            print(
                "#" * 100
            )

            print(
                source
            )


    print(
        "\nMatched code cells:",
        matched_cells_2
    )


# ======================================================================================
# 5. SIMPLE CONTENT AUDIT OF THE 12 RECORDS
# ======================================================================================

content_rows = []


for _, row in indeterminate.iterrows():

    text = clean_text(
        row["Ischemia"]
    )


    content_rows.append(
        {
            "ID":
                int(
                    row["ID"]
                ),

            "contains_undefined":
                "undefined ischemia/scar/supp.nstemi"
                in text,

            "contains_stemi_word":
                "stemi"
                in text,

            "contains_supp_nstemi":
                "supp.nstemi"
                in text,

            "n_lines":
                len(
                    split_lines(
                        row["Ischemia"]
                    )
                ),

            "raw_Ischemia":
                row["Ischemia"],
        }
    )


content_df = pd.DataFrame(
    content_rows
)


print("\n" + "=" * 100)
print("INDETERMINATE CONTENT SUMMARY")
print("=" * 100)

print(
    content_df.to_string(
        index=False
    )
)


print(
    """
NO MODEL INFERENCE OR TRAINING PERFORMED.

Please paste back:

1. CURRENTLY INDETERMINATE ISCHAEMIA RECORDS
2. HISTORICAL ISCHAEMIA-RELATED CODE CELLS
3. INDETERMINATE CONTENT SUMMARY
"""
)

In [ ]:
# ======================================================================================
# STEP 5C — LUDB EXTERNAL FROZEN-REPRESENTATION TRANSFER
#
# NO GAZE-MODEL TRAINING.
# NO ENDPOINT CHANGES.
# NO MODEL SELECTION.
#
# Models:
#   BASE
#   CONSULTANT
#   ALL_CLINICIANS
#
# Representation:
#   dense_1 (64 dimensions)
#
# External dataset:
#   LUDB
#
# Evaluation:
#   5-fold stratified out-of-fold linear probes.
#
# PRIMARY TASKS:
#   Conduction
#   LVH
#   Ischaemia_related
#
# EXPLORATORY:
#   AFib
#   Paced
#
# Preprocessing frozen in Step 5B:
#   500 -> 100 Hz using scipy.signal.resample_poly
#   no additional amplitude normalisation
#
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import gc
import json
import hashlib
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import wfdb

from scipy.signal import resample_poly

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
)


# ======================================================================================
# 1. FIXED SETTINGS
# ======================================================================================

SEED = 2026
N_FOLDS = 5

FEATURE_LAYER = "dense_1"
FEATURE_DIM = 64

PRED_BATCH_SIZE = 64


EXPECTED_LEADS = [
    "I",
    "II",
    "III",
    "AVR",
    "AVL",
    "AVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]


PRIMARY_TASKS = [
    "Conduction",
    "LVH",
    "Ischaemia_related",
]


EXPLORATORY_TASKS = [
    "AFib",
    "Paced",
]


TASK_COLUMNS = {
    "Conduction":
        "target_Conduction",

    "LVH":
        "target_LVH",

    "Ischaemia_related":
        "target_Ischaemia_related",

    "AFib":
        "target_AFib",

    "Paced":
        "target_Paced",
}


EXPECTED_COUNTS = {

    "Conduction": {
        "N": 200,
        "positive": 66,
    },

    "LVH": {
        "N": 200,
        "positive": 108,
    },

    "Ischaemia_related": {
        "N": 188,
        "positive": 39,
    },

    "AFib": {
        "N": 200,
        "positive": 15,
    },

    "Paced": {
        "N": 200,
        "positive": 10,
    },
}


# ======================================================================================
# 2. PATHS
# ======================================================================================

ROOT = Path(
    "definitive_clean_gaze"
)


LUDB_ROOT = Path(
    "Test ECG"
)


LUDB_DATA = (
    LUDB_ROOT
    /
    "data"
)


FINAL_DIR = (
    ROOT
    /
    "final_models"
)


LUDB_DIR = (
    ROOT
    /
    "ludb_evaluation"
)


TARGET_PATH = (
    LUDB_DIR
    /
    "ludb_frozen_targets.csv"
)


STEP5B_MANIFEST_PATH = (
    LUDB_DIR
    /
    "ludb_step5b_manifest.json"
)


STEP1_MANIFEST_PATH = (
    ROOT
    /
    "step1_manifest.json"
)


FINAL_MODEL_MANIFEST_PATH = (
    FINAL_DIR
    /
    "final_model_manifest.json"
)


CONSULTANT_WEIGHTS = (
    FINAL_DIR
    /
    "clean_consultant.weights.h5"
)


ALL_WEIGHTS = (
    FINAL_DIR
    /
    "clean_all_clinicians.weights.h5"
)


for path in [
    TARGET_PATH,
    STEP5B_MANIFEST_PATH,
    STEP1_MANIFEST_PATH,
    FINAL_MODEL_MANIFEST_PATH,
    CONSULTANT_WEIGHTS,
    ALL_WEIGHTS,
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


# ======================================================================================
# 3. HASH HELPER
# ======================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# ======================================================================================
# 4. VERIFY FROZEN MODEL FILES
# ======================================================================================

with open(
    STEP1_MANIFEST_PATH,
    "r"
) as f:

    step1_manifest = json.load(
        f
    )


with open(
    FINAL_MODEL_MANIFEST_PATH,
    "r"
) as f:

    final_model_manifest = json.load(
        f
    )


with open(
    STEP5B_MANIFEST_PATH,
    "r"
) as f:

    step5b_manifest = json.load(
        f
    )


BASE_MODEL_PATH = Path(
    step1_manifest[
        "base_model"
    ]
)


BASE_HASH = sha256_file(
    BASE_MODEL_PATH
)


CONSULTANT_HASH = sha256_file(
    CONSULTANT_WEIGHTS
)


ALL_HASH = sha256_file(
    ALL_WEIGHTS
)


print("=" * 100)
print("FROZEN MODEL VERIFICATION")
print("=" * 100)


print(
    "Base:",
    BASE_HASH
)


print(
    "Consultant:",
    CONSULTANT_HASH
)


print(
    "All clinicians:",
    ALL_HASH
)


if (
    BASE_HASH
    !=
    step1_manifest[
        "base_model_sha256"
    ]
):

    raise RuntimeError(
        "Base model hash mismatch."
    )


if (
    CONSULTANT_HASH
    !=
    final_model_manifest[
        "consultant_weights_sha256"
    ]
):

    raise RuntimeError(
        "Consultant weight hash mismatch."
    )


if (
    ALL_HASH
    !=
    final_model_manifest[
        "all_clinician_weights_sha256"
    ]
):

    raise RuntimeError(
        "All-clinician weight hash mismatch."
    )


print(
    "\n✓ Frozen model hashes verified."
)


# ======================================================================================
# 5. VERIFY STEP-5B FROZEN REPRESENTATION/PREPROCESSING
# ======================================================================================

if step5b_manifest[
    "feature_layer"
] != FEATURE_LAYER:

    raise RuntimeError(
        "Feature-layer definition changed."
    )


if step5b_manifest[
    "feature_dimension"
] != FEATURE_DIM:

    raise RuntimeError(
        "Feature dimension changed."
    )


if step5b_manifest[
    "native_sampling_frequency_Hz"
] != 500:

    raise RuntimeError(
        "Unexpected LUDB native sampling frequency."
    )


if step5b_manifest[
    "model_sampling_frequency_Hz"
] != 100:

    raise RuntimeError(
        "Unexpected model sampling frequency."
    )


print(
    "✓ Step-5B representation and preprocessing definition verified."
)


# ======================================================================================
# 6. LOAD FROZEN TARGETS
#
# IMPORTANT:
# We do NOT reconstruct clinical labels here.
# We use the target file frozen before model inference.
# ======================================================================================

targets = pd.read_csv(
    TARGET_PATH
)


if len(
    targets
) != 200:

    raise RuntimeError(
        f"Expected 200 LUDB target rows; found {len(targets)}."
    )


if targets[
    "ID"
].nunique() != 200:

    raise RuntimeError(
        "LUDB IDs are not unique."
    )


print("\n" + "=" * 100)
print("FROZEN LUDB TARGET AUDIT")
print("=" * 100)


target_audit_rows = []


for task_name, column in TASK_COLUMNS.items():

    valid = targets[
        column
    ].notna()


    y = targets.loc[
        valid,
        column
    ].astype(
        int
    )


    actual_n = int(
        len(
            y
        )
    )


    actual_positive = int(
        y.sum()
    )


    expected_n = EXPECTED_COUNTS[
        task_name
    ][
        "N"
    ]


    expected_positive = EXPECTED_COUNTS[
        task_name
    ][
        "positive"
    ]


    if (
        actual_n != expected_n
        or
        actual_positive != expected_positive
    ):

        raise RuntimeError(
            f"{task_name}: frozen-target count mismatch."
        )


    target_audit_rows.append(
        {
            "task":
                task_name,

            "role":
                (
                    "primary"
                    if task_name in PRIMARY_TASKS
                    else "exploratory"
                ),

            "N":
                actual_n,

            "positive":
                actual_positive,

            "negative":
                actual_n
                -
                actual_positive,

            "excluded":
                200
                -
                actual_n,
        }
    )


target_audit_df = pd.DataFrame(
    target_audit_rows
)


print(
    target_audit_df.to_string(
        index=False
    )
)


print(
    "\n✓ Frozen LUDB endpoints verified."
)


# ======================================================================================
# 7. FROZEN LUDB PREPROCESSING
# ======================================================================================

def preprocess_ludb_record(
    record_id
):

    path = (
        LUDB_DATA
        /
        str(
            int(
                record_id
            )
        )
    )


    signal, fields = wfdb.rdsamp(
        str(
            path
        )
    )


    if signal.shape != (
        5000,
        12
    ):

        raise RuntimeError(
            f"Record {record_id}: native shape {signal.shape}"
        )


    if float(
        fields[
            "fs"
        ]
    ) != 500.0:

        raise RuntimeError(
            f"Record {record_id}: unexpected fs."
        )


    leads = [
        str(
            lead
        ).upper()
        for lead
        in fields[
            "sig_name"
        ]
    ]


    if leads != EXPECTED_LEADS:

        raise RuntimeError(
            f"Record {record_id}: lead-order mismatch."
        )


    if not np.all(
        np.isfinite(
            signal
        )
    ):

        raise RuntimeError(
            f"Record {record_id}: non-finite raw signal."
        )


    downsampled = resample_poly(
        signal,
        up=1,
        down=5,
        axis=0
    )


    if downsampled.shape != (
        1000,
        12
    ):

        raise RuntimeError(
            f"Record {record_id}: bad resampled shape."
        )


    if not np.all(
        np.isfinite(
            downsampled
        )
    ):

        raise RuntimeError(
            f"Record {record_id}: non-finite resampled signal."
        )


    # NO amplitude normalisation.
    #
    # Frozen model convention:
    #     (lead, time, channel)

    X = (
        downsampled.T[
            :,
            :,
            None
        ]
    ).astype(
        np.float32
    )


    return X


# ======================================================================================
# 8. PREPROCESS ALL 200 LUDB ECGs
# ======================================================================================

print("\n" + "=" * 100)
print("PREPROCESSING ALL LUDB ECGs")
print("=" * 100)


record_ids = targets[
    "ID"
].astype(
    int
).to_numpy()


X_ludb = np.empty(
    (
        200,
        12,
        1000,
        1
    ),
    dtype=np.float32
)


for i, record_id in enumerate(
    record_ids
):

    X_ludb[
        i
    ] = preprocess_ludb_record(
        record_id
    )


    if (
        (i + 1) % 50 == 0
        or
        (i + 1) == 200
    ):

        print(
            f"Prepared {i + 1}/200"
        )


if not np.all(
    np.isfinite(
        X_ludb
    )
):

    raise RuntimeError(
        "Non-finite prepared LUDB array."
    )


print(
    "\nX_ludb:",
    X_ludb.shape
)


print(
    "Minimum:",
    float(
        X_ludb.min()
    )
)


print(
    "Maximum:",
    float(
        X_ludb.max()
    )
)


# Save the exact frozen preprocessed external dataset.
X_SAVE_PATH = (
    LUDB_DIR
    /
    "ludb_preprocessed_100hz.npy"
)


np.save(
    X_SAVE_PATH,
    X_ludb
)


X_HASH = sha256_file(
    X_SAVE_PATH
)


print(
    "\nPreprocessed LUDB SHA256:"
)

print(
    X_HASH
)


# ======================================================================================
# 9. MODEL LOADER
# ======================================================================================

def load_variant(
    weight_path=None
):

    model = tf.keras.models.load_model(
        BASE_MODEL_PATH,
        compile=False
    )


    if weight_path is not None:

        model.load_weights(
            weight_path
        )


    if model.output_shape != (
        None,
        23
    ):

        raise RuntimeError(
            f"Unexpected model output: {model.output_shape}"
        )


    feature_layer = model.get_layer(
        FEATURE_LAYER
    )


    if feature_layer.output.shape[-1] != FEATURE_DIM:

        raise RuntimeError(
            "Unexpected dense_1 feature dimension."
        )


    feature_model = tf.keras.Model(
        inputs=model.input,
        outputs=feature_layer.output
    )


    return (
        model,
        feature_model
    )


# ======================================================================================
# 10. LOAD THREE FROZEN MODELS
# ======================================================================================

base_model, base_feature_model = load_variant()


consultant_model, consultant_feature_model = load_variant(
    CONSULTANT_WEIGHTS
)


all_model, all_feature_model = load_variant(
    ALL_WEIGHTS
)


# ======================================================================================
# 11. EXTRACT REPRESENTATIONS
# ======================================================================================

print("\n" + "=" * 100)
print("EXTRACTING FROZEN LUDB REPRESENTATIONS")
print("=" * 100)


FEATURE_MODELS = {

    "BASE":
        base_feature_model,

    "CONSULTANT":
        consultant_feature_model,

    "ALL_CLINICIANS":
        all_feature_model,
}


features = {}


for model_name, feature_model in FEATURE_MODELS.items():

    print(
        "\n",
        model_name
    )


    Z = feature_model.predict(
        X_ludb,
        batch_size=PRED_BATCH_SIZE,
        verbose=1
    )


    Z = np.asarray(
        Z,
        dtype=np.float32
    )


    if Z.shape != (
        200,
        64
    ):

        raise RuntimeError(
            f"{model_name}: bad feature shape {Z.shape}"
        )


    if not np.all(
        np.isfinite(
            Z
        )
    ):

        raise RuntimeError(
            f"{model_name}: non-finite features."
        )


    features[
        model_name
    ] = Z


    np.save(
        LUDB_DIR
        /
        f"ludb_features_{model_name.lower()}.npy",

        Z
    )


    print(
        "shape:",
        Z.shape
    )


# ======================================================================================
# 12. REPRESENTATION CHANGE AUDIT
# ======================================================================================

feature_change_rows = []


for model_name in [
    "CONSULTANT",
    "ALL_CLINICIANS",
]:

    absolute_change = np.abs(
        features[
            model_name
        ]
        -
        features[
            "BASE"
        ]
    )


    feature_change_rows.append(
        {
            "model":
                model_name,

            "mean_absolute_feature_change":
                float(
                    absolute_change.mean()
                ),

            "median_absolute_feature_change":
                float(
                    np.median(
                        absolute_change
                    )
                ),

            "max_absolute_feature_change":
                float(
                    absolute_change.max()
                ),
        }
    )


feature_change_df = pd.DataFrame(
    feature_change_rows
)


print("\n" + "=" * 100)
print("LUDB FEATURE-CHANGE AUDIT")
print("=" * 100)


print(
    feature_change_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 13. FREEZE PROBE CONFIGURATION
#
# StandardScaler is fitted ONLY on the training fold.
#
# This removes trivial feature-scale differences between model variants while
# preserving differences in linear separability.
# ======================================================================================

def make_probe():

    return Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
                "classifier",
                LogisticRegression(
                    C=1.0,
                    solver="lbfgs",
                    class_weight="balanced",
                    max_iter=10000,
                    random_state=SEED,
                )
            ),
        ]
    )


# ======================================================================================
# 14. METRIC HELPER
# ======================================================================================

def binary_metrics(
    y_true,
    y_prob
):

    y_pred = (
        y_prob
        >=
        0.5
    ).astype(
        int
    )


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1
        ]
    ).ravel()


    sensitivity = (
        tp
        /
        (
            tp + fn
        )
        if (
            tp + fn
        )
        else np.nan
    )


    specificity = (
        tn
        /
        (
            tn + fp
        )
        if (
            tn + fp
        )
        else np.nan
    )


    return {

        "AUROC":
            float(
                roc_auc_score(
                    y_true,
                    y_prob
                )
            ),

        "AP":
            float(
                average_precision_score(
                    y_true,
                    y_prob
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    zero_division=0
                )
            ),

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "TP":
            int(
                tp
            ),

        "TN":
            int(
                tn
            ),

        "FP":
            int(
                fp
            ),

        "FN":
            int(
                fn
            ),
    }


# ======================================================================================
# 15. OOF EVALUATION
# ======================================================================================

summary_rows = []

prediction_rows = []

fold_rows = []


for task_name, target_column in TASK_COLUMNS.items():

    print(
        "\n\n"
        +
        "#" * 100
    )

    print(
        f"TASK — {task_name}"
    )

    print(
        "#" * 100
    )


    valid_mask = targets[
        target_column
    ].notna().to_numpy()


    valid_indices = np.where(
        valid_mask
    )[0]


    y = targets.loc[
        valid_mask,
        target_column
    ].astype(
        int
    ).to_numpy()


    ids = record_ids[
        valid_indices
    ]


    print(
        "Role:",
        (
            "PRIMARY"
            if task_name in PRIMARY_TASKS
            else "EXPLORATORY"
        )
    )


    print(
        "N:",
        len(
            y
        )
    )


    print(
        "Positive:",
        int(
            y.sum()
        )
    )


    print(
        "Negative:",
        int(
            len(
                y
            )
            -
            y.sum()
        )
    )


    splitter = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED
    )


    splits = list(
        splitter.split(
            np.zeros(
                len(
                    y
                )
            ),
            y
        )
    )


    # --------------------------------------------------------------
    # Save exact folds BEFORE model-specific probing.
    # --------------------------------------------------------------

    fold_assignment = np.zeros(
        len(
            y
        ),
        dtype=int
    )


    for fold, (
        train_idx,
        test_idx
    ) in enumerate(
        splits,
        start=1
    ):

        fold_assignment[
            test_idx
        ] = fold


        fold_rows.append(
            {
                "task":
                    task_name,

                "fold":
                    fold,

                "train_N":
                    int(
                        len(
                            train_idx
                        )
                    ),

                "test_N":
                    int(
                        len(
                            test_idx
                        )
                    ),

                "train_positive":
                    int(
                        y[
                            train_idx
                        ].sum()
                    ),

                "test_positive":
                    int(
                        y[
                            test_idx
                        ].sum()
                    ),
            }
        )


        print(
            f"Fold {fold}: "
            f"train={len(train_idx)} "
            f"(pos={int(y[train_idx].sum())}), "
            f"test={len(test_idx)} "
            f"(pos={int(y[test_idx].sum())})"
        )


    # --------------------------------------------------------------
    # OOF prediction for each representation.
    # --------------------------------------------------------------

    for model_name in [
        "BASE",
        "CONSULTANT",
        "ALL_CLINICIANS",
    ]:

        Z = features[
            model_name
        ][
            valid_indices
        ]


        probabilities = np.full(
            len(
                y
            ),
            np.nan,
            dtype=np.float64
        )


        for fold, (
            train_idx,
            test_idx
        ) in enumerate(
            splits,
            start=1
        ):

            probe = make_probe()


            with warnings.catch_warnings():

                warnings.simplefilter(
                    "error"
                )


                probe.fit(
                    Z[
                        train_idx
                    ],
                    y[
                        train_idx
                    ]
                )


            probabilities[
                test_idx
            ] = probe.predict_proba(
                Z[
                    test_idx
                ]
            )[
                :,
                1
            ]


        if not np.all(
            np.isfinite(
                probabilities
            )
        ):

            raise RuntimeError(
                f"{task_name}/{model_name}: incomplete OOF predictions."
            )


        metrics = binary_metrics(
            y,
            probabilities
        )


        summary_rows.append(
            {
                "task":
                    task_name,

                "role":
                    (
                        "primary"
                        if task_name in PRIMARY_TASKS
                        else "exploratory"
                    ),

                "model":
                    model_name,

                "N":
                    int(
                        len(
                            y
                        )
                    ),

                "positive":
                    int(
                        y.sum()
                    ),

                "negative":
                    int(
                        len(
                            y
                        )
                        -
                        y.sum()
                    ),

                **metrics,
            }
        )


        predicted_labels = (
            probabilities
            >=
            0.5
        ).astype(
            int
        )


        for i in range(
            len(
                y
            )
        ):

            prediction_rows.append(
                {
                    "task":
                        task_name,

                    "role":
                        (
                            "primary"
                            if task_name in PRIMARY_TASKS
                            else "exploratory"
                        ),

                    "model":
                        model_name,

                    "record_id":
                        int(
                            ids[
                                i
                            ]
                        ),

                    "fold":
                        int(
                            fold_assignment[
                                i
                            ]
                        ),

                    "true_label":
                        int(
                            y[
                                i
                            ]
                        ),

                    "probability":
                        float(
                            probabilities[
                                i
                            ]
                        ),

                    "predicted_label":
                        int(
                            predicted_labels[
                                i
                            ]
                        ),
                }
            )


# ======================================================================================
# 16. TABLES
# ======================================================================================

summary_df = pd.DataFrame(
    summary_rows
)


predictions_df = pd.DataFrame(
    prediction_rows
)


fold_df = pd.DataFrame(
    fold_rows
)


# ======================================================================================
# 17. PAIRED POINT DIFFERENCES VS BASE
# ======================================================================================

comparison_rows = []


for task_name in TASK_COLUMNS:

    task_summary = summary_df[
        summary_df[
            "task"
        ]
        ==
        task_name
    ].set_index(
        "model"
    )


    base = task_summary.loc[
        "BASE"
    ]


    for model_name in [
        "CONSULTANT",
        "ALL_CLINICIANS",
    ]:

        row = task_summary.loc[
            model_name
        ]


        comparison_rows.append(
            {
                "task":
                    task_name,

                "role":
                    row[
                        "role"
                    ],

                "comparison":
                    (
                        model_name
                        +
                        "_vs_BASE"
                    ),

                "delta_AUROC":
                    float(
                        row[
                            "AUROC"
                        ]
                        -
                        base[
                            "AUROC"
                        ]
                    ),

                "delta_AP":
                    float(
                        row[
                            "AP"
                        ]
                        -
                        base[
                            "AP"
                        ]
                    ),

                "delta_balanced_accuracy":
                    float(
                        row[
                            "balanced_accuracy"
                        ]
                        -
                        base[
                            "balanced_accuracy"
                        ]
                    ),
            }
        )


comparison_df = pd.DataFrame(
    comparison_rows
)


# ======================================================================================
# 18. SAVE
# ======================================================================================

summary_df.to_csv(
    LUDB_DIR
    /
    "ludb_step5c_oof_summary.csv",
    index=False
)


predictions_df.to_csv(
    LUDB_DIR
    /
    "ludb_step5c_oof_predictions.csv",
    index=False
)


comparison_df.to_csv(
    LUDB_DIR
    /
    "ludb_step5c_point_differences.csv",
    index=False
)


fold_df.to_csv(
    LUDB_DIR
    /
    "ludb_step5c_fold_manifest.csv",
    index=False
)


feature_change_df.to_csv(
    LUDB_DIR
    /
    "ludb_step5c_feature_change_audit.csv",
    index=False
)


# ======================================================================================
# 19. PRINT PRIMARY RESULTS
# ======================================================================================

print(
    "\n\n"
    +
    "=" * 110
)

print(
    "LUDB PRIMARY EXTERNAL REPRESENTATION-TRANSFER RESULTS"
)

print(
    "=" * 110
)


primary_summary = summary_df[
    summary_df[
        "role"
    ]
    ==
    "primary"
]


print(
    primary_summary[
        [
            "task",
            "model",
            "N",
            "positive",
            "AUROC",
            "AP",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


print(
    "\nPRIMARY POINT DIFFERENCES VS BASE"
)


print(
    comparison_df[
        comparison_df[
            "role"
        ]
        ==
        "primary"
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 20. PRINT EXPLORATORY RESULTS
# ======================================================================================

print(
    "\n\n"
    +
    "=" * 110
)

print(
    "LUDB EXPLORATORY REPRESENTATION-TRANSFER RESULTS"
)

print(
    "=" * 110
)


exploratory_summary = summary_df[
    summary_df[
        "role"
    ]
    ==
    "exploratory"
]


print(
    exploratory_summary[
        [
            "task",
            "model",
            "N",
            "positive",
            "AUROC",
            "AP",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


print(
    "\nEXPLORATORY POINT DIFFERENCES VS BASE"
)


print(
    comparison_df[
        comparison_df[
            "role"
        ]
        ==
        "exploratory"
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 21. EVALUATION MANIFEST
# ======================================================================================

manifest = {

    "stage":
        "Step 5C LUDB frozen external representation-transfer evaluation",

    "interpretation":
        (
            "External linear-probe evaluation of frozen ECG representations; "
            "NOT direct validation of the PTB-XL 23-class diagnostic head."
        ),

    "LUDB_records":
        200,

    "preprocessed_LUDB_sha256":
        X_HASH,

    "native_sampling_frequency_Hz":
        500,

    "model_sampling_frequency_Hz":
        100,

    "resampling":
        "scipy.signal.resample_poly(up=1, down=5, axis=time)",

    "additional_amplitude_normalisation":
        False,

    "feature_layer":
        FEATURE_LAYER,

    "feature_dimension":
        FEATURE_DIM,

    "primary_tasks":
        PRIMARY_TASKS,

    "exploratory_tasks":
        EXPLORATORY_TASKS,

    "ischaemia_endpoint":
        (
            "Positive: definite 'Ischemia:', 'Scar formation:' or 'STEMI:' "
            "annotation. Negative: empty Ischemia field. "
            "Undefined ischemia/scar/supp.NSTEMI-only records excluded."
        ),

    "ischaemia_N":
        188,

    "ischaemia_positive":
        39,

    "cross_validation":
        "5-fold StratifiedKFold out-of-fold",

    "CV_seed":
        SEED,

    "probe":
        (
            "Training-fold StandardScaler followed by logistic regression "
            "(C=1.0, lbfgs, class_weight=balanced)."
        ),

    "threshold_for_secondary_classification_metrics":
        0.5,

    "primary_transfer_metrics":
        [
            "AUROC",
            "Average Precision",
        ],

    "gaze_model_training_performed":
        False,

    "endpoint_tuning_after_model_inference":
        False,

    "base_model_sha256":
        BASE_HASH,

    "consultant_weights_sha256":
        CONSULTANT_HASH,

    "all_clinician_weights_sha256":
        ALL_HASH,
}


with open(
    LUDB_DIR
    /
    "ludb_step5c_manifest.json",
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ======================================================================================
# 22. FINISH
# ======================================================================================

print(
    "\n\n"
    +
    "=" * 100
)

print(
    "STEP 5C COMPLETE"
)

print(
    "=" * 100
)


print(
    """
Please paste back:

1. LUDB FEATURE-CHANGE AUDIT

2. LUDB PRIMARY EXTERNAL REPRESENTATION-TRANSFER RESULTS

3. PRIMARY POINT DIFFERENCES VS BASE

4. LUDB EXPLORATORY REPRESENTATION-TRANSFER RESULTS

5. EXPLORATORY POINT DIFFERENCES VS BASE

Do NOT change the models, tasks, folds, feature layer or preprocessing
after seeing these results.

Afterwards Step 5D will compute paired bootstrap confidence intervals
using ONLY the saved OOF predictions.
"""
)

In [ ]:
# ======================================================================================
# STEP 5D — LUDB PAIRED BOOTSTRAP OF SAVED OOF PREDICTIONS
#
# NO MODEL INFERENCE.
# NO MODEL TRAINING.
# NO LINEAR-PROBE TRAINING.
#
# Uses ONLY:
#   definitive_clean_gaze/ludb_evaluation/ludb_step5c_oof_predictions.csv
#
# Purpose:
#   Quantify sampling uncertainty around the paired OOF performance
#   differences observed in Step 5C.
#
# Primary metrics:
#   AUROC
#   Average Precision (AP)
#
# Secondary:
#   balanced accuracy
#
# Comparisons:
#   CONSULTANT vs BASE
#   ALL_CLINICIANS vs BASE
#   ALL_CLINICIANS vs CONSULTANT
#
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
)


# ======================================================================================
# 1. SETTINGS
# ======================================================================================

SEED = 2026
N_BOOT = 5000

CI_PERCENTILES = [
    2.5,
    97.5,
]


PRIMARY_TASKS = [
    "Conduction",
    "LVH",
    "Ischaemia_related",
]


EXPLORATORY_TASKS = [
    "AFib",
    "Paced",
]


TASKS = (
    PRIMARY_TASKS
    +
    EXPLORATORY_TASKS
)


MODELS = [
    "BASE",
    "CONSULTANT",
    "ALL_CLINICIANS",
]


COMPARISONS = [

    (
        "CONSULTANT_vs_BASE",
        "CONSULTANT",
        "BASE",
    ),

    (
        "ALL_CLINICIANS_vs_BASE",
        "ALL_CLINICIANS",
        "BASE",
    ),

    (
        "ALL_CLINICIANS_vs_CONSULTANT",
        "ALL_CLINICIANS",
        "CONSULTANT",
    ),
]


# ======================================================================================
# 2. PATHS
# ======================================================================================

LUDB_DIR = Path(
    "definitive_clean_gaze/ludb_evaluation"
)


PRED_PATH = (
    LUDB_DIR
    /
    "ludb_step5c_oof_predictions.csv"
)


if not PRED_PATH.exists():

    raise FileNotFoundError(
        PRED_PATH
    )


# ======================================================================================
# 3. LOAD SAVED OOF PREDICTIONS
# ======================================================================================

pred = pd.read_csv(
    PRED_PATH
)


print("=" * 110)
print("SAVED LUDB OOF PREDICTION AUDIT")
print("=" * 110)


print(
    "Rows:",
    len(
        pred
    )
)


print(
    "Tasks:",
    sorted(
        pred[
            "task"
        ].unique()
    )
)


print(
    "Models:",
    sorted(
        pred[
            "model"
        ].unique()
    )
)


if set(
    pred[
        "task"
    ].unique()
) != set(
    TASKS
):

    raise RuntimeError(
        "Unexpected LUDB tasks in saved predictions."
    )


if set(
    pred[
        "model"
    ].unique()
) != set(
    MODELS
):

    raise RuntimeError(
        "Unexpected model names in saved predictions."
    )


# ======================================================================================
# 4. VERIFY EXACT PAIRED RECORD SETS
# ======================================================================================

for task in TASKS:

    sub = pred[
        pred[
            "task"
        ]
        ==
        task
    ]


    reference_ids = None


    for model in MODELS:

        m = (
            sub[
                sub[
                    "model"
                ]
                ==
                model
            ]
            .sort_values(
                "record_id"
            )
        )


        ids = m[
            "record_id"
        ].to_numpy()


        if reference_ids is None:

            reference_ids = ids

        else:

            if not np.array_equal(
                reference_ids,
                ids
            ):

                raise RuntimeError(
                    f"{task}: record sets differ between models."
                )


    print(
        f"{task:20s} paired records = {len(reference_ids)}"
    )


print(
    "\n✓ Every model was evaluated on identical records within each task."
)


# ======================================================================================
# 5. METRIC FUNCTION
# ======================================================================================

def calculate_metrics(
    y,
    prob
):

    y = np.asarray(
        y,
        dtype=int
    )


    prob = np.asarray(
        prob,
        dtype=float
    )


    if len(
        np.unique(
            y
        )
    ) < 2:

        return {
            "AUROC":
                np.nan,

            "AP":
                np.nan,

            "balanced_accuracy":
                np.nan,
        }


    predicted = (
        prob
        >=
        0.5
    ).astype(
        int
    )


    return {

        "AUROC":
            float(
                roc_auc_score(
                    y,
                    prob
                )
            ),

        "AP":
            float(
                average_precision_score(
                    y,
                    prob
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    y,
                    predicted
                )
            ),
    }


# ======================================================================================
# 6. CONSTRUCT ONE PAIRED TABLE PER TASK
# ======================================================================================

task_data = {}


for task in TASKS:

    sub = pred[
        pred[
            "task"
        ]
        ==
        task
    ]


    model_frames = {}


    for model in MODELS:

        model_frames[
            model
        ] = (
            sub[
                sub[
                    "model"
                ]
                ==
                model
            ]
            [
                [
                    "record_id",
                    "true_label",
                    "probability",
                ]
            ]
            .sort_values(
                "record_id"
            )
            .reset_index(
                drop=True
            )
        )


    reference = model_frames[
        "BASE"
    ]


    y = reference[
        "true_label"
    ].to_numpy(
        dtype=int
    )


    record_ids = reference[
        "record_id"
    ].to_numpy()


    probabilities = {}


    for model in MODELS:

        frame = model_frames[
            model
        ]


        if not np.array_equal(
            frame[
                "record_id"
            ].to_numpy(),
            record_ids
        ):

            raise RuntimeError(
                f"{task}/{model}: ID mismatch."
            )


        if not np.array_equal(
            frame[
                "true_label"
            ].to_numpy(
                dtype=int
            ),
            y
        ):

            raise RuntimeError(
                f"{task}/{model}: target mismatch."
            )


        probabilities[
            model
        ] = frame[
            "probability"
        ].to_numpy(
            dtype=float
        )


    task_data[
        task
    ] = {

        "record_ids":
            record_ids,

        "y":
            y,

        "probabilities":
            probabilities,
    }


# ======================================================================================
# 7. POINT ESTIMATES FROM SAVED OOF PREDICTIONS
# ======================================================================================

point_rows = []


for task in TASKS:

    data = task_data[
        task
    ]


    y = data[
        "y"
    ]


    role = (
        "primary"
        if task in PRIMARY_TASKS
        else "exploratory"
    )


    for model in MODELS:

        m = calculate_metrics(
            y,
            data[
                "probabilities"
            ][
                model
            ]
        )


        point_rows.append(
            {
                "task":
                    task,

                "role":
                    role,

                "model":
                    model,

                "N":
                    int(
                        len(
                            y
                        )
                    ),

                "positive":
                    int(
                        y.sum()
                    ),

                **m,
            }
        )


point_df = pd.DataFrame(
    point_rows
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "POINT ESTIMATES RECONSTRUCTED FROM SAVED OOF PREDICTIONS"
)

print(
    "=" * 110
)


print(
    point_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 8. BOOTSTRAP STORAGE
# ======================================================================================

METRICS = [
    "AUROC",
    "AP",
    "balanced_accuracy",
]


bootstrap_store = {}


for task in TASKS:

    bootstrap_store[
        task
    ] = {}


    for comparison_name, _, _ in COMPARISONS:

        bootstrap_store[
            task
        ][
            comparison_name
        ] = {

            metric:
                np.full(
                    N_BOOT,
                    np.nan,
                    dtype=np.float64
                )

            for metric in METRICS
        }


# ======================================================================================
# 9. PAIRED RECORD-LEVEL BOOTSTRAP
# ======================================================================================

rng = np.random.default_rng(
    SEED
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "LUDB PAIRED BOOTSTRAP"
)

print(
    "=" * 110
)


for task in TASKS:

    data = task_data[
        task
    ]


    y = data[
        "y"
    ]


    probabilities = data[
        "probabilities"
    ]


    n = len(
        y
    )


    print(
        f"\nTask: {task} | N={n} | positive={int(y.sum())}"
    )


    for b in range(
        N_BOOT
    ):

        # Same sampled records for all three models:
        # paired bootstrap.
        idx = rng.integers(
            low=0,
            high=n,
            size=n
        )


        yb = y[
            idx
        ]


        # Rare tasks can occasionally yield a one-class bootstrap.
        if len(
            np.unique(
                yb
            )
        ) < 2:

            continue


        boot_metrics = {}


        for model in MODELS:

            boot_metrics[
                model
            ] = calculate_metrics(
                yb,
                probabilities[
                    model
                ][
                    idx
                ]
            )


        for (
            comparison_name,
            model_a,
            model_b
        ) in COMPARISONS:

            for metric in METRICS:

                a = boot_metrics[
                    model_a
                ][
                    metric
                ]


                bb = boot_metrics[
                    model_b
                ][
                    metric
                ]


                if (
                    np.isfinite(
                        a
                    )
                    and
                    np.isfinite(
                        bb
                    )
                ):

                    bootstrap_store[
                        task
                    ][
                        comparison_name
                    ][
                        metric
                    ][
                        b
                    ] = (
                        a
                        -
                        bb
                    )


        if (
            (b + 1) % 1000 == 0
            or
            (b + 1) == N_BOOT
        ):

            print(
                f"  {b + 1}/{N_BOOT}"
            )


# ======================================================================================
# 10. SUMMARY HELPER
# ======================================================================================

def bootstrap_summary(
    values,
    point_delta
):

    values = np.asarray(
        values,
        dtype=float
    )


    valid = values[
        np.isfinite(
            values
        )
    ]


    if len(
        valid
    ) == 0:

        return {

            "point_delta":
                float(
                    point_delta
                ),

            "bootstrap_mean_delta":
                np.nan,

            "CI_low":
                np.nan,

            "CI_high":
                np.nan,

            "valid_bootstraps":
                0,

            "fraction_below_zero":
                np.nan,

            "fraction_above_zero":
                np.nan,

            "direction":
                "undefined",
        }


    ci_low, ci_high = np.percentile(
        valid,
        CI_PERCENTILES
    )


    if ci_low > 0:

        direction = "improved"

    elif ci_high < 0:

        direction = "worsened"

    else:

        direction = "uncertain"


    return {

        "point_delta":
            float(
                point_delta
            ),

        "bootstrap_mean_delta":
            float(
                np.mean(
                    valid
                )
            ),

        "CI_low":
            float(
                ci_low
            ),

        "CI_high":
            float(
                ci_high
            ),

        "valid_bootstraps":
            int(
                len(
                    valid
                )
            ),

        "fraction_below_zero":
            float(
                np.mean(
                    valid < 0
                )
            ),

        "fraction_above_zero":
            float(
                np.mean(
                    valid > 0
                )
            ),

        "direction":
            direction,
    }


# ======================================================================================
# 11. BUILD BOOTSTRAP SUMMARY
# ======================================================================================

summary_rows = []


for task in TASKS:

    role = (
        "primary"
        if task in PRIMARY_TASKS
        else "exploratory"
    )


    data = task_data[
        task
    ]


    y = data[
        "y"
    ]


    point_metrics = {}


    for model in MODELS:

        point_metrics[
            model
        ] = calculate_metrics(
            y,
            data[
                "probabilities"
            ][
                model
            ]
        )


    for (
        comparison_name,
        model_a,
        model_b
    ) in COMPARISONS:

        for metric in METRICS:

            point_delta = (
                point_metrics[
                    model_a
                ][
                    metric
                ]
                -
                point_metrics[
                    model_b
                ][
                    metric
                ]
            )


            result = bootstrap_summary(
                bootstrap_store[
                    task
                ][
                    comparison_name
                ][
                    metric
                ],
                point_delta
            )


            summary_rows.append(
                {
                    "task":
                        task,

                    "role":
                        role,

                    "comparison":
                        comparison_name,

                    "metric":
                        metric,

                    "N":
                        int(
                            len(
                                y
                            )
                        ),

                    "positive":
                        int(
                            y.sum()
                        ),

                    **result,
                }
            )


bootstrap_summary_df = pd.DataFrame(
    summary_rows
)


# ======================================================================================
# 12. SAVE BOOTSTRAP REPLICATES
# ======================================================================================

npz_payload = {}


for task in TASKS:

    for comparison_name, _, _ in COMPARISONS:

        for metric in METRICS:

            key = (
                f"{task}"
                f"__{comparison_name}"
                f"__{metric}"
            )


            npz_payload[
                key
            ] = (
                bootstrap_store[
                    task
                ][
                    comparison_name
                ][
                    metric
                ]
            )


np.savez_compressed(
    LUDB_DIR
    /
    "ludb_step5d_bootstrap_replicates.npz",

    **npz_payload
)


# ======================================================================================
# 13. SAVE TABLES
# ======================================================================================

bootstrap_summary_df.to_csv(
    LUDB_DIR
    /
    "ludb_step5d_paired_bootstrap_summary.csv",
    index=False
)


point_df.to_csv(
    LUDB_DIR
    /
    "ludb_step5d_point_estimates.csv",
    index=False
)


# ======================================================================================
# 14. PRINT PRIMARY AUROC/AP RESULTS
# ======================================================================================

primary_main = bootstrap_summary_df[
    (
        bootstrap_summary_df[
            "role"
        ]
        ==
        "primary"
    )
    &
    (
        bootstrap_summary_df[
            "metric"
        ].isin(
            [
                "AUROC",
                "AP",
            ]
        )
    )
]


print(
    "\n\n"
    +
    "=" * 120
)

print(
    "LUDB PRIMARY TASKS — PAIRED BOOTSTRAP DIFFERENCES"
)

print(
    "=" * 120
)


print(
    primary_main[
        [
            "task",
            "comparison",
            "metric",
            "N",
            "positive",
            "point_delta",
            "CI_low",
            "CI_high",
            "valid_bootstraps",
            "direction",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 15. PRINT EXPLORATORY AUROC/AP RESULTS
# ======================================================================================

exploratory_main = bootstrap_summary_df[
    (
        bootstrap_summary_df[
            "role"
        ]
        ==
        "exploratory"
    )
    &
    (
        bootstrap_summary_df[
            "metric"
        ].isin(
            [
                "AUROC",
                "AP",
            ]
        )
    )
]


print(
    "\n\n"
    +
    "=" * 120
)

print(
    "LUDB EXPLORATORY TASKS — PAIRED BOOTSTRAP DIFFERENCES"
)

print(
    "=" * 120
)


print(
    exploratory_main[
        [
            "task",
            "comparison",
            "metric",
            "N",
            "positive",
            "point_delta",
            "CI_low",
            "CI_high",
            "valid_bootstraps",
            "direction",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 16. PRIMARY SUMMARY: ONLY GAZE VS BASE
# ======================================================================================

primary_vs_base = primary_main[
    primary_main[
        "comparison"
    ].isin(
        [
            "CONSULTANT_vs_BASE",
            "ALL_CLINICIANS_vs_BASE",
        ]
    )
]


print(
    "\n\n"
    +
    "=" * 120
)

print(
    "PRIMARY GAZE-vs-BASE SUMMARY"
)

print(
    "=" * 120
)


print(
    primary_vs_base[
        [
            "task",
            "comparison",
            "metric",
            "point_delta",
            "CI_low",
            "CI_high",
            "direction",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ======================================================================================
# 17. MANIFEST
# ======================================================================================

manifest = {

    "stage":
        "Step 5D LUDB paired bootstrap from frozen OOF predictions",

    "source_predictions":
        str(
            PRED_PATH
        ),

    "bootstrap_repetitions":
        N_BOOT,

    "bootstrap_seed":
        SEED,

    "resampling_unit":
        "LUDB record (one record per unique LUDB ID)",

    "paired":
        True,

    "same_bootstrap_sample_for_all_models":
        True,

    "confidence_interval":
        "percentile 95% CI (2.5th, 97.5th percentiles)",

    "primary_tasks":
        PRIMARY_TASKS,

    "exploratory_tasks":
        EXPLORATORY_TASKS,

    "primary_metrics":
        [
            "AUROC",
            "Average Precision",
        ],

    "secondary_metric":
        "balanced accuracy",

    "comparisons":
        [
            comparison_name
            for comparison_name, _, _
            in COMPARISONS
        ],

    "model_inference_performed":
        False,

    "linear_probe_training_performed":
        False,

    "gaze_model_training_performed":
        False,

    "endpoint_changes_performed":
        False,
}


with open(
    LUDB_DIR
    /
    "ludb_step5d_manifest.json",
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ======================================================================================
# 18. FINISH
# ======================================================================================

print(
    "\n\n"
    +
    "=" * 110
)

print(
    "STEP 5D COMPLETE — SAVED OOF PREDICTIONS ONLY"
)

print(
    "=" * 110
)


print(
    """
Please paste back:

1. LUDB PRIMARY TASKS — PAIRED BOOTSTRAP DIFFERENCES

2. LUDB EXPLORATORY TASKS — PAIRED BOOTSTRAP DIFFERENCES

3. PRIMARY GAZE-vs-BASE SUMMARY

Do not rerun or modify any models, probes, endpoints, folds, or preprocessing.
"""
)

In [ ]:
# ======================================================================================
# STEP 6A — VISUALISE CLINICIAN GAZE vs MODEL LEAD ATTENTION
#
# READ-ONLY MODEL INFERENCE.
# NO TRAINING.
# NO WEIGHT CHANGES.
#
# Produces:
#
#   Consultant supervision:
#       Clinician gaze
#       Base-model lead attention
#       Consultant-gaze model attention
#       Change = Consultant model - Base
#
#   All-clinician supervision:
#       Clinician gaze
#       Base-model lead attention
#       All-clinician model attention
#       Change = All-clinician model - Base
#
# Attention definition is EXACTLY the frozen primary definition:
#
#   add_3 : (batch, 12, 55, 128)
#   lead RMS = sqrt(mean(add_3^2 across time and channels))
#   normalise across 12 leads
#
# ======================================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

import matplotlib.pyplot as plt


# ======================================================================================
# 1. PATHS
# ======================================================================================

ROOT = Path("definitive_clean_gaze")

X_PATH = Path(
    "final_digitisation_v9/X_gaze_final.npy"
)

GAZE_TARGET_PATH = (
    ROOT
    /
    "clean_gaze_targets.npz"
)

BASE_MODEL_PATH = Path(
    "12 leads.h5"
)

CONSULTANT_WEIGHTS = (
    ROOT
    /
    "final_models"
    /
    "clean_consultant.weights.h5"
)

ALL_WEIGHTS = (
    ROOT
    /
    "final_models"
    /
    "clean_all_clinicians.weights.h5"
)

OUT_DIR = (
    ROOT
    /
    "visualisations"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for path in [
    X_PATH,
    GAZE_TARGET_PATH,
    BASE_MODEL_PATH,
    CONSULTANT_WEIGHTS,
    ALL_WEIGHTS,
]:
    if not path.exists():
        raise FileNotFoundError(path)


# ======================================================================================
# 2. FIXED ORDERS
# ======================================================================================

ECG_KEYS = [
    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]


STANDARD_LEADS = [
    "I",
    "II",
    "III",
    "aVR",
    "aVL",
    "aVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]


# Physical 3 × 4 ECG display layout
DISPLAY_LAYOUT = [
    ["I",   "aVR", "V1", "V4"],
    ["II",  "aVL", "V2", "V5"],
    ["III", "aVF", "V3", "V6"],
]


LEAD_INDEX = {
    lead: i
    for i, lead in enumerate(STANDARD_LEADS)
}


# ======================================================================================
# 3. LOAD FROZEN ECGs
# ======================================================================================

X = np.load(
    X_PATH
).astype(
    np.float32
)


print("=" * 100)
print("FROZEN GAZE ECG DATA")
print("=" * 100)

print(
    "Shape:",
    X.shape
)


if X.shape != (
    10,
    12,
    1000,
    1
):
    raise RuntimeError(
        f"Unexpected X shape: {X.shape}"
    )


# ======================================================================================
# 4. LOAD CLEAN GAZE TARGETS
#
# Robustly identify the two 10 × 12 matrices.
# ======================================================================================

npz = np.load(
    GAZE_TARGET_PATH
)


print(
    "\nNPZ keys:",
    npz.files
)


candidate_arrays = {
    key: np.asarray(npz[key])
    for key in npz.files
    if np.asarray(npz[key]).shape == (10, 12)
}


if len(candidate_arrays) < 2:
    raise RuntimeError(
        "Could not find two (10,12) gaze-target arrays."
    )


consultant_target = None
all_target = None


for key, array in candidate_arrays.items():

    lower = key.lower()

    if (
        "consult" in lower
        and
        "all" not in lower
    ):
        consultant_target = array.astype(
            np.float64
        )

    if (
        "all" in lower
        or
        "clinician" in lower
    ):
        if "consult" not in lower or "all" in lower:
            all_target = array.astype(
                np.float64
            )


# Fallback if names are different but exactly two matrices exist
if (
    consultant_target is None
    or
    all_target is None
):

    print(
        "\nAutomatic name matching incomplete."
    )

    print(
        "Candidate arrays:"
    )

    for key in candidate_arrays:
        print(
            key
        )

    # Prefer arrays by mean difference from one another only if exactly two.
    if len(candidate_arrays) == 2:

        keys = list(
            candidate_arrays.keys()
        )

        consultant_target = (
            candidate_arrays[
                keys[0]
            ]
            .astype(
                np.float64
            )
        )

        all_target = (
            candidate_arrays[
                keys[1]
            ]
            .astype(
                np.float64
            )
        )

        print(
            "\nWARNING:"
            "\nUsing candidate arrays in stored order:"
            f"\n  Consultant assumed = {keys[0]}"
            f"\n  All clinicians assumed = {keys[1]}"
        )

    else:

        raise RuntimeError(
            "Please inspect the NPZ keys before proceeding."
        )


# Validate target distributions
for name, target in [
    (
        "Consultant",
        consultant_target
    ),
    (
        "All clinicians",
        all_target
    ),
]:

    if target.shape != (
        10,
        12
    ):
        raise RuntimeError(
            f"{name} target shape incorrect."
        )

    if np.any(
        target < 0
    ):
        raise RuntimeError(
            f"{name}: negative gaze weights."
        )

    sums = target.sum(
        axis=1
    )

    if not np.allclose(
        sums,
        1.0,
        atol=1e-5
    ):
        raise RuntimeError(
            f"{name}: target rows do not sum to 1."
        )


print(
    "\n✓ Clean gaze targets loaded."
)


# ======================================================================================
# 5. LOAD THREE FROZEN MODELS
# ======================================================================================

def load_variant(
    weights=None
):

    model = tf.keras.models.load_model(
        BASE_MODEL_PATH,
        compile=False
    )

    if weights is not None:

        model.load_weights(
            weights
        )

    return model


base_model = load_variant()

consultant_model = load_variant(
    CONSULTANT_WEIGHTS
)

all_model = load_variant(
    ALL_WEIGHTS
)


# ======================================================================================
# 6. FROZEN RMS LEAD-ATTENTION READOUT
# ======================================================================================

def rms_lead_attention(
    model,
    X
):

    feature_model = tf.keras.Model(
        inputs=model.input,
        outputs=model.get_layer(
            "add_3"
        ).output
    )


    feature = feature_model.predict(
        X,
        batch_size=10,
        verbose=0
    )


    print(
        model.name,
        "add_3:",
        feature.shape
    )


    if feature.shape != (
        10,
        12,
        55,
        128
    ):
        raise RuntimeError(
            f"Unexpected add_3 shape: {feature.shape}"
        )


    # EXACT frozen attention definition
    lead_energy = np.sqrt(
        np.mean(
            np.square(
                feature.astype(
                    np.float64
                )
            ),
            axis=(
                2,
                3
            )
        )
    )


    lead_attention = (
        lead_energy
        /
        lead_energy.sum(
            axis=1,
            keepdims=True
        )
    )


    return lead_attention


print("\n" + "=" * 100)
print("MODEL ATTENTION EXTRACTION")
print("=" * 100)


base_attention = rms_lead_attention(
    base_model,
    X
)

consultant_attention = rms_lead_attention(
    consultant_model,
    X
)

all_attention = rms_lead_attention(
    all_model,
    X
)


for name, array in [
    (
        "BASE",
        base_attention
    ),
    (
        "CONSULTANT",
        consultant_attention
    ),
    (
        "ALL_CLINICIANS",
        all_attention
    ),
]:

    print(
        name,
        "shape:",
        array.shape,
        "| row sums:",
        np.round(
            array.sum(
                axis=1
            ),
            6
        )
    )


# ======================================================================================
# 7. CONVERT STANDARD LEAD ORDER -> PHYSICAL ECG GRID
# ======================================================================================

def vector_to_display_grid(
    vector
):

    grid = np.zeros(
        (
            3,
            4
        ),
        dtype=float
    )


    for row in range(
        3
    ):

        for col in range(
            4
        ):

            lead = DISPLAY_LAYOUT[
                row
            ][
                col
            ]

            grid[
                row,
                col
            ] = vector[
                LEAD_INDEX[
                    lead
                ]
            ]


    return grid


# ======================================================================================
# 8. COSINE SIMILARITY HELPER
# ======================================================================================

def cosine(
    a,
    b
):

    return float(
        np.dot(
            a,
            b
        )
        /
        (
            np.linalg.norm(
                a
            )
            *
            np.linalg.norm(
                b
            )
            +
            1e-12
        )
    )


# ======================================================================================
# 9. DRAW ONE 3×4 LEAD MAP
# ======================================================================================

def draw_lead_map(
    ax,
    vector,
    title,
    vmin,
    vmax,
    cmap,
    show_values=True,
):

    grid = vector_to_display_grid(
        vector
    )


    image = ax.imshow(
        grid,
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
        aspect="equal"
    )


    ax.set_xticks([])
    ax.set_yticks([])


    for row in range(
        3
    ):

        for col in range(
            4
        ):

            lead = DISPLAY_LAYOUT[
                row
            ][
                col
            ]

            value = grid[
                row,
                col
            ]


            if show_values:

                label = (
                    f"{lead}\n"
                    f"{100 * value:.1f}%"
                )

            else:

                label = lead


            # Select text contrast using local cell intensity.
            midpoint = (
                vmin
                +
                vmax
            ) / 2


            text_colour = (
                "white"
                if value > midpoint
                else "black"
            )


            ax.text(
                col,
                row,
                label,
                ha="center",
                va="center",
                fontsize=9,
                color=text_colour,
                fontweight="bold",
            )


    ax.set_title(
        title,
        fontsize=11,
        pad=8
    )


    return image


# ======================================================================================
# 10. INDIVIDUAL ECG FIGURES
# ======================================================================================

def save_individual_figures(
    population_name,
    gaze_target,
    guided_attention,
    filename_prefix,
):

    absolute_max = max(
        float(
            gaze_target.max()
        ),
        float(
            base_attention.max()
        ),
        float(
            guided_attention.max()
        ),
    )


    delta = (
        guided_attention
        -
        base_attention
    )


    delta_abs_max = float(
        np.max(
            np.abs(
                delta
            )
        )
    )


    for i, ecg in enumerate(
        ECG_KEYS
    ):

        base_cos = cosine(
            base_attention[
                i
            ],
            gaze_target[
                i
            ]
        )


        guided_cos = cosine(
            guided_attention[
                i
            ],
            gaze_target[
                i
            ]
        )


        fig, axes = plt.subplots(
            1,
            4,
            figsize=(
                15,
                3.7
            )
        )


        im1 = draw_lead_map(
            axes[
                0
            ],
            gaze_target[
                i
            ],
            f"{population_name} gaze",
            vmin=0,
            vmax=absolute_max,
            cmap="viridis",
        )


        draw_lead_map(
            axes[
                1
            ],
            base_attention[
                i
            ],
            (
                "Base model\n"
                f"cosine={base_cos:.3f}"
            ),
            vmin=0,
            vmax=absolute_max,
            cmap="viridis",
        )


        draw_lead_map(
            axes[
                2
            ],
            guided_attention[
                i
            ],
            (
                "Gaze-guided model\n"
                f"cosine={guided_cos:.3f}"
            ),
            vmin=0,
            vmax=absolute_max,
            cmap="viridis",
        )


        im4 = draw_lead_map(
            axes[
                3
            ],
            delta[
                i
            ],
            (
                "Change in model attention\n"
                "guided − base"
            ),
            vmin=-delta_abs_max,
            vmax=delta_abs_max,
            cmap="RdBu_r",
        )


        fig.suptitle(
            (
                f"{ecg} — {population_name} supervision"
                f" | Δ cosine = {guided_cos - base_cos:+.3f}"
            ),
            fontsize=13,
            fontweight="bold",
        )


        # Shared attention colour bar
        cb1 = fig.colorbar(
            im1,
            ax=axes[
                :3
            ],
            fraction=0.025,
            pad=0.02,
        )

        cb1.set_label(
            "Normalised lead attention"
        )


        # Delta colour bar
        cb2 = fig.colorbar(
            im4,
            ax=axes[
                3
            ],
            fraction=0.08,
            pad=0.04,
        )

        cb2.set_label(
            "Change in attention"
        )


        fig.subplots_adjust(
            left=0.03,
            right=0.96,
            top=0.78,
            bottom=0.08,
            wspace=0.35,
        )


        png_path = (
            OUT_DIR
            /
            f"{filename_prefix}_{ecg}.png"
        )


        pdf_path = (
            OUT_DIR
            /
            f"{filename_prefix}_{ecg}.pdf"
        )


        fig.savefig(
            png_path,
            dpi=300,
            bbox_inches="tight"
        )


        fig.savefig(
            pdf_path,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


# ======================================================================================
# 11. MAKE INDIVIDUAL FIGURES
# ======================================================================================

save_individual_figures(
    population_name="Consultant",
    gaze_target=consultant_target,
    guided_attention=consultant_attention,
    filename_prefix="consultant_attention",
)


save_individual_figures(
    population_name="All-clinician",
    gaze_target=all_target,
    guided_attention=all_attention,
    filename_prefix="all_clinician_attention",
)


# ======================================================================================
# 12. MASTER 10-ECG FIGURE
#
# One figure per gaze population.
# Values omitted inside cells to keep the master figure readable.
# ======================================================================================

def make_master_figure(
    population_name,
    gaze_target,
    guided_attention,
    filename,
):

    delta = (
        guided_attention
        -
        base_attention
    )


    absolute_max = max(
        float(
            gaze_target.max()
        ),
        float(
            base_attention.max()
        ),
        float(
            guided_attention.max()
        ),
    )


    delta_abs_max = float(
        np.max(
            np.abs(
                delta
            )
        )
    )


    fig, axes = plt.subplots(
        10,
        4,
        figsize=(
            14,
            28
        )
    )


    last_absolute_image = None
    last_delta_image = None


    for i, ecg in enumerate(
        ECG_KEYS
    ):

        base_cos = cosine(
            base_attention[
                i
            ],
            gaze_target[
                i
            ]
        )


        guided_cos = cosine(
            guided_attention[
                i
            ],
            gaze_target[
                i
            ]
        )


        last_absolute_image = draw_lead_map(
            axes[
                i,
                0
            ],
            gaze_target[
                i
            ],
            (
                f"{ecg}\n"
                f"{population_name} gaze"
            ),
            0,
            absolute_max,
            "viridis",
            show_values=False,
        )


        draw_lead_map(
            axes[
                i,
                1
            ],
            base_attention[
                i
            ],
            (
                "Base"
                f"\ncos={base_cos:.3f}"
            ),
            0,
            absolute_max,
            "viridis",
            show_values=False,
        )


        draw_lead_map(
            axes[
                i,
                2
            ],
            guided_attention[
                i
            ],
            (
                "Gaze-guided"
                f"\ncos={guided_cos:.3f}"
            ),
            0,
            absolute_max,
            "viridis",
            show_values=False,
        )


        last_delta_image = draw_lead_map(
            axes[
                i,
                3
            ],
            delta[
                i
            ],
            (
                "Δ attention"
                f"\nΔcos={guided_cos - base_cos:+.3f}"
            ),
            -delta_abs_max,
            delta_abs_max,
            "RdBu_r",
            show_values=False,
        )


    fig.suptitle(
        (
            f"Clinician gaze and model lead-attention distributions — "
            f"{population_name} supervision"
        ),
        fontsize=16,
        fontweight="bold",
        y=0.995,
    )


    fig.subplots_adjust(
        left=0.04,
        right=0.94,
        top=0.975,
        bottom=0.03,
        hspace=0.65,
        wspace=0.30,
    )


    # Shared colour bars
    cbar_ax1 = fig.add_axes(
        [
            0.955,
            0.54,
            0.012,
            0.34,
        ]
    )


    cb1 = fig.colorbar(
        last_absolute_image,
        cax=cbar_ax1,
    )

    cb1.set_label(
        "Normalised attention"
    )


    cbar_ax2 = fig.add_axes(
        [
            0.955,
            0.12,
            0.012,
            0.34,
        ]
    )


    cb2 = fig.colorbar(
        last_delta_image,
        cax=cbar_ax2,
    )

    cb2.set_label(
        "Guided − base"
    )


    png_path = (
        OUT_DIR
        /
        f"{filename}.png"
    )


    pdf_path = (
        OUT_DIR
        /
        f"{filename}.pdf"
    )


    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight"
    )


    fig.savefig(
        pdf_path,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ======================================================================================
# 13. CREATE MASTER FIGURES
# ======================================================================================

make_master_figure(
    population_name="Consultant",
    gaze_target=consultant_target,
    guided_attention=consultant_attention,
    filename="MASTER_consultant_gaze_vs_model_attention",
)


make_master_figure(
    population_name="All-clinician",
    gaze_target=all_target,
    guided_attention=all_attention,
    filename="MASTER_all_clinician_gaze_vs_model_attention",
)


# ======================================================================================
# 14. SAVE EXACT NUMERICAL DISTRIBUTIONS
# ======================================================================================

rows = []


for i, ecg in enumerate(
    ECG_KEYS
):

    for j, lead in enumerate(
        STANDARD_LEADS
    ):

        rows.append(
            {
                "ECG":
                    ecg,

                "lead":
                    lead,

                "consultant_gaze":
                    consultant_target[
                        i,
                        j
                    ],

                "all_clinician_gaze":
                    all_target[
                        i,
                        j
                    ],

                "base_attention":
                    base_attention[
                        i,
                        j
                    ],

                "consultant_model_attention":
                    consultant_attention[
                        i,
                        j
                    ],

                "all_clinician_model_attention":
                    all_attention[
                        i,
                        j
                    ],

                "consultant_change_vs_base":
                    consultant_attention[
                        i,
                        j
                    ]
                    -
                    base_attention[
                        i,
                        j
                    ],

                "all_clinician_change_vs_base":
                    all_attention[
                        i,
                        j
                    ]
                    -
                    base_attention[
                        i,
                        j
                    ],
            }
        )


attention_df = pd.DataFrame(
    rows
)


attention_df.to_csv(
    OUT_DIR
    /
    "lead_attention_visualisation_values.csv",
    index=False
)


# ======================================================================================
# 15. SUMMARY
# ======================================================================================

print("\n" + "=" * 100)
print("STEP 6A COMPLETE")
print("=" * 100)


print(
    "\nFigures saved to:"
)

print(
    OUT_DIR
)


print(
    """
Main files:

  MASTER_consultant_gaze_vs_model_attention.png
  MASTER_consultant_gaze_vs_model_attention.pdf

  MASTER_all_clinician_gaze_vs_model_attention.png
  MASTER_all_clinician_gaze_vs_model_attention.pdf

Plus 20 individual ECG figures:
  consultant_attention_*.png / .pdf
  all_clinician_attention_*.png / .pdf

Numerical source data:
  lead_attention_visualisation_values.csv
"""
)